In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

BASE = "/content/drive/MyDrive/Stage_PFA/experiments"

print("Stage_PFA directory exists:", os.path.exists(BASE))

if os.path.exists(BASE):
    print("\nContents:")
    for item in os.listdir(BASE):
        print(" -", item)

Stage_PFA directory exists: True

Contents:
 - Exp01_YOLOV8N
 - Exp02_YOLOV8N
 - Exp04_YOLO_RT-DETR_1
 - Exp07B_YOLOV8_EPOCHS_100
 - Exp03_YOLO11N_EPOCHS_100


In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 7.6 MB/s eta 0:00:00


In [ ]:
!pip install onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 38.2 MB/s eta 0:00:00


In [ ]:
import onnx
import onnxruntime as ort

print("ONNX:", onnx.__version__)
print("ONNX Runtime:", ort.__version__)

ONNX: 1.22.0
ONNX Runtime: 1.29.0


In [ ]:
# ============================================================
# STAGE PFA — DETECTION MODEL OPTIMIZATION WORKSPACE
# STEP 1C — GPU VERSION
# ============================================================
#
# PURPOSE:
#   1. Mount Google Drive
#   2. Verify the Colab GPU/CUDA environment
#   3. Define the exact five models
#   4. Verify every checkpoint exists
#   5. Load every model onto GPU
#   6. Record model + GPU information
#   7. Create a clean optimization directory
#   8. Save model registry and experiment metadata
#
# IMPORTANT:
#   Original .pt checkpoints are NEVER modified.
#
#   GPU is used for model loading/verification here.
#   The major GPU speed benefit will occur during:
#       - FP32 validation
#       - ONNX validation
#       - FP16 optimization
#       - INT8 calibration/validation
# ============================================================


# ============================================================
# 1. IMPORT REQUIRED PYTHON MODULES
# ============================================================

import sys
import subprocess
import os
import json
import time
import shutil

print("=" * 90)
print("STEP 1C — GPU OPTIMIZATION WORKSPACE INITIALIZATION")
print("=" * 90)


# ============================================================
# 2. VERIFY / INSTALL ULTRALYTICS
# ============================================================

try:

    import ultralytics

    print(
        f"✓ Ultralytics already installed: "
        f"{ultralytics.__version__}"
    )

except ImportError:

    print("Installing Ultralytics...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "ultralytics"
    ])

    import ultralytics

    print(
        f"✓ Ultralytics installed: "
        f"{ultralytics.__version__}"
    )


# ============================================================
# 3. IMPORT PYTORCH
# ============================================================

import torch


# ============================================================
# 4. VERIFY GPU / CUDA
# ============================================================

print("\n" + "=" * 90)
print("GPU / CUDA ENVIRONMENT CHECK")
print("=" * 90)

print(
    f"\nPyTorch version : "
    f"{torch.__version__}"
)

print(
    f"CUDA available  : "
    f"{torch.cuda.is_available()}"
)

print(
    f"CUDA version    : "
    f"{torch.version.cuda}"
)

if not torch.cuda.is_available():

    raise RuntimeError(
        "\n"
        "============================================================\n"
        "CUDA GPU NOT AVAILABLE\n"
        "============================================================\n\n"
        "Please change the Colab runtime to GPU:\n\n"
        "Runtime → Change runtime type → GPU\n\n"
        "Then reconnect the runtime and run this cell again.\n"
    )


# ------------------------------------------------------------
# GPU information
# ------------------------------------------------------------

GPU_INDEX = 0

GPU_NAME = torch.cuda.get_device_name(
    GPU_INDEX
)

GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(
        GPU_INDEX
    ).total_memory
    / (1024 ** 3)
)

CUDA_CAPABILITY = (
    torch.cuda.get_device_capability(
        GPU_INDEX
    )
)

print(
    f"GPU device      : "
    f"{GPU_NAME}"
)

print(
    f"GPU memory      : "
    f"{GPU_MEMORY_GB:.2f} GB"
)

print(
    f"Compute capability: "
    f"{CUDA_CAPABILITY[0]}.{CUDA_CAPABILITY[1]}"
)

print("\n✓ CUDA GPU is available")


# ============================================================
# 5. DEFINE PROJECT ROOT
# ============================================================

STAGE_PFA = (
    "/content/drive/MyDrive/Stage_PFA"
)

if not os.path.isdir(STAGE_PFA):

    raise FileNotFoundError(
        f"\nStage_PFA directory not found:\n"
        f"{STAGE_PFA}"
    )

print(
    f"\n✓ Project directory found:\n"
    f"  {STAGE_PFA}"
)


# ============================================================
# 6. DEFINE THE FIVE EXACT MODELS
# ============================================================
#
# DO NOT CHANGE MODEL NAMES.
#
# These names are used later by the optimization pipeline.
# ============================================================

MODEL_PATHS = {

    # --------------------------------------------------------
    # YOLOv8n — 50 epochs
    # --------------------------------------------------------

    "yolov8n_50ep":
        "/content/drive/MyDrive/Stage_PFA/"
        "experiments/Exp01_YOLOV8N/"
        "weights/best.pt",


    # --------------------------------------------------------
    # YOLOv8n — 75 epochs
    # --------------------------------------------------------

    "yolov8n_75ep":
        "/content/drive/MyDrive/Stage_PFA/"
        "experiments/Exp02_YOLOV8N/"
        "weights/best.pt",


    # --------------------------------------------------------
    # YOLOv8n — 100 epochs
    # --------------------------------------------------------

    "yolov8n_100ep":
        "/content/drive/MyDrive/Stage_PFA/"
        "experiments/Exp07B_YOLOV8_EPOCHS_100/"
        "weights/best.pt",


    # --------------------------------------------------------
    # YOLO11n — 100 epochs
    # --------------------------------------------------------

    "yolo11n":
        "/content/drive/MyDrive/Stage_PFA/experiments/Exp03_YOLO11N_EPOCHS_100/weights/best.pt",


    # --------------------------------------------------------
    # RT-DETR — 50 epochs
    # --------------------------------------------------------

    "rtdetr_50ep":
        "/content/drive/MyDrive/Stage_PFA/"
        "experiments/Exp04_YOLO_RT-DETR_1/"
        "weights/best.pt",
}


# ============================================================
# 7. CHECK MODEL CHECKPOINTS
# ============================================================

print("\n" + "=" * 90)
print("CHECKING MODEL CHECKPOINTS")
print("=" * 90)

missing_models = []

for model_name, model_path in MODEL_PATHS.items():

    if not os.path.isfile(model_path):

        missing_models.append(
            (
                model_name,
                model_path
            )
        )

        print(
            f"✗ {model_name}"
        )

        print(
            f"    NOT FOUND: "
            f"{model_path}"
        )

    else:

        size_mb = (
            os.path.getsize(model_path)
            / (1024 ** 2)
        )

        print(
            f"✓ {model_name}"
        )

        print(
            f"    {model_path}"
        )

        print(
            f"    Size: "
            f"{size_mb:.2f} MB"
        )


if missing_models:

    print(
        "\n" + "!" * 90
    )

    print(
        "ERROR: SOME MODEL PATHS ARE INVALID"
    )

    print(
        "!" * 90
    )

    for name, path in missing_models:

        print(
            f"\n{name}"
        )

        print(path)

    raise FileNotFoundError(
        "\nCorrect the MODEL_PATHS dictionary "
        "and run this cell again."
    )


print(
    "\n✓ All five checkpoints were found."
)


# ============================================================
# 8. CREATE OPTIMIZATION DIRECTORY STRUCTURE
# ============================================================

OPT_ROOT = os.path.join(
    STAGE_PFA,
    "MODEL_OPTIMIZATION"
)


DIRECTORIES = {

    "root":
        OPT_ROOT,

    # Original model references
    "original":
        os.path.join(
            OPT_ROOT,
            "01_original"
        ),

    # FP32 ONNX
    "onnx_fp32":
        os.path.join(
            OPT_ROOT,
            "02_onnx_fp32"
        ),

    # FP16 ONNX
    "onnx_fp16":
        os.path.join(
            OPT_ROOT,
            "03_onnx_fp16"
        ),

    # INT8 ONNX
    "onnx_int8":
        os.path.join(
            OPT_ROOT,
            "04_onnx_int8"
        ),

    # Calibration
    "calibration":
        os.path.join(
            OPT_ROOT,
            "05_calibration"
        ),

    # Validation
    "validation":
        os.path.join(
            OPT_ROOT,
            "06_validation"
        ),

    # Benchmark results
    "benchmarks":
        os.path.join(
            OPT_ROOT,
            "07_benchmarks"
        ),

    # Final selected models
    "final":
        os.path.join(
            OPT_ROOT,
            "08_final"
        ),

    # Logs
    "logs":
        os.path.join(
            OPT_ROOT,
            "09_logs"
        ),
}


for path in DIRECTORIES.values():

    os.makedirs(
        path,
        exist_ok=True
    )


print(
    "\n" + "=" * 90
)

print(
    "OPTIMIZATION DIRECTORY STRUCTURE"
)

print(
    "=" * 90
)


for key, path in DIRECTORIES.items():

    print(
        f"{key:15s} → {path}"
    )


# ============================================================
# 9. LOAD MODELS ON GPU AND COLLECT INFORMATION
# ============================================================

from ultralytics import YOLO


print(
    "\n" + "=" * 90
)

print(
    "LOADING AND INSPECTING MODELS ON GPU"
)

print(
    "=" * 90
)


MODEL_INFO = {}


for model_name, model_path in MODEL_PATHS.items():

    print(
        f"\n{'-' * 90}"
    )

    print(
        f"MODEL: {model_name}"
    )

    print(
        f"{'-' * 90}"
    )


    try:

        # ----------------------------------------------------
        # Load checkpoint
        # ----------------------------------------------------

        print(
            "→ Loading checkpoint..."
        )

        model = YOLO(
            model_path
        )


        # ----------------------------------------------------
        # Move model to GPU
        # ----------------------------------------------------

        print(
            "→ Moving model to CUDA..."
        )

        model.to(
            f"cuda:{GPU_INDEX}"
        )


        # ----------------------------------------------------
        # Basic file information
        # ----------------------------------------------------

        size_mb = (
            os.path.getsize(model_path)
            / (1024 ** 2)
        )


        # ----------------------------------------------------
        # Underlying PyTorch model
        # ----------------------------------------------------

        torch_model = model.model


        # ----------------------------------------------------
        # Parameter count
        # ----------------------------------------------------

        parameters = sum(

            p.numel()

            for p in torch_model.parameters()

        )


        # ----------------------------------------------------
        # Trainable parameters
        # ----------------------------------------------------

        trainable_parameters = sum(

            p.numel()

            for p in torch_model.parameters()

            if p.requires_grad

        )


        # ----------------------------------------------------
        # Model task
        # ----------------------------------------------------

        task = getattr(
            model,
            "task",
            "unknown"
        )


        # ----------------------------------------------------
        # Class names
        # ----------------------------------------------------

        names = getattr(
            model,
            "names",
            None
        )


        # ----------------------------------------------------
        # Detect actual model device
        # ----------------------------------------------------

        try:

            actual_device = str(
                next(
                    torch_model.parameters()
                ).device
            )

        except Exception:

            actual_device = "unknown"


        # ----------------------------------------------------
        # Save information
        # ----------------------------------------------------

        MODEL_INFO[model_name] = {

            "checkpoint":
                model_path,

            "checkpoint_size_mb":
                round(
                    size_mb,
                    3
                ),

            "parameters":
                parameters,

            "trainable_parameters":
                trainable_parameters,

            "task":
                task,

            "names":
                names,

            "device":
                actual_device,

            "gpu":
                GPU_NAME,

            "gpu_memory_gb":
                round(
                    GPU_MEMORY_GB,
                    3
                ),

            "cuda_version":
                str(
                    torch.version.cuda
                ),

            "pytorch_version":
                torch.__version__,

            "ultralytics_version":
                ultralytics.__version__,

        }


        # ----------------------------------------------------
        # Print information
        # ----------------------------------------------------

        print(
            "✓ Model loaded successfully"
        )

        print(
            f"  Device:             "
            f"{actual_device}"
        )

        print(
            f"  Task:               "
            f"{task}"
        )

        print(
            f"  Parameters:         "
            f"{parameters:,}"
        )

        print(
            f"  Trainable params:   "
            f"{trainable_parameters:,}"
        )

        print(
            f"  Checkpoint size:    "
            f"{size_mb:.2f} MB"
        )

        print(
            f"  Classes:            "
            f"{names}"
        )


        # ----------------------------------------------------
        # Release GPU memory before next model
        # ----------------------------------------------------

        del model
        del torch_model

        torch.cuda.empty_cache()


    except Exception as e:

        print(
            f"✗ FAILED TO LOAD "
            f"{model_name}"
        )

        print(
            f"  Error: {repr(e)}"
        )

        raise


# ============================================================
# 10. SAVE MODEL INVENTORY
# ============================================================

inventory_path = os.path.join(

    DIRECTORIES["logs"],

    "model_inventory.json"

)


with open(

    inventory_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        MODEL_INFO,

        f,

        indent=4,

        ensure_ascii=False,

        default=str

    )


print(
    "\n✓ Model inventory saved:"
)

print(
    f"  {inventory_path}"
)


# ============================================================
# 11. SAVE GPU ENVIRONMENT INFORMATION
# ============================================================

GPU_INFO = {

    "pytorch_version":
        torch.__version__,

    "ultralytics_version":
        ultralytics.__version__,

    "cuda_available":
        torch.cuda.is_available(),

    "cuda_version":
        str(
            torch.version.cuda
        ),

    "gpu_index":
        GPU_INDEX,

    "gpu_name":
        GPU_NAME,

    "gpu_memory_gb":
        round(
            GPU_MEMORY_GB,
            3
        ),

    "compute_capability":
        (
            f"{CUDA_CAPABILITY[0]}."
            f"{CUDA_CAPABILITY[1]}"
        ),

    "device":
        f"cuda:{GPU_INDEX}",

    "experiment_stage":
        "Step 1C",

    "execution_environment":
        "Google Colab GPU",

}


gpu_info_path = os.path.join(

    DIRECTORIES["logs"],

    "gpu_environment.json"

)


with open(

    gpu_info_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        GPU_INFO,

        f,

        indent=4

    )


print(
    "\n✓ GPU environment information saved:"
)

print(
    f"  {gpu_info_path}"
)


# ============================================================
# 12. CREATE HUMAN-READABLE MODEL SUMMARY
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "MODEL INVENTORY SUMMARY"
)

print(
    "=" * 90
)


print(

    f"{'MODEL':<20}"
    f"{'PARAMETERS':>18}"
    f"{'SIZE (MB)':>14}"
    f"{'TASK':>12}"
    f"{'DEVICE':>14}"

)


print(
    "-" * 90
)


for model_name, info in MODEL_INFO.items():

    print(

        f"{model_name:<20}"

        f"{info['parameters']:>18,}"

        f"{info['checkpoint_size_mb']:>14.2f}"

        f"{str(info['task']):>12}"

        f"{info['device']:>14}"

    )


# ============================================================
# 13. SAVE MODEL PATHS
# ============================================================

paths_file = os.path.join(

    DIRECTORIES["logs"],

    "model_paths.json"

)


with open(

    paths_file,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        MODEL_PATHS,

        f,

        indent=4

    )


print(
    "\n✓ Model paths saved:"
)

print(
    f"  {paths_file}"
)


# ============================================================
# 14. SAVE COMPLETE STEP 1C METADATA
# ============================================================

STEP1C_METADATA = {

    "step":
        "1C",

    "description":
        "GPU optimization workspace initialization",

    "timestamp":
        time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),

    "execution_environment":
        "Google Colab GPU",

    "gpu":
        GPU_INFO,

    "models":
        MODEL_INFO,

    "model_paths":
        MODEL_PATHS,

    "optimization_root":
        OPT_ROOT,

}


metadata_path = os.path.join(

    DIRECTORIES["logs"],

    "step1c_gpu_metadata.json"

)


with open(

    metadata_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        STEP1C_METADATA,

        f,

        indent=4,

        ensure_ascii=False,

        default=str

    )


print(
    "\n✓ Complete Step 1C metadata saved:"
)

print(
    f"  {metadata_path}"
)


# ============================================================
# 15. FINAL GPU MEMORY STATUS
# ============================================================

torch.cuda.empty_cache()

allocated_gb = (
    torch.cuda.memory_allocated(
        GPU_INDEX
    )
    / (1024 ** 3)
)

reserved_gb = (
    torch.cuda.memory_reserved(
        GPU_INDEX
    )
    / (1024 ** 3)
)


print(
    "\n" + "=" * 90
)

print(
    "FINAL GPU MEMORY STATUS"
)

print(
    "=" * 90
)

print(
    f"GPU allocated : "
    f"{allocated_gb:.3f} GB"
)

print(
    f"GPU reserved  : "
    f"{reserved_gb:.3f} GB"
)


# ============================================================
# FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "STEP 1C — GPU VERSION COMPLETED SUCCESSFULLY"
)

print(
    "=" * 90
)


print(
    """
Five models registered:

  1. YOLOv8n — 50 epochs
  2. YOLOv8n — 75 epochs
  3. YOLOv8n — 100 epochs
  4. YOLO11n — 100 epochs
  5. RT-DETR — 50 epochs

GPU environment verified.

All five checkpoints loaded successfully on CUDA.

Original .pt checkpoints remain untouched.
No model has been converted.
No model has been modified.

GPU metadata has been recorded.

NEXT:

    STEP 2
        ↓
    FP32 GPU baseline
        ↓
    ONNX FP32 export
        ↓
    ONNX FP32 verification
        ↓
    STEP 2D
        ↓
    .pt FP32 vs ONNX FP32 dataset equivalence
        ↓
    STEP 3
        ↓
    FP16 optimization
        ↓
    INT8 optimization
        ↓
    Raspberry Pi deployment benchmark
"""
)

print(
    "\n✓ READY FOR STEP 2"
)

STEP 1C — GPU OPTIMIZATION WORKSPACE INITIALIZATION
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
✓ Ultralytics already installed: 8.4.130

GPU / CUDA ENVIRONMENT CHECK

PyTorch version : 2.11.0+cu128
CUDA available  : True
CUDA version    : 12.8
GPU device      : Tesla T4
GPU memory      : 14.56 GB
Compute capability: 7.5

✓ CUDA GPU is available

✓ Project directory found:
  /content/drive/MyDrive/Stage_PFA

CHECKING MODEL CHECKPOINTS
✓ yolov8n_50ep
    /content/drive/MyDrive/Stage_PFA/experiments/Exp01_YOLOV8N/weights/best.pt
    Size: 5.95 MB
✓ yolov8n_75ep
    /content/drive/MyDrive/Stage_PFA/experiments/Exp02_YOLOV8N/weights/best.pt
    Size: 5.96 MB
✓ yolov8n_100ep
    /content/drive/MyDrive/Stage_PFA/experiments/Exp07B

In [ ]:
# ============================================================
# STAGE PFA — DETECTION MODEL OPTIMIZATION
# STEP 2 — GPU FP32 BASELINE + ONNX FP32 EXPORT
# ============================================================
#
# PURPOSE:
#
#   PHASE A:
#       Establish FP32 inference baseline on Colab GPU
#
#   PHASE B:
#       Export all five models to ONNX FP32
#
#   PHASE C:
#       Validate ONNX structure and ONNX Runtime loading
#
# IMPORTANT:
#
#   - PyTorch inference uses CUDA GPU.
#   - GPU timing is synchronized correctly.
#   - ONNX export uses the GPU where supported.
#   - ONNX structural validation itself does not require GPU.
#   - Original .pt checkpoints remain untouched.
#
# MODELS:
#
#   1. YOLOv8n — 50 epochs
#   2. YOLOv8n — 75 epochs
#   3. YOLOv8n — 100 epochs
#   4. YOLO11n — 100 epochs
#   5. RT-DETR — 50 epochs
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import sys
import json
import time
import shutil
import warnings

import numpy as np
import pandas as pd

import torch
import onnx
import onnxruntime as ort

from pathlib import Path
from ultralytics import YOLO

warnings.filterwarnings("ignore")


# ============================================================
# 2. HEADER
# ============================================================

print("=" * 100)
print("STEP 2 — GPU FP32 BASELINE + ONNX FP32 EXPORT + VERIFICATION")
print("=" * 100)


# ============================================================
# 3. VERIFY CUDA
# ============================================================

print("\n" + "=" * 100)
print("GPU / CUDA VERIFICATION")
print("=" * 100)

print(f"\nPyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"CUDA version    : {torch.version.cuda}")


if not torch.cuda.is_available():

    raise RuntimeError(
        "\nCUDA GPU is not available.\n"
        "Go to:\n"
        "Runtime → Change runtime type → GPU\n"
        "and restart the runtime."
    )


DEVICE = "cuda:0"
GPU_INDEX = 0

GPU_NAME = torch.cuda.get_device_name(GPU_INDEX)

GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(GPU_INDEX).total_memory
    / (1024 ** 3)
)

print(f"GPU             : {GPU_NAME}")
print(f"GPU memory      : {GPU_MEMORY_GB:.2f} GB")
print(f"Device          : {DEVICE}")

print("\n✓ CUDA GPU verified")


# ============================================================
# 4. PROJECT PATHS
# ============================================================

STAGE_PFA = (
    "/content/drive/MyDrive/Stage_PFA"
)

DATA_YAML = (
    "/content/drive/MyDrive/PFE/"
    "Dataset/Vehicles_Dataset/data.yaml"
)

DATASET_ROOT = (
    "/content/drive/MyDrive/PFE/"
    "Dataset/Vehicles_Dataset"
)

OPT_ROOT = os.path.join(
    STAGE_PFA,
    "MODEL_OPTIMIZATION"
)

MODEL_PATHS_FILE = os.path.join(
    OPT_ROOT,
    "09_logs",
    "model_paths.json"
)

# IMPORTANT:
# These directory names match STEP 1C.
FP32_DIR = os.path.join(
    OPT_ROOT,
    "01_original"
)

ONNX_DIR = os.path.join(
    OPT_ROOT,
    "02_onnx_fp32"
)

LOG_DIR = os.path.join(
    OPT_ROOT,
    "09_logs"
)

BENCHMARK_DIR = os.path.join(
    OPT_ROOT,
    "07_benchmarks"
)

VALIDATION_DIR = os.path.join(
    OPT_ROOT,
    "06_validation"
)


for path in [
    FP32_DIR,
    ONNX_DIR,
    LOG_DIR,
    BENCHMARK_DIR,
    VALIDATION_DIR
]:

    os.makedirs(
        path,
        exist_ok=True
    )


print("\n✓ Project paths configured")


# ============================================================
# 5. VERIFY DATASET
# ============================================================

print("\n" + "=" * 100)
print("DATASET VERIFICATION")
print("=" * 100)

if not os.path.isfile(DATA_YAML):

    raise FileNotFoundError(
        f"data.yaml not found:\n{DATA_YAML}"
    )

if not os.path.isdir(DATASET_ROOT):

    raise FileNotFoundError(
        f"Dataset directory not found:\n{DATASET_ROOT}"
    )

print("✓ Dataset found")
print(f"  data.yaml : {DATA_YAML}")
print(f"  root      : {DATASET_ROOT}")


# ============================================================
# 6. LOAD MODEL REGISTRY FROM STEP 1C
# ============================================================

print("\n" + "=" * 100)
print("LOADING MODEL REGISTRY")
print("=" * 100)

if not os.path.isfile(MODEL_PATHS_FILE):

    raise FileNotFoundError(
        f"model_paths.json not found:\n{MODEL_PATHS_FILE}\n\n"
        "Run Step 1C first."
    )


with open(
    MODEL_PATHS_FILE,
    "r",
    encoding="utf-8"
) as f:

    MODEL_PATHS = json.load(f)


print(
    f"✓ Model registry loaded:\n"
    f"  {MODEL_PATHS_FILE}"
)


print("\nRegistered models:")

for name, path in MODEL_PATHS.items():

    print(
        f"  {name:<20} → {path}"
    )


# ============================================================
# 7. STANDARDIZED EXPERIMENT SETTINGS
# ============================================================

print("\n" + "=" * 100)
print("EXPERIMENT CONFIGURATION")
print("=" * 100)


# ------------------------------------------------------------
# Input resolution
# ------------------------------------------------------------

IMGSZ = 640


# ------------------------------------------------------------
# Benchmark settings
# ------------------------------------------------------------

WARMUP_RUNS = 10

BENCHMARK_RUNS = 50


# ------------------------------------------------------------
# Detection settings
# ------------------------------------------------------------

CONF = 0.25

IOU = 0.45


# ------------------------------------------------------------
# ONNX settings
# ------------------------------------------------------------

ONNX_OPSET = 16


print(f"Input size       : {IMGSZ} × {IMGSZ}")
print(f"Device           : {DEVICE}")
print(f"Warmup runs      : {WARMUP_RUNS}")
print(f"Benchmark runs   : {BENCHMARK_RUNS}")
print(f"Confidence       : {CONF}")
print(f"IoU              : {IOU}")
print(f"ONNX opset       : {ONNX_OPSET}")


# ============================================================
# 8. FIND ONE CONTROLLED TEST IMAGE
# ============================================================

def find_one_test_image():

    import yaml

    with open(
        DATA_YAML,
        "r",
        encoding="utf-8"
    ) as f:

        data = yaml.safe_load(f)


    test_value = data.get(
        "test",
        None
    )


    candidates = []


    # --------------------------------------------------------
    # Candidate from data.yaml
    # --------------------------------------------------------

    if test_value:

        test_path = Path(
            test_value
        )

        if not test_path.is_absolute():

            test_path = (
                Path(DATASET_ROOT)
                / test_path
            )

        candidates.append(
            test_path
        )


    # --------------------------------------------------------
    # Standard YOLO dataset structure
    # --------------------------------------------------------

    candidates.extend([

        Path(DATASET_ROOT)
        / "test"
        / "images",

        Path(DATASET_ROOT)
        / "test",

        Path(DATASET_ROOT)
        / "valid"
        / "images",

        Path(DATASET_ROOT)
        / "val"
        / "images",

    ])


    image_extensions = {

        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".webp"

    }


    # --------------------------------------------------------
    # Search only candidate directories
    # --------------------------------------------------------

    for candidate in candidates:

        if candidate.is_file():

            if (
                candidate.suffix.lower()
                in image_extensions
            ):

                return str(candidate)


        if candidate.is_dir():

            for p in candidate.iterdir():

                if (

                    p.is_file()

                    and

                    p.suffix.lower()
                    in image_extensions

                ):

                    return str(p)


    raise FileNotFoundError(
        "Could not locate a test image using "
        "data.yaml or the standard dataset structure."
    )


TEST_IMAGE = find_one_test_image()


print("\n✓ Controlled benchmark image:")
print(f"  {TEST_IMAGE}")


# ============================================================
# 9. GPU SYNCHRONIZATION HELPER
# ============================================================

def synchronize_gpu():

    """
    Synchronize CUDA before/after timing.

    CUDA operations are asynchronous.
    Without synchronization, measured latency can be misleading.
    """

    if torch.cuda.is_available():

        torch.cuda.synchronize(
            GPU_INDEX
        )


# ============================================================
# 10. GPU MODEL BENCHMARK FUNCTION
# ============================================================

def benchmark_model_gpu(
    model,
    image_path
):

    """
    Benchmark PyTorch FP32 inference on CUDA.

    NOTE:
    The timing here measures the complete Ultralytics
    predict() call, including preprocessing/postprocessing
    overhead as observed from Python.

    GPU synchronization is used so that CUDA execution
    is actually included in the measurement.
    """


    # --------------------------------------------------------
    # Warmup
    # --------------------------------------------------------

    print(
        f"  → Warmup: "
        f"{WARMUP_RUNS} runs..."
    )


    for _ in range(
        WARMUP_RUNS
    ):

        model.predict(

            source=image_path,

            imgsz=IMGSZ,

            conf=CONF,

            iou=IOU,

            device=DEVICE,

            verbose=False,

            save=False,

            show=False

        )


    # Make sure all GPU operations have completed
    synchronize_gpu()


    # --------------------------------------------------------
    # Benchmark
    # --------------------------------------------------------

    print(
        f"  → Benchmarking: "
        f"{BENCHMARK_RUNS} runs..."
    )


    times_ms = []


    for _ in range(
        BENCHMARK_RUNS
    ):


        synchronize_gpu()


        start = time.perf_counter()


        model.predict(

            source=image_path,

            imgsz=IMGSZ,

            conf=CONF,

            iou=IOU,

            device=DEVICE,

            verbose=False,

            save=False,

            show=False

        )


        synchronize_gpu()


        end = time.perf_counter()


        elapsed_ms = (
            end - start
        ) * 1000.0


        times_ms.append(
            elapsed_ms
        )


    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    mean_latency = float(
        np.mean(times_ms)
    )

    median_latency = float(
        np.median(times_ms)
    )

    std_latency = float(
        np.std(times_ms)
    )

    min_latency = float(
        np.min(times_ms)
    )

    max_latency = float(
        np.max(times_ms)
    )


    fps = (
        1000.0
        / mean_latency
    )


    return {

        "latency_ms_mean":
            mean_latency,

        "latency_ms_median":
            median_latency,

        "latency_ms_std":
            std_latency,

        "latency_ms_min":
            min_latency,

        "latency_ms_max":
            max_latency,

        "fps":
            fps,

        "warmup_runs":
            WARMUP_RUNS,

        "benchmark_runs":
            BENCHMARK_RUNS,

    }


# ============================================================
# 11. PHASE A — FP32 GPU BASELINE
# ============================================================

print("\n" + "=" * 100)
print("PHASE A — FP32 GPU BASELINE")
print("=" * 100)


baseline_results = []


for model_name, model_path in MODEL_PATHS.items():


    print(
        "\n" + "-" * 100
    )

    print(
        f"MODEL: {model_name}"
    )

    print(
        "-" * 100
    )


    if not os.path.isfile(
        model_path
    ):

        print(
            f"✗ Checkpoint not found:\n"
            f"  {model_path}"
        )

        continue


    try:


        # ----------------------------------------------------
        # Load model
        # ----------------------------------------------------

        print(
            "→ Loading model..."
        )


        model = YOLO(
            model_path
        )


        # ----------------------------------------------------
        # Move model to GPU
        # ----------------------------------------------------

        print(
            "→ Moving model to CUDA..."
        )


        model.to(
            DEVICE
        )


        # ----------------------------------------------------
        # Verify model device
        # ----------------------------------------------------

        actual_device = str(

            next(
                model.model.parameters()
            ).device

        )


        # ----------------------------------------------------
        # Parameter count
        # ----------------------------------------------------

        parameters = sum(

            p.numel()

            for p in model.model.parameters()

        )


        # ----------------------------------------------------
        # Checkpoint size
        # ----------------------------------------------------

        checkpoint_size_mb = (
            os.path.getsize(
                model_path
            )
            / (1024 ** 2)
        )


        print(
            f"✓ Model loaded"
        )

        print(
            f"  Device     : "
            f"{actual_device}"
        )

        print(
            f"  Parameters : "
            f"{parameters:,}"
        )

        print(
            f"  Size       : "
            f"{checkpoint_size_mb:.2f} MB"
        )


        # ----------------------------------------------------
        # Benchmark
        # ----------------------------------------------------

        benchmark = benchmark_model_gpu(

            model,

            TEST_IMAGE

        )


        print(
            "\n  FP32 GPU results:"
        )

        print(
            f"    Mean latency   : "
            f"{benchmark['latency_ms_mean']:.3f} ms"
        )

        print(
            f"    Median latency : "
            f"{benchmark['latency_ms_median']:.3f} ms"
        )

        print(
            f"    Std latency    : "
            f"{benchmark['latency_ms_std']:.3f} ms"
        )

        print(
            f"    FPS            : "
            f"{benchmark['fps']:.2f}"
        )


        # ----------------------------------------------------
        # Store result
        # ----------------------------------------------------

        baseline_results.append({

            "model":
                model_name,

            "model_path":
                model_path,

            "parameters":
                parameters,

            "model_size_mb":
                checkpoint_size_mb,

            "input_size":
                IMGSZ,

            "precision":
                "FP32",

            "device":
                actual_device,

            "gpu":
                GPU_NAME,

            "gpu_memory_gb":
                GPU_MEMORY_GB,

            "pytorch_version":
                torch.__version__,

            "cuda_version":
                str(
                    torch.version.cuda
                ),

            **benchmark

        })


        # ----------------------------------------------------
        # Free model
        # ----------------------------------------------------

        del model

        torch.cuda.empty_cache()

        synchronize_gpu()


    except Exception as e:


        print(
            f"\n✗ ERROR: "
            f"{repr(e)}"
        )


        baseline_results.append({

            "model":
                model_name,

            "model_path":
                model_path,

            "device":
                DEVICE,

            "status":
                "failed",

            "error":
                str(e)

        })


# ============================================================
# 12. SAVE FP32 GPU BASELINE
# ============================================================

baseline_df = pd.DataFrame(
    baseline_results
)


baseline_csv = os.path.join(

    BENCHMARK_DIR,

    "step2_fp32_gpu_baseline.csv"

)


baseline_json = os.path.join(

    LOG_DIR,

    "step2_fp32_gpu_baseline.json"

)


baseline_df.to_csv(

    baseline_csv,

    index=False

)


with open(

    baseline_json,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        baseline_results,

        f,

        indent=4,

        ensure_ascii=False,

        default=str

    )


print(
    "\n✓ FP32 GPU baseline saved:"
)

print(
    f"  CSV  : {baseline_csv}"
)

print(
    f"  JSON : {baseline_json}"
)


# ============================================================
# 13. PHASE B — ONNX FP32 EXPORT
# ============================================================

print("\n" + "=" * 100)
print("PHASE B — ONNX FP32 EXPORT")
print("=" * 100)


onnx_results = []


for model_name, model_path in MODEL_PATHS.items():


    print(
        "\n" + "-" * 100
    )

    print(
        f"EXPORTING: {model_name}"
    )

    print(
        "-" * 100
    )


    if not os.path.isfile(
        model_path
    ):

        print(
            "✗ Checkpoint missing"
        )

        continue


    try:


        # ----------------------------------------------------
        # Load model
        # ----------------------------------------------------

        model = YOLO(
            model_path
        )


        # ----------------------------------------------------
        # Export
        # ----------------------------------------------------

        print(
            "→ Exporting ONNX FP32..."
        )

        print(
            f"  Device : {DEVICE}"
        )

        print(
            f"  Opset  : {ONNX_OPSET}"
        )


        exported_path = model.export(

            format="onnx",

            imgsz=IMGSZ,

            device=GPU_INDEX,

            half=False,

            simplify=True,

            opset=ONNX_OPSET,

            dynamic=False,

            nms=False

        )


        exported_path = str(
            exported_path
        )


        # ----------------------------------------------------
        # Destination
        # ----------------------------------------------------

        destination = os.path.join(

            ONNX_DIR,

            f"{model_name}_fp32.onnx"

        )


        # ----------------------------------------------------
        # Copy export to optimization directory
        # ----------------------------------------------------

        shutil.copy2(

            exported_path,

            destination

        )


        # ----------------------------------------------------
        # Size
        # ----------------------------------------------------

        size_mb = (

            os.path.getsize(
                destination
            )
            / (1024 ** 2)

        )


        print(
            "\n✓ Export successful"
        )

        print(
            f"  ONNX : {destination}"
        )

        print(
            f"  Size : {size_mb:.2f} MB"
        )


        onnx_results.append({

            "model":
                model_name,

            "source_pt":
                model_path,

            "onnx_path":
                destination,

            "onnx_size_mb":
                size_mb,

            "precision":
                "FP32",

            "opset":
                ONNX_OPSET,

            "export_device":
                DEVICE,

            "status":
                "export_success"

        })


        # ----------------------------------------------------
        # Cleanup
        # ----------------------------------------------------

        del model

        torch.cuda.empty_cache()

        synchronize_gpu()


    except Exception as e:


        print(
            f"\n✗ EXPORT ERROR:"
        )

        print(
            repr(e)
        )


        onnx_results.append({

            "model":
                model_name,

            "source_pt":
                model_path,

            "precision":
                "FP32",

            "opset":
                ONNX_OPSET,

            "export_device":
                DEVICE,

            "status":
                "export_failed",

            "error":
                str(e)

        })


# ============================================================
# 14. PHASE C — ONNX STRUCTURAL VALIDATION
# ============================================================

print("\n" + "=" * 100)
print("PHASE C — ONNX STRUCTURAL + RUNTIME VALIDATION")
print("=" * 100)


for result in onnx_results:


    if result.get(
        "status"
    ) != "export_success":

        continue


    model_name = result[
        "model"
    ]

    onnx_path = result[
        "onnx_path"
    ]


    print(
        "\n" + "-" * 100
    )

    print(
        f"VALIDATING: {model_name}"
    )

    print(
        "-" * 100
    )


    try:


        # ----------------------------------------------------
        # Load ONNX
        # ----------------------------------------------------

        print(
            "→ Loading ONNX model..."
        )


        onnx_model = onnx.load(
            onnx_path
        )


        # ----------------------------------------------------
        # Structural check
        # ----------------------------------------------------

        print(
            "→ Running ONNX checker..."
        )


        onnx.checker.check_model(
            onnx_model
        )


        print(
            "✓ ONNX structural check passed"
        )


        # ----------------------------------------------------
        # ONNX Runtime providers
        # ----------------------------------------------------

        available_providers = (
            ort.get_available_providers()
        )


        print(
            "\n  ONNX Runtime providers:"
        )

        for provider in available_providers:

            print(
                f"    - {provider}"
            )


        # ----------------------------------------------------
        # Create CPU session
        #
        # This stage checks whether ONNX Runtime can
        # successfully load the graph.
        #
        # We do NOT claim this is a GPU benchmark.
        # ----------------------------------------------------

        print(
            "\n→ Creating ONNX Runtime session..."
        )


        session = ort.InferenceSession(

            onnx_path,

            providers=[
                "CPUExecutionProvider"
            ]

        )


        inputs = (
            session.get_inputs()
        )

        outputs = (
            session.get_outputs()
        )


        print(
            "✓ ONNX Runtime session created"
        )

        print(
            f"  Inputs  : "
            f"{len(inputs)}"
        )

        print(
            f"  Outputs : "
            f"{len(outputs)}"
        )


        # ----------------------------------------------------
        # Input information
        # ----------------------------------------------------

        if inputs:

            print(
                "\n  Input tensor:"
            )

            print(
                f"    Name : "
                f"{inputs[0].name}"
            )

            print(
                f"    Shape: "
                f"{inputs[0].shape}"
            )

            print(
                f"    Type : "
                f"{inputs[0].type}"
            )


        # ----------------------------------------------------
        # Output information
        # ----------------------------------------------------

        if outputs:

            print(
                "\n  Output tensor:"
            )

            print(
                f"    Name : "
                f"{outputs[0].name}"
            )

            print(
                f"    Shape: "
                f"{outputs[0].shape}"
            )

            print(
                f"    Type : "
                f"{outputs[0].type}"
            )


        result[
            "onnx_runtime"
        ] = "success"


        result[
            "runtime_provider"
        ] = "CPUExecutionProvider"


        result[
            "status"
        ] = "validated"


        del session

        del onnx_model


    except Exception as e:


        print(
            f"\n✗ VALIDATION ERROR:"
        )

        print(
            repr(e)
        )


        result[
            "onnx_runtime"
        ] = "failed"


        result[
            "validation_error"
        ] = str(e)


        result[
            "status"
        ] = "validation_failed"


# ============================================================
# 15. SAVE ONNX RESULTS
# ============================================================

onnx_df = pd.DataFrame(
    onnx_results
)


onnx_csv = os.path.join(

    VALIDATION_DIR,

    "step2_onnx_fp32_validation.csv"

)


onnx_json = os.path.join(

    LOG_DIR,

    "step2_onnx_fp32_results.json"

)


onnx_df.to_csv(

    onnx_csv,

    index=False

)


with open(

    onnx_json,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        onnx_results,

        f,

        indent=4,

        ensure_ascii=False,

        default=str

    )


print(
    "\n✓ ONNX results saved:"
)

print(
    f"  CSV  : {onnx_csv}"
)

print(
    f"  JSON : {onnx_json}"
)


# ============================================================
# 16. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("STEP 2 — FINAL SUMMARY")
print("=" * 100)


# ------------------------------------------------------------
# FP32 summary
# ------------------------------------------------------------

print(
    "\nFP32 GPU BASELINE"
)

print(
    "-" * 100
)


if not baseline_df.empty:

    columns = [

        "model",

        "parameters",

        "model_size_mb",

        "latency_ms_mean",

        "latency_ms_median",

        "fps",

        "device",

    ]


    available_columns = [

        c for c in columns

        if c in baseline_df.columns

    ]


    display(

        baseline_df[
            available_columns
        ]

    )


# ------------------------------------------------------------
# ONNX summary
# ------------------------------------------------------------

print(
    "\nONNX FP32"
)

print(
    "-" * 100
)


if not onnx_df.empty:

    columns = [

        "model",

        "onnx_size_mb",

        "opset",

        "status",

    ]


    available_columns = [

        c for c in columns

        if c in onnx_df.columns

    ]


    display(

        onnx_df[
            available_columns
        ]

    )


# ============================================================
# 17. SAVE STEP 2 ENVIRONMENT METADATA
# ============================================================

STEP2_METADATA = {

    "step":
        "2",

    "description":
        "FP32 GPU baseline and ONNX FP32 export",

    "timestamp":
        time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),

    "execution_environment":
        "Google Colab GPU",

    "device":
        DEVICE,

    "gpu":
        GPU_NAME,

    "gpu_memory_gb":
        GPU_MEMORY_GB,

    "pytorch_version":
        torch.__version__,

    "cuda_version":
        str(
            torch.version.cuda
        ),

    "input_size":
        IMGSZ,

    "confidence":
        CONF,

    "iou":
        IOU,

    "warmup_runs":
        WARMUP_RUNS,

    "benchmark_runs":
        BENCHMARK_RUNS,

    "onnx_opset":
        ONNX_OPSET,

    "test_image":
        TEST_IMAGE,

    "models":
        list(
            MODEL_PATHS.keys()
        ),

}


metadata_path = os.path.join(

    LOG_DIR,

    "step2_gpu_metadata.json"

)


with open(

    metadata_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        STEP2_METADATA,

        f,

        indent=4,

        ensure_ascii=False,

        default=str

    )


print(
    "\n✓ Step 2 metadata saved:"
)

print(
    f"  {metadata_path}"
)


# ============================================================
# 18. FINAL GPU MEMORY CLEANUP
# ============================================================

torch.cuda.empty_cache()

synchronize_gpu()


allocated_gb = (

    torch.cuda.memory_allocated(
        GPU_INDEX
    )
    / (1024 ** 3)

)

reserved_gb = (

    torch.cuda.memory_reserved(
        GPU_INDEX
    )
    / (1024 ** 3)

)


print(
    "\n" + "=" * 100
)

print(
    "FINAL GPU MEMORY STATUS"
)

print(
    "=" * 100
)

print(
    f"GPU allocated : "
    f"{allocated_gb:.3f} GB"
)

print(
    f"GPU reserved  : "
    f"{reserved_gb:.3f} GB"
)


# ============================================================
# FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "STEP 2 COMPLETED"
)

print(
    "=" * 100
)


print(
    """
The following stages have been performed:

    ✓ FP32 baseline on Tesla T4 GPU
    ✓ GPU-synchronized latency measurements
    ✓ ONNX FP32 export
    ✓ ONNX structural validation
    ✓ ONNX Runtime loading validation
    ✓ Experimental metadata saved

IMPORTANT:

The ONNX Runtime validation above confirms that the exported
ONNX graphs can be loaded successfully.

It is NOT yet the .pt-vs-ONNX dataset-level accuracy test.

That is the NEXT stage.

NEXT:

    STEP 2D
        ↓
    .pt FP32 vs ONNX FP32
    dataset-level equivalence
        ↓
    Compare:
        • Precision
        • Recall
        • mAP50
        • mAP50-95
        • inference speed
        ↓
    If equivalence is acceptable
        ↓
    FP16 optimization
"""
)

print(
    "\n✓ READY FOR STEP 2D"
)

STEP 2 — GPU FP32 BASELINE + ONNX FP32 EXPORT + VERIFICATION

GPU / CUDA VERIFICATION

PyTorch version : 2.11.0+cu128
CUDA available  : True
CUDA version    : 12.8
GPU             : Tesla T4
GPU memory      : 14.56 GB
Device          : cuda:0

✓ CUDA GPU verified

✓ Project paths configured

DATASET VERIFICATION
✓ Dataset found
  data.yaml : /content/drive/MyDrive/PFE/Dataset/Vehicles_Dataset/data.yaml
  root      : /content/drive/MyDrive/PFE/Dataset/Vehicles_Dataset

LOADING MODEL REGISTRY
✓ Model registry loaded:
  /content/drive/MyDrive/Stage_PFA/MODEL_OPTIMIZATION/09_logs/model_paths.json

Registered models:
  yolov8n_50ep         → /content/drive/MyDrive/Stage_PFA/experiments/Exp01_YOLOV8N/weights/best.pt
  yolov8n_75ep         → /content/drive/MyDrive/Stage_PFA/experiments/Exp02_YOLOV8N/weights/best.pt
  yolov8n_100ep        → /content/drive/MyDrive/Stage_PFA/experiments/Exp07B_YOLOV8_EPOCHS_100/weights/best.pt
  yolo11n              → /content/drive/MyDrive/Stage_PFA/experiments

,model,parameters,model_size_mb,latency_ms_mean,latency_ms_median,fps,device
0,yolov8n_50ep,3011823,5.951639,18.582443,17.668240,53.814236,cuda:0
1,yolov8n_75ep,3011823,5.964762,20.753702,21.158468,48.184174,cuda:0
2,yolov8n_100ep,3011823,5.956278,15.381921,14.937566,65.011386,cuda:0
3,yolo11n,2590815,5.214502,18.547036,18.225916,53.916970,cuda:0
4,rtdetr_50ep,32816351,63.165586,58.339249,57.195356,17.141119,cuda:0



ONNX FP32
----------------------------------------------------------------------------------------------------


,model,onnx_size_mb,opset,status
0,yolov8n_50ep,11.700084,16,validated
1,yolov8n_75ep,11.700084,16,validated
2,yolov8n_100ep,11.700084,16,validated
3,yolo11n,10.115925,16,validated
4,rtdetr_50ep,125.526609,16,validated



✓ Step 2 metadata saved:
  /content/drive/MyDrive/Stage_PFA/MODEL_OPTIMIZATION/09_logs/step2_gpu_metadata.json

FINAL GPU MEMORY STATUS
GPU allocated : 0.123 GB
GPU reserved  : 0.154 GB

STEP 2 COMPLETED

The following stages have been performed:

    ✓ FP32 baseline on Tesla T4 GPU
    ✓ GPU-synchronized latency measurements
    ✓ ONNX FP32 export
    ✓ ONNX structural validation
    ✓ ONNX Runtime loading validation
    ✓ Experimental metadata saved

IMPORTANT:

The ONNX Runtime validation above confirms that the exported
ONNX graphs can be loaded successfully.

It is NOT yet the .pt-vs-ONNX dataset-level accuracy test.

That is the NEXT stage.

NEXT:

    STEP 2D
        ↓
    .pt FP32 vs ONNX FP32
    dataset-level equivalence
        ↓
    Compare:
        • Precision
        • Recall
        • mAP50
        • mAP50-95
        • inference speed
        ↓
    If equivalence is acceptable
        ↓
    FP16 optimization


✓ READY FOR STEP 2D


In [ ]:
# ============================================================
# FIX — INSTALL CUDA 12.8 COMPATIBLE ONNX RUNTIME GPU
# ============================================================

!pip uninstall -y onnxruntime onnxruntime-gpu

!pip install -q "onnxruntime-gpu==1.26.0"

Found existing installation: onnxruntime 1.29.0
Uninstalling onnxruntime-1.29.0:
  Successfully uninstalled onnxruntime-1.29.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 1.7 MB/s eta 0:00:00


In [ ]:
# ============================================================
# ONNX RUNTIME GPU VERIFICATION
# ============================================================

import torch
import onnxruntime as ort

print("=" * 100)
print("GPU / ONNX RUNTIME VERIFICATION")
print("=" * 100)

print(f"\nPyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"CUDA version    : {torch.version.cuda}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory      : "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

print("\n" + "=" * 100)
print("ONNX RUNTIME")
print("=" * 100)

print(f"\nONNX Runtime version : {ort.__version__}")

providers = ort.get_available_providers()

print("\nAvailable ONNX Runtime providers:")

for provider in providers:
    print(f"  - {provider}")

if "CUDAExecutionProvider" not in providers:

    raise RuntimeError(
        "\nCUDAExecutionProvider is STILL NOT available.\n\n"
        "Do NOT continue with Step 2D.\n\n"
        f"Available providers:\n{providers}"
    )

print("\n✓ CUDAExecutionProvider is available")
print("✓ ONNX Runtime GPU is ready")

GPU / ONNX RUNTIME VERIFICATION

PyTorch version : 2.11.0+cu128
CUDA available  : True
CUDA version    : 12.8
GPU             : Tesla T4
GPU memory      : 14.56 GB

ONNX RUNTIME

ONNX Runtime version : 1.26.0

Available ONNX Runtime providers:
  - TensorrtExecutionProvider
  - CUDAExecutionProvider
  - CPUExecutionProvider

✓ CUDAExecutionProvider is available
✓ ONNX Runtime GPU is ready


In [ ]:
# ============================================================
# STEP 2D — GPU VERSION
# FP32 .PT vs ONNX FP32 DATASET-LEVEL EQUIVALENCE
#
# Purpose:
#   Compare original PyTorch FP32 models against their
#   exported ONNX FP32 versions using the SAME test dataset.
#
# GPU:
#   Tesla T4
#
# Metrics:
#   Precision
#   Recall
#   mAP50
#   mAP50-95
#   Inference time
#
# IMPORTANT:
#   ONNX Runtime MUST use CUDAExecutionProvider.
# ============================================================


# ============================================================
# 1. INSTALL / VERIFY REQUIRED PACKAGES
# ============================================================

# print("=" * 100)
# print("STEP 2D — GPU FP32 PT vs ONNX FP32")
# print("=" * 100)

# print("\n→ Installing required packages...")

# !pip -q install ultralytics onnx onnxruntime-gpu


# ============================================================
# 2. IMPORTS
# ============================================================

import os
import json
import time
import warnings

import numpy as np
import pandas as pd
import torch
import onnxruntime as ort

from ultralytics import YOLO

warnings.filterwarnings("ignore")


# ============================================================
# 3. GPU / CUDA VERIFICATION
# ============================================================

print("\n")
print("=" * 100)
print("GPU / CUDA VERIFICATION")
print("=" * 100)

print(f"\nPyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"CUDA version    : {torch.version.cuda}")

if not torch.cuda.is_available():

    raise RuntimeError(
        "\nCUDA GPU is NOT available.\n"
        "Go to:\n"
        "Runtime → Change runtime type → GPU\n"
        "and reconnect the runtime."
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
)

print(f"GPU             : {GPU_NAME}")
print(f"GPU memory      : {GPU_MEMORY_GB:.2f} GB")

print("\n✓ PyTorch CUDA GPU verified")


# ============================================================
# 4. ONNX RUNTIME GPU VERIFICATION
# ============================================================

print("\n")
print("=" * 100)
print("ONNX RUNTIME GPU VERIFICATION")
print("=" * 100)

ORT_VERSION = ort.__version__

available_providers = ort.get_available_providers()

print(
    f"\nONNX Runtime version : {ORT_VERSION}"
)

print(
    "\nAvailable ONNX Runtime providers:"
)

for provider in available_providers:

    print(
        f"  - {provider}"
    )


# ------------------------------------------------------------
# CRITICAL CHECK
# ------------------------------------------------------------

if "CUDAExecutionProvider" not in available_providers:

    raise RuntimeError(
        "\nCUDAExecutionProvider is NOT available.\n\n"
        "ONNX Runtime is therefore NOT ready for GPU inference.\n\n"
        f"Available providers:\n{available_providers}"
    )


print(
    "\n✓ CUDAExecutionProvider is available"
)

print(
    "✓ ONNX Runtime GPU is ready"
)


# ============================================================
# 5. DEFINE ONNX GPU PROVIDERS
# ============================================================

ORT_PROVIDERS = [

    "CUDAExecutionProvider",

    "CPUExecutionProvider"

]

print("\n")
print("=" * 100)
print("ONNX RUNTIME PROVIDER CONFIGURATION")
print("=" * 100)

print(
    "\nPrimary provider:"
    "\n  CUDAExecutionProvider"
)

print(
    "\nFallback provider:"
    "\n  CPUExecutionProvider"
)


# ============================================================
# 6. PATH CONFIGURATION
# ============================================================

DATA_YAML = (
    "/content/drive/MyDrive/PFE/Dataset/"
    "Vehicles_Dataset/data.yaml"
)

OPT_ROOT = (
    "/content/drive/MyDrive/Stage_PFA/"
    "MODEL_OPTIMIZATION"
)

MODEL_PATHS_FILE = os.path.join(
    OPT_ROOT,
    "09_logs",
    "model_paths.json"
)

# IMPORTANT:
# Step 2 saved the ONNX models here.
ONNX_DIR = os.path.join(
    OPT_ROOT,
    "02_onnx_fp32"
)

LOG_DIR = os.path.join(
    OPT_ROOT,
    "09_logs"
)

RESULTS_DIR = os.path.join(
    OPT_ROOT,
    "02_fp32_baseline"
)

os.makedirs(
    LOG_DIR,
    exist_ok=True
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)


# ============================================================
# 7. EXPERIMENT CONFIGURATION
# ============================================================

IMGSZ = 640

# Safe starting batch size for Tesla T4
BATCH_SIZE = 8

# Automatic fallback
FALLBACK_BATCH_SIZE = 4

# Dataset split
SPLIT = "test"

# For mAP evaluation, low confidence is preferred
CONF = 0.001

# Standard IoU threshold for NMS during validation
IOU = 0.7

# Disable plots
PLOTS = False

# GPU
DEVICE = 0


print("\n")
print("=" * 100)
print("STEP 2D — EXPERIMENT CONFIGURATION")
print("=" * 100)

print(
    f"\nDataset      : {DATA_YAML}"
)

print(
    f"Split        : {SPLIT}"
)

print(
    f"Image size   : {IMGSZ} × {IMGSZ}"
)

print(
    f"Batch size   : {BATCH_SIZE}"
)

print(
    f"Fallback     : {FALLBACK_BATCH_SIZE}"
)

print(
    f"Device       : CUDA:{DEVICE}"
)

print(
    f"Confidence   : {CONF}"
)

print(
    f"IoU          : {IOU}"
)

print(
    f"ONNX Runtime : {ORT_VERSION}"
)

print(
    f"ORT Provider : CUDAExecutionProvider"
)


# ============================================================
# 8. VERIFY DATASET
# ============================================================

print("\n")
print("=" * 100)
print("DATASET VERIFICATION")
print("=" * 100)

if not os.path.exists(DATA_YAML):

    raise FileNotFoundError(
        f"\nDataset YAML not found:\n{DATA_YAML}"
    )

print(
    "\n✓ Dataset YAML found"
)

print(
    f"  {DATA_YAML}"
)


# ============================================================
# 9. VERIFY MODEL REGISTRY
# ============================================================

print("\n")
print("=" * 100)
print("LOADING MODEL REGISTRY")
print("=" * 100)

if not os.path.exists(MODEL_PATHS_FILE):

    raise FileNotFoundError(
        f"\nModel registry not found:\n"
        f"{MODEL_PATHS_FILE}"
    )

with open(
    MODEL_PATHS_FILE,
    "r",
    encoding="utf-8"
) as f:

    MODEL_PATHS = json.load(f)


print(
    "\n✓ Model registry loaded"
)

print(
    "\nRegistered models:"
)

for name, path in MODEL_PATHS.items():

    print(
        f"  {name:<20} → {path}"
    )


# ============================================================
# 10. DEFINE ONNX PATHS
# ============================================================

print("\n")
print("=" * 100)
print("ONNX MODEL REGISTRY")
print("=" * 100)

ONNX_PATHS = {}

for model_name in MODEL_PATHS.keys():

    lower_name = model_name.lower()

    if (
        "rtdetr" in lower_name
        or "rt-detr" in lower_name
    ):

        onnx_filename = (
            "rtdetr_50ep_fp32.onnx"
        )

    else:

        onnx_filename = (
            f"{model_name}_fp32.onnx"
        )

    ONNX_PATHS[model_name] = os.path.join(
        ONNX_DIR,
        onnx_filename
    )


for name, path in ONNX_PATHS.items():

    print(
        f"  {name:<20} → {path}"
    )


# ============================================================
# 11. VERIFY ALL ONNX FILES
# ============================================================

print("\n")
print("=" * 100)
print("ONNX FILE VERIFICATION")
print("=" * 100)

missing_onnx = []

for model_name, path in ONNX_PATHS.items():

    if os.path.exists(path):

        size_mb = (
            os.path.getsize(path)
            / (1024 ** 2)
        )

        print(
            f"\n✓ {model_name}"
        )

        print(
            f"  File : {path}"
        )

        print(
            f"  Size : {size_mb:.2f} MB"
        )

    else:

        print(
            f"\n✗ {model_name}"
        )

        print(
            f"  Missing: {path}"
        )

        missing_onnx.append(
            model_name
        )


if missing_onnx:

    raise FileNotFoundError(
        "\nThe following ONNX models are missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_onnx
        )
    )


print(
    "\n✓ All ONNX models found"
)


# ============================================================
# 12. EXPLICIT ONNX RUNTIME GPU TEST
# ============================================================

print("\n")
print("=" * 100)
print("ONNX RUNTIME CUDA SESSION TEST")
print("=" * 100)

# Test the first ONNX model
test_model_name = list(
    ONNX_PATHS.keys()
)[0]

test_onnx_path = ONNX_PATHS[
    test_model_name
]

print(
    f"\nTesting model:"
    f"\n  {test_model_name}"
)

print(
    f"\nCreating ONNX Runtime session..."
)

try:

    test_session = ort.InferenceSession(
        test_onnx_path,
        providers=ORT_PROVIDERS
    )

    active_providers = (
        test_session.get_providers()
    )

    print(
        "\nSession providers:"
    )

    for provider in active_providers:

        print(
            f"  - {provider}"
        )

    if (
        "CUDAExecutionProvider"
        not in active_providers
    ):

        raise RuntimeError(
            "\nCUDAExecutionProvider was requested "
            "but is NOT active in the ONNX Runtime session."
        )

    print(
        "\n✓ ONNX Runtime session is using CUDA"
    )

    del test_session

except Exception as e:

    raise RuntimeError(
        "\nONNX Runtime GPU session test failed.\n\n"
        f"Error:\n{e}"
    )


# ============================================================
# 13. METRIC EXTRACTION
# ============================================================

def extract_metrics(results):

    metrics = results.box

    output = {

        "precision": float(
            metrics.mp
        ),

        "recall": float(
            metrics.mr
        ),

        "map50": float(
            metrics.map50
        ),

        "map50_95": float(
            metrics.map
        )

    }

    # --------------------------------------------------------
    # Ultralytics timing
    # --------------------------------------------------------

    if hasattr(
        results,
        "speed"
    ):

        speed = results.speed

        output[
            "preprocess_ms"
        ] = float(
            speed.get(
                "preprocess",
                np.nan
            )
        )

        output[
            "inference_ms"
        ] = float(
            speed.get(
                "inference",
                np.nan
            )
        )

        output[
            "postprocess_ms"
        ] = float(
            speed.get(
                "postprocess",
                np.nan
            )
        )

    else:

        output[
            "preprocess_ms"
        ] = np.nan

        output[
            "inference_ms"
        ] = np.nan

        output[
            "postprocess_ms"
        ] = np.nan

    return output


# ============================================================
# 14. VALIDATION FUNCTION
# ============================================================

def evaluate_model(
    model_path,
    model_name,
    model_type
):

    print("\n")
    print("-" * 100)
    print(
        f"{model_type.upper()} — {model_name}"
    )
    print("-" * 100)

    if not os.path.exists(model_path):

        print(
            f"\n✗ File not found:"
            f"\n  {model_path}"
        )

        return {

            "model": model_name,

            "type": model_type,

            "path": model_path,

            "status": "file_not_found"

        }

    model = None
    results = None

    try:

        # ----------------------------------------------------
        # LOAD MODEL
        # ----------------------------------------------------

        print(
            "\n→ Loading model..."
        )

        model = YOLO(
            model_path
        )

        print(
            "✓ Model loaded"
        )

        # ----------------------------------------------------
        # ONNX PROVIDER INFORMATION
        # ----------------------------------------------------

        active_ort_providers = None

        if model_type == "ONNX_FP32":

            try:

                session = ort.InferenceSession(
                    model_path,
                    providers=ORT_PROVIDERS
                )

                active_ort_providers = (
                    session.get_providers()
                )

                print(
                    "\n✓ ONNX Runtime providers:"
                )

                for provider in (
                    active_ort_providers
                ):

                    print(
                        f"  - {provider}"
                    )

                if (
                    "CUDAExecutionProvider"
                    not in active_ort_providers
                ):

                    raise RuntimeError(
                        "CUDAExecutionProvider "
                        "is not active."
                    )

                print(
                    "\n✓ ONNX inference backend "
                    "is configured for GPU"
                )

                del session

            except Exception as e:

                print(
                    "\n✗ ONNX Runtime CUDA "
                    "session verification failed:"
                )

                print(
                    str(e)
                )

                return {

                    "model": model_name,

                    "type": model_type,

                    "path": model_path,

                    "status":
                        "onnx_gpu_session_failed",

                    "error":
                        str(e)

                }

        # ----------------------------------------------------
        # CLEAR GPU
        # ----------------------------------------------------

        torch.cuda.empty_cache()

        torch.cuda.synchronize()

        print(
            "\n→ Running dataset validation..."
        )

        print(
            "  Device : CUDA:0"
        )

        print(
            f"  Batch  : {BATCH_SIZE}"
        )

        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------

        start_time = time.perf_counter()

        results = model.val(

            data=DATA_YAML,

            split=SPLIT,

            imgsz=IMGSZ,

            batch=BATCH_SIZE,

            device=DEVICE,

            conf=CONF,

            iou=IOU,

            plots=PLOTS,

            verbose=False

        )

        torch.cuda.synchronize()

        total_time = (
            time.perf_counter()
            - start_time
        )

        # ----------------------------------------------------
        # EXTRACT METRICS
        # ----------------------------------------------------

        metrics = extract_metrics(
            results
        )

        print(
            "\n✓ Validation completed"
        )

        print(
            f"\n  Precision : "
            f"{metrics['precision']:.6f}"
        )

        print(
            f"  Recall    : "
            f"{metrics['recall']:.6f}"
        )

        print(
            f"  mAP50     : "
            f"{metrics['map50']:.6f}"
        )

        print(
            f"  mAP50-95  : "
            f"{metrics['map50_95']:.6f}"
        )

        print(
            f"  Preprocess: "
            f"{metrics['preprocess_ms']:.3f} ms"
        )

        print(
            f"  Inference : "
            f"{metrics['inference_ms']:.3f} ms"
        )

        print(
            f"  Postprocess:"
            f" {metrics['postprocess_ms']:.3f} ms"
        )

        print(
            f"  Total     : "
            f"{total_time:.2f} s"
        )

        # ----------------------------------------------------
        # RESULT
        # ----------------------------------------------------

        result = {

            "model": model_name,

            "type": model_type,

            "path": model_path,

            "status": "success",

            "device": "CUDA:0",

            "gpu": GPU_NAME,

            "batch_size": BATCH_SIZE,

            "imgsz": IMGSZ,

            "confidence": CONF,

            "iou": IOU,

            "ort_version":
                ORT_VERSION,

            "ort_providers":
                active_ort_providers,

            **metrics,

            "validation_total_seconds":
                total_time

        }

        return result

    except RuntimeError as e:

        # ----------------------------------------------------
        # GPU OOM
        # ----------------------------------------------------

        if (
            "out of memory"
            in str(e).lower()
        ):

            print(
                "\n⚠️ GPU OUT OF MEMORY"
            )

            print(
                f"Retrying {model_name} "
                f"with batch={FALLBACK_BATCH_SIZE}..."
            )

            try:

                torch.cuda.empty_cache()

                del model

                model = YOLO(
                    model_path
                )

                start_time = (
                    time.perf_counter()
                )

                results = model.val(

                    data=DATA_YAML,

                    split=SPLIT,

                    imgsz=IMGSZ,

                    batch=FALLBACK_BATCH_SIZE,

                    device=DEVICE,

                    conf=CONF,

                    iou=IOU,

                    plots=PLOTS,

                    verbose=False

                )

                torch.cuda.synchronize()

                total_time = (
                    time.perf_counter()
                    - start_time
                )

                metrics = extract_metrics(
                    results
                )

                print(
                    "\n✓ Validation completed "
                    f"with batch={FALLBACK_BATCH_SIZE}"
                )

                print(
                    f"  Precision : "
                    f"{metrics['precision']:.6f}"
                )

                print(
                    f"  Recall    : "
                    f"{metrics['recall']:.6f}"
                )

                print(
                    f"  mAP50     : "
                    f"{metrics['map50']:.6f}"
                )

                print(
                    f"  mAP50-95  : "
                    f"{metrics['map50_95']:.6f}"
                )

                return {

                    "model": model_name,

                    "type": model_type,

                    "path": model_path,

                    "status": "success",

                    "device": "CUDA:0",

                    "gpu": GPU_NAME,

                    "batch_size":
                        FALLBACK_BATCH_SIZE,

                    "imgsz": IMGSZ,

                    "confidence": CONF,

                    "iou": IOU,

                    "ort_version":
                        ORT_VERSION,

                    "ort_providers":
                        (
                            active_ort_providers
                            if model_type
                            == "ONNX_FP32"
                            else None
                        ),

                    **metrics,

                    "validation_total_seconds":
                        total_time

                }

            except Exception as retry_error:

                print(
                    "\n✗ Retry failed:"
                )

                print(
                    str(retry_error)
                )

                torch.cuda.empty_cache()

                return {

                    "model": model_name,

                    "type": model_type,

                    "path": model_path,

                    "status":
                        "validation_failed",

                    "error":
                        str(retry_error)

                }

        else:

            print(
                "\n✗ Runtime error:"
            )

            print(
                str(e)
            )

            torch.cuda.empty_cache()

            return {

                "model": model_name,

                "type": model_type,

                "path": model_path,

                "status":
                    "validation_failed",

                "error":
                    str(e)

            }

    except Exception as e:

        print(
            "\n✗ VALIDATION ERROR:"
        )

        print(
            str(e)
        )

        torch.cuda.empty_cache()

        return {

            "model": model_name,

            "type": model_type,

            "path": model_path,

            "status":
                "validation_failed",

            "error":
                str(e)

        }

    finally:

        try:

            del model

        except:

            pass

        try:

            del results

        except:

            pass

        torch.cuda.empty_cache()


# ============================================================
# 15. PHASE A — ORIGINAL .PT MODELS
# ============================================================

print("\n")
print("=" * 100)
print("PHASE A — ORIGINAL .PT FP32 MODELS ON TESLA T4")
print("=" * 100)

pt_results = []

for model_name, model_path in MODEL_PATHS.items():

    result = evaluate_model(

        model_path=model_path,

        model_name=model_name,

        model_type="PT_FP32"

    )

    pt_results.append(
        result
    )


# ============================================================
# 16. PHASE B — ONNX FP32 MODELS
# ============================================================

print("\n")
print("=" * 100)
print("PHASE B — ONNX FP32 MODELS ON GPU")
print("=" * 100)

onnx_results = []

for model_name, onnx_path in ONNX_PATHS.items():

    result = evaluate_model(

        model_path=onnx_path,

        model_name=model_name,

        model_type="ONNX_FP32"

    )

    onnx_results.append(
        result
    )


# ============================================================
# 17. CREATE DATAFRAMES
# ============================================================

pt_df = pd.DataFrame(
    pt_results
)

onnx_df = pd.DataFrame(
    onnx_results
)


# ============================================================
# 18. PT vs ONNX COMPARISON
# ============================================================

print("\n")
print("=" * 100)
print("PT FP32 vs ONNX FP32 COMPARISON")
print("=" * 100)

comparison_rows = []


for model_name in MODEL_PATHS.keys():

    pt_match = pt_df[
        pt_df["model"] == model_name
    ]

    onnx_match = onnx_df[
        onnx_df["model"] == model_name
    ]

    if (
        pt_match.empty
        or onnx_match.empty
    ):

        continue

    pt = pt_match.iloc[0]
    onnx = onnx_match.iloc[0]

    row = {

        "model":
            model_name,

        # ----------------------------------------------------
        # STATUS
        # ----------------------------------------------------

        "pt_status":
            pt.get(
                "status",
                "unknown"
            ),

        "onnx_status":
            onnx.get(
                "status",
                "unknown"
            ),

        # ----------------------------------------------------
        # PT METRICS
        # ----------------------------------------------------

        "pt_precision":
            pt.get(
                "precision",
                np.nan
            ),

        "pt_recall":
            pt.get(
                "recall",
                np.nan
            ),

        "pt_map50":
            pt.get(
                "map50",
                np.nan
            ),

        "pt_map50_95":
            pt.get(
                "map50_95",
                np.nan
            ),

        "pt_inference_ms":
            pt.get(
                "inference_ms",
                np.nan
            ),

        # ----------------------------------------------------
        # ONNX METRICS
        # ----------------------------------------------------

        "onnx_precision":
            onnx.get(
                "precision",
                np.nan
            ),

        "onnx_recall":
            onnx.get(
                "recall",
                np.nan
            ),

        "onnx_map50":
            onnx.get(
                "map50",
                np.nan
            ),

        "onnx_map50_95":
            onnx.get(
                "map50_95",
                np.nan
            ),

        "onnx_inference_ms":
            onnx.get(
                "inference_ms",
                np.nan
            )

    }


    # ========================================================
    # ABSOLUTE DELTAS
    # ========================================================

    row[
        "delta_precision"
    ] = (
        row["onnx_precision"]
        - row["pt_precision"]
    )

    row[
        "delta_recall"
    ] = (
        row["onnx_recall"]
        - row["pt_recall"]
    )

    row[
        "delta_map50"
    ] = (
        row["onnx_map50"]
        - row["pt_map50"]
    )

    row[
        "delta_map50_95"
    ] = (
        row["onnx_map50_95"]
        - row["pt_map50_95"]
    )

    row[
        "delta_inference_ms"
    ] = (
        row["onnx_inference_ms"]
        - row["pt_inference_ms"]
    )


    # ========================================================
    # PERCENTAGE CHANGES
    # ========================================================

    # Precision

    if (
        not pd.isna(
            row["pt_precision"]
        )
        and row["pt_precision"] != 0
    ):

        row[
            "precision_change_percent"
        ] = (

            row["delta_precision"]
            /
            row["pt_precision"]
            *
            100

        )

    else:

        row[
            "precision_change_percent"
        ] = np.nan


    # Recall

    if (
        not pd.isna(
            row["pt_recall"]
        )
        and row["pt_recall"] != 0
    ):

        row[
            "recall_change_percent"
        ] = (

            row["delta_recall"]
            /
            row["pt_recall"]
            *
            100

        )

    else:

        row[
            "recall_change_percent"
        ] = np.nan


    # mAP50

    if (
        not pd.isna(
            row["pt_map50"]
        )
        and row["pt_map50"] != 0
    ):

        row[
            "map50_change_percent"
        ] = (

            row["delta_map50"]
            /
            row["pt_map50"]
            *
            100

        )

    else:

        row[
            "map50_change_percent"
        ] = np.nan


    # mAP50-95

    if (
        not pd.isna(
            row["pt_map50_95"]
        )
        and row["pt_map50_95"] != 0
    ):

        row[
            "map50_95_change_percent"
        ] = (

            row["delta_map50_95"]
            /
            row["pt_map50_95"]
            *
            100

        )

    else:

        row[
            "map50_95_change_percent"
        ] = np.nan


    comparison_rows.append(
        row
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


# ============================================================
# 19. SAVE RAW RESULTS
# ============================================================

pt_csv = os.path.join(
    RESULTS_DIR,
    "step2d_gpu_pt_fp32_validation.csv"
)

onnx_csv = os.path.join(
    RESULTS_DIR,
    "step2d_gpu_onnx_fp32_validation.csv"
)

comparison_csv = os.path.join(
    RESULTS_DIR,
    "step2d_gpu_pt_vs_onnx_comparison.csv"
)


pt_df.to_csv(
    pt_csv,
    index=False
)

onnx_df.to_csv(
    onnx_csv,
    index=False
)

comparison_df.to_csv(
    comparison_csv,
    index=False
)


# ============================================================
# 20. SAVE COMPLETE JSON LOG
# ============================================================

experiment_log = {

    "experiment":
        "Step 2D — GPU FP32 PT vs ONNX FP32",

    "purpose":
        "Dataset-level equivalence verification",

    "dataset":
        DATA_YAML,

    "split":
        SPLIT,

    "imgsz":
        IMGSZ,

    "device":
        "CUDA:0",

    "gpu":
        GPU_NAME,

    "gpu_memory_gb":
        GPU_MEMORY_GB,

    "batch_size":
        BATCH_SIZE,

    "fallback_batch_size":
        FALLBACK_BATCH_SIZE,

    "confidence":
        CONF,

    "iou":
        IOU,

    "onnx_runtime_version":
        ORT_VERSION,

    "onnx_runtime_available_providers":
        available_providers,

    "onnx_runtime_requested_providers":
        ORT_PROVIDERS,

    "pt_results":
        pt_results,

    "onnx_results":
        onnx_results,

    "comparison":
        comparison_rows

}


json_path = os.path.join(
    LOG_DIR,
    "step2d_gpu_fp32_pt_vs_onnx.json"
)


with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        experiment_log,
        f,
        indent=4,
        default=str
    )


# ============================================================
# 21. FINAL COMPARISON TABLE
# ============================================================

print("\n")
print("=" * 100)
print("STEP 2D — GPU FINAL COMPARISON")
print("=" * 100)

if not comparison_df.empty:

    display(

        comparison_df[

            [

                "model",

                "pt_precision",
                "onnx_precision",

                "delta_precision",

                "pt_recall",
                "onnx_recall",

                "delta_recall",

                "pt_map50",
                "onnx_map50",

                "delta_map50",

                "pt_map50_95",
                "onnx_map50_95",

                "delta_map50_95",

                "pt_inference_ms",
                "onnx_inference_ms"

            ]

        ].round(6)

    )

else:

    print(
        "\n⚠️ No comparison results available."
    )


# ============================================================
# 22. EQUIVALENCE ANALYSIS
# ============================================================

print("\n")
print("=" * 100)
print("ONNX FP32 DATASET-LEVEL EQUIVALENCE ANALYSIS")
print("=" * 100)

print(
    """
Equivalence criterion used:

    |ΔmAP50-95| ≤ 0.005
        → Equivalent / negligible difference

    0.005 < |ΔmAP50-95| ≤ 0.010
        → Small difference

    |ΔmAP50-95| > 0.010
        → Significant difference

NOTE:
The criterion is used as a practical engineering threshold
for determining whether FP32 ONNX export preserved model
accuracy.
"""
)


if not comparison_df.empty:

    for _, row in comparison_df.iterrows():

        model_name = row[
            "model"
        ]

        delta = row[
            "delta_map50_95"
        ]

        delta_precision = row[
            "delta_precision"
        ]

        delta_recall = row[
            "delta_recall"
        ]

        delta_map50 = row[
            "delta_map50"
        ]


        if pd.isna(delta):

            status = (
                "⚠️ Unable to determine"
            )

        elif abs(delta) <= 0.005:

            status = (
                "✓ Equivalent — "
                "negligible difference"
            )

        elif abs(delta) <= 0.010:

            status = (
                "⚠️ Small difference"
            )

        else:

            status = (
                "❗ Significant difference"
            )


        print(
            f"\n{model_name}"
        )

        print(
            f"  ΔPrecision : "
            f"{delta_precision:+.6f}"
        )

        print(
            f"  ΔRecall    : "
            f"{delta_recall:+.6f}"
        )

        print(
            f"  ΔmAP50     : "
            f"{delta_map50:+.6f}"
        )

        print(
            f"  ΔmAP50-95  : "
            f"{delta:+.6f}"
        )

        print(
            f"  Assessment  : "
            f"{status}"
        )


# ============================================================
# 23. GPU MEMORY STATUS
# ============================================================

print("\n")
print("=" * 100)
print("FINAL GPU MEMORY STATUS")
print("=" * 100)

print(
    f"\nGPU allocated : "
    f"{torch.cuda.memory_allocated() / (1024 ** 3):.3f} GB"
)

print(
    f"GPU reserved  : "
    f"{torch.cuda.memory_reserved() / (1024 ** 3):.3f} GB"
)


# ============================================================
# 24. FILE SUMMARY
# ============================================================

print("\n")
print("=" * 100)
print("FILES SAVED")
print("=" * 100)

print(
    f"\nPT GPU validation:"
    f"\n  {pt_csv}"
)

print(
    f"\nONNX GPU validation:"
    f"\n  {onnx_csv}"
)

print(
    f"\nPT vs ONNX comparison:"
    f"\n  {comparison_csv}"
)

print(
    f"\nComplete experiment log:"
    f"\n  {json_path}"
)


# ============================================================
# 25. FINAL STATUS
# ============================================================

successful_pt = sum(

    r.get("status")
    == "success"

    for r in pt_results

)

successful_onnx = sum(

    r.get("status")
    == "success"

    for r in onnx_results

)


print("\n")
print("=" * 100)
print("STEP 2D COMPLETED")
print("=" * 100)

print(
    f"\nPT models successfully evaluated:"
    f" {successful_pt}/{len(MODEL_PATHS)}"
)

print(
    f"ONNX models successfully evaluated:"
    f" {successful_onnx}/{len(ONNX_PATHS)}"
)

print(
    "\nONNX Runtime GPU provider:"
)

print(
    "  CUDAExecutionProvider ✓"
)

print(
    """
============================================================

INTERPRETATION

This experiment establishes whether the FP32 ONNX exports
preserve the accuracy of the original FP32 PyTorch models
at the DATASET level.

The comparison includes:

    • Precision
    • Recall
    • mAP50
    • mAP50-95
    • Inference time

The original .pt checkpoints are NOT modified.

The previous Step 2 FP32 GPU benchmark is also preserved.

If the ONNX models maintain acceptable accuracy equivalence,
the next optimization stage can proceed to:

    STEP 3
        ↓
    FP16 OPTIMIZATION
        ↓
    FP16 accuracy verification
        ↓
    FP16 GPU benchmark
        ↓
    Comparison against FP32

IMPORTANT:

Do NOT proceed to FP16 until the complete Step 2D output
has been checked.

============================================================
"""
)



GPU / CUDA VERIFICATION

PyTorch version : 2.11.0+cu128
CUDA available  : True
CUDA version    : 12.8
GPU             : Tesla T4
GPU memory      : 14.56 GB

✓ PyTorch CUDA GPU verified


ONNX RUNTIME GPU VERIFICATION

ONNX Runtime version : 1.26.0

Available ONNX Runtime providers:
  - TensorrtExecutionProvider
  - CUDAExecutionProvider
  - CPUExecutionProvider

✓ CUDAExecutionProvider is available
✓ ONNX Runtime GPU is ready


ONNX RUNTIME PROVIDER CONFIGURATION

Primary provider:
  CUDAExecutionProvider

Fallback provider:
  CPUExecutionProvider


STEP 2D — EXPERIMENT CONFIGURATION

Dataset      : /content/drive/MyDrive/PFE/Dataset/Vehicles_Dataset/data.yaml
Split        : test
Image size   : 640 × 640
Batch size   : 8
Fallback     : 4
Device       : CUDA:0
Confidence   : 0.001
IoU          : 0.7
ONNX Runtime : 1.26.0
ORT Provider : CUDAExecutionProvider


DATASET VERIFICATION

✓ Dataset YAML found
  /content/drive/MyDrive/PFE/Dataset/Vehicles_Dataset/data.yaml


LOADING MODEL REGI

,model,pt_precision,onnx_precision,delta_precision,pt_recall,onnx_recall,delta_recall,pt_map50,onnx_map50,delta_map50,pt_map50_95,onnx_map50_95,delta_map50_95,pt_inference_ms,onnx_inference_ms
0,yolov8n_50ep,0.903086,0.887913,-0.015173,0.837268,0.839940,0.002672,0.916761,0.910925,-0.005835,0.762446,0.751492,-0.010954,4.331852,6.335872
1,yolov8n_75ep,0.931754,0.925661,-0.006093,0.887200,0.867793,-0.019407,0.945971,0.941885,-0.004086,0.809393,0.800370,-0.009023,3.301515,6.561386
2,yolov8n_100ep,0.935103,0.922465,-0.012638,0.889164,0.886448,-0.002716,0.942811,0.937756,-0.005056,0.814313,0.807515,-0.006798,3.379789,6.821570
3,yolo11n,0.947521,0.939853,-0.007668,0.879859,0.877142,-0.002717,0.944769,0.940973,-0.003795,0.813504,0.805061,-0.008443,4.033204,6.934375
4,rtdetr_50ep,0.950456,0.950456,0.000000,0.904294,0.904294,-0.000000,0.960301,0.960301,-0.000001,0.850561,0.850559,-0.000002,36.672278,36.879168




ONNX FP32 DATASET-LEVEL EQUIVALENCE ANALYSIS

Equivalence criterion used:

    |ΔmAP50-95| ≤ 0.005
        → Equivalent / negligible difference

    0.005 < |ΔmAP50-95| ≤ 0.010
        → Small difference

    |ΔmAP50-95| > 0.010
        → Significant difference

NOTE:
The criterion is used as a practical engineering threshold
for determining whether FP32 ONNX export preserved model
accuracy.


yolov8n_50ep
  ΔPrecision : -0.015173
  ΔRecall    : +0.002672
  ΔmAP50     : -0.005835
  ΔmAP50-95  : -0.010954
  Assessment  : ❗ Significant difference

yolov8n_75ep
  ΔPrecision : -0.006093
  ΔRecall    : -0.019407
  ΔmAP50     : -0.004086
  ΔmAP50-95  : -0.009023
  Assessment  : ⚠️ Small difference

yolov8n_100ep
  ΔPrecision : -0.012638
  ΔRecall    : -0.002716
  ΔmAP50     : -0.005056
  ΔmAP50-95  : -0.006798
  Assessment  : ⚠️ Small difference

yolo11n
  ΔPrecision : -0.007668
  ΔRecall    : -0.002717
  ΔmAP50     : -0.003795
  ΔmAP50-95  : -0.008443
  Assessment  : ⚠️ Small difference



In [ ]:
# ============================================================
# STEP 3 — CORRECTED FP16 ONNX OPTIMIZATION
# ============================================================
#
# IMPORTANT:
#   - Deletes all previous FP16 ONNX files
#   - Does NOT modify FP32 ONNX models
#   - Does NOT modify .PT models
#   - Uses ONNX Runtime's FP16 converter
#   - Keeps model I/O in FP32 for compatibility
#   - Correctly handles mixed FP16/FP32 operators such as Resize
#
# ============================================================

import os
import sys
import json
import time
import shutil
import traceback
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import onnx
import onnxruntime as ort
import torch

from ultralytics import YOLO


# ============================================================
# CONFIGURATION
# ============================================================

DATA_YAML = (
    "/content/drive/MyDrive/PFE/Dataset/"
    "Vehicles_Dataset/data.yaml"
)

SPLIT = "test"

IMG_SIZE = 640
BATCH_SIZE = 8
DEVICE = "cuda:0"

CONF = 0.001
IOU = 0.7

# ------------------------------------------------------------
# Existing FP32 ONNX directory
# ------------------------------------------------------------

FP32_DIR = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "MODEL_OPTIMIZATION/02_onnx_fp32"
)

# ------------------------------------------------------------
# FP16 output directory
# ------------------------------------------------------------

FP16_DIR = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "MODEL_OPTIMIZATION/03_onnx_fp16"
)

# ------------------------------------------------------------
# Baseline directory
# ------------------------------------------------------------

BASELINE_DIR = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "MODEL_OPTIMIZATION/02_fp32_baseline"
)

# ------------------------------------------------------------
# Log directory
# ------------------------------------------------------------

LOG_DIR = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "MODEL_OPTIMIZATION/09_logs"
)


# ============================================================
# MODEL REGISTRY
# ============================================================

MODELS = {
    "yolov8n_50ep": {
        "fp32": FP32_DIR / "yolov8n_50ep_fp32.onnx",
        "fp16": FP16_DIR / "yolov8n_50ep_fp16.onnx",
    },

    "yolov8n_75ep": {
        "fp32": FP32_DIR / "yolov8n_75ep_fp32.onnx",
        "fp16": FP16_DIR / "yolov8n_75ep_fp16.onnx",
    },

    "yolov8n_100ep": {
        "fp32": FP32_DIR / "yolov8n_100ep_fp32.onnx",
        "fp16": FP16_DIR / "yolov8n_100ep_fp16.onnx",
    },

    "yolo11n": {
        "fp32": FP32_DIR / "yolo11n_fp32.onnx",
        "fp16": FP16_DIR / "yolo11n_fp16.onnx",
    },

    "rtdetr_50ep": {
        "fp32": FP32_DIR / "rtdetr_50ep_fp32.onnx",
        "fp16": FP16_DIR / "rtdetr_50ep_fp16.onnx",
    },
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def section(title):
    print("\n")
    print("=" * 100)
    print(title)
    print("=" * 100)


def subsection(title):
    print("\n")
    print("-" * 100)
    print(title)
    print("-" * 100)


def file_size_mb(path):
    return os.path.getsize(path) / (1024 ** 2)


# ============================================================
# STEP 3A — ENVIRONMENT
# ============================================================

section("STEP 3A — FP16 GPU ENVIRONMENT VERIFICATION")

print("→ Checking Python packages...")

try:
    import onnx
    print(f"✓ ONNX available")
    print(f"  ONNX version : {onnx.__version__}")
except Exception as e:
    raise RuntimeError(f"ONNX import failed: {e}")


try:
    import onnxruntime
    print(f"✓ ONNX Runtime available")
    print(f"  ONNX Runtime version : {onnxruntime.__version__}")
except Exception as e:
    raise RuntimeError(f"ONNX Runtime import failed: {e}")


# ------------------------------------------------------------
# IMPORTANT:
# Use ONNX Runtime's own FP16 converter.
# ------------------------------------------------------------

try:
    from onnxruntime.transformers.float16 import (
        convert_float_to_float16
    )

    print("✓ ONNX Runtime FP16 converter available")

except Exception as e:
    raise RuntimeError(
        "\nCould not import ONNX Runtime FP16 converter.\n"
        "Error:\n"
        f"{e}\n"
    )


print("\nUltralytics version :", __import__("ultralytics").__version__)
print("PyTorch version     :", torch.__version__)
print("CUDA available      :", torch.cuda.is_available())


if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not available. "
        "Do not continue with Step 3."
    )


print("CUDA version        :", torch.version.cuda)
print("GPU                 :", torch.cuda.get_device_name(0))

gpu_memory_gb = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
)

print(f"GPU memory          : {gpu_memory_gb:.2f} GB")

print("\n✓ PyTorch CUDA GPU verified")


# ============================================================
# ONNX RUNTIME PROVIDER VERIFICATION
# ============================================================

print("\n" + "=" * 100)
print("ONNX RUNTIME GPU VERIFICATION")
print("=" * 100)

available_providers = ort.get_available_providers()

print("\nAvailable ONNX Runtime providers:")

for provider in available_providers:
    print(f"  - {provider}")


if "CUDAExecutionProvider" not in available_providers:
    raise RuntimeError(
        "\nCUDAExecutionProvider is NOT available.\n"
        "Do not continue with Step 3."
    )

print("\n✓ CUDAExecutionProvider is available")
print("✓ ONNX Runtime GPU is ready")


# ============================================================
# STEP 3 CONFIGURATION
# ============================================================

section("STEP 3 — FP16 CONFIGURATION")

print(f"""
Dataset       : {DATA_YAML}
Split         : {SPLIT}
Image size    : {IMG_SIZE} × {IMG_SIZE}
Batch size    : {BATCH_SIZE}
Device        : {DEVICE}
Confidence    : {CONF}
IoU           : {IOU}
ONNX Runtime  : {ort.__version__}
ORT Provider  : CUDAExecutionProvider

FP16 strategy :
    ONNX Runtime mixed FP16 conversion

Input type:
    FP32

Output type:
    FP32

Internal computation:
    FP16 where supported

Resize:
    Handled by ONNX Runtime FP16 converter

Original FP32 ONNX:
    PRESERVED

Original .PT:
    PRESERVED
""")


# ============================================================
# DATASET VERIFICATION
# ============================================================

section("DATASET VERIFICATION")

if not os.path.exists(DATA_YAML):
    raise FileNotFoundError(
        f"Dataset YAML not found:\n{DATA_YAML}"
    )

print("✓ Dataset YAML found")
print(f"  {DATA_YAML}")


# ============================================================
# FP32 MODEL VERIFICATION
# ============================================================

section("FP32 ONNX MODEL VERIFICATION")

for name, paths in MODELS.items():

    fp32_path = paths["fp32"]

    if not fp32_path.exists():
        raise FileNotFoundError(
            f"\nMissing FP32 ONNX model:\n{fp32_path}"
        )

    print(f"\n✓ {name}")
    print(f"  File : {fp32_path}")
    print(f"  Size : {file_size_mb(fp32_path):.2f} MB")


print("\n✓ All FP32 ONNX models found")


# ============================================================
# STEP 3B
# DELETE OLD FP16 MODELS
# ============================================================

section("STEP 3B — CLEANING PREVIOUS FP16 OUTPUTS")

print(f"""
FP16 directory:

{FP16_DIR}

IMPORTANT:
All previous FP16 ONNX models in this directory
will be deleted.

The FP32 ONNX directory will NOT be touched.
""")


# ------------------------------------------------------------
# Delete entire FP16 directory
# ------------------------------------------------------------

if FP16_DIR.exists():

    print("→ Removing previous FP16 directory...")

    shutil.rmtree(FP16_DIR)

    print("✓ Previous FP16 directory deleted")

else:

    print("✓ Previous FP16 directory does not exist")


# ------------------------------------------------------------
# Recreate clean directory
# ------------------------------------------------------------

FP16_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("✓ New clean FP16 directory created")

print(f"\n{FP16_DIR}")


# ------------------------------------------------------------
# Verify that no old ONNX files remain
# ------------------------------------------------------------

remaining_onnx = list(
    FP16_DIR.glob("*.onnx")
)

if len(remaining_onnx) != 0:

    raise RuntimeError(
        "Old FP16 ONNX files still exist!"
    )

print("\n✓ FP16 directory is completely clean")


# ============================================================
# DELETE OLD STEP 3 RESULTS
# ============================================================

section("STEP 3B — CLEANING PREVIOUS STEP 3 RESULTS")

old_result_files = [

    BASELINE_DIR /
    "step3_gpu_onnx_fp16_validation.csv",

    BASELINE_DIR /
    "step3_gpu_onnx_fp32_vs_fp16_comparison.csv",

    FP16_DIR /
    "step3_fp16_conversion_results.csv",

    FP16_DIR /
    "step3_fp16_session_verification.csv",

    LOG_DIR /
    "step3_fp16_gpu_complete_experiment.json",
]


for path in old_result_files:

    if path.exists():

        print(f"→ Deleting old result:")
        print(f"  {path}")

        path.unlink()

        print("  ✓ Deleted")


print("\n✓ Previous Step 3 result files cleaned")


# ============================================================
# STEP 3C
# FP32 → FP16 CONVERSION
# ============================================================

section("STEP 3C — FP32 ONNX → MIXED FP16 ONNX CONVERSION")

conversion_results = []

for name, paths in MODELS.items():

    subsection(f"FP16 CONVERSION — {name}")

    fp32_path = paths["fp32"]
    fp16_path = paths["fp16"]

    print("\nFP32 source:")
    print(f"  {fp32_path}")

    print("\nFP16 destination:")
    print(f"  {fp16_path}")

    fp32_size = file_size_mb(fp32_path)

    print(f"\nFP32 model size : {fp32_size:.2f} MB")

    try:

        print("→ Loading FP32 ONNX model...")

        start_time = time.time()

        model = onnx.load(
            str(fp32_path)
        )

        load_time = time.time() - start_time

        print(
            f"✓ FP32 model loaded "
            f"({load_time:.2f} s)"
        )

        # ----------------------------------------------------
        # Convert using ONNX Runtime's converter
        #
        # keep_io_types=True means:
        #
        #   Input  = FLOAT32
        #   Output = FLOAT32
        #
        # while internal supported operations are converted
        # to FLOAT16.
        #
        # This is important for compatibility with Ultralytics
        # and avoids the previous Resize type mismatch.
        # ----------------------------------------------------

        print(
            "→ Converting FP32 ONNX to "
            "mixed FP16 using ONNX Runtime..."
        )

        start_time = time.time()

        model_fp16 = convert_float_to_float16(
            model,
            keep_io_types=True,
        )

        conversion_time = (
            time.time() - start_time
        )

        print(
            f"✓ FP16 conversion completed "
            f"in {conversion_time:.2f} seconds"
        )

        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        print("→ Saving FP16 ONNX model...")

        onnx.save(
            model_fp16,
            str(fp16_path)
        )

        print("✓ FP16 model saved")

        # ----------------------------------------------------
        # Graph checker
        # ----------------------------------------------------

        print("→ Running ONNX graph checker...")

        checked_model = onnx.load(
            str(fp16_path)
        )

        onnx.checker.check_model(
            checked_model
        )

        print("✓ ONNX graph check passed")

        # ----------------------------------------------------
        # Reload test
        # ----------------------------------------------------

        print("→ Reloading saved FP16 model...")

        reloaded_model = onnx.load(
            str(fp16_path)
        )

        print("✓ Saved model reload successful")

        # ----------------------------------------------------
        # Size
        # ----------------------------------------------------

        fp16_size = file_size_mb(fp16_path)

        reduction_mb = (
            fp32_size - fp16_size
        )

        reduction_percent = (
            reduction_mb / fp32_size
        ) * 100

        print("\nModel size comparison:")

        print(
            f"  FP32 size       : "
            f"{fp32_size:.2f} MB"
        )

        print(
            f"  FP16 size       : "
            f"{fp16_size:.2f} MB"
        )

        print(
            f"  Size reduction  : "
            f"{reduction_mb:.2f} MB"
        )

        print(
            f"  Reduction       : "
            f"{reduction_percent:.2f}%"
        )

        conversion_results.append({
            "model": name,
            "status": "success",
            "fp32_size_mb": fp32_size,
            "fp16_size_mb": fp16_size,
            "size_reduction_mb": reduction_mb,
            "size_reduction_percent": reduction_percent,
            "conversion_time_s": conversion_time,
        })

    except Exception as e:

        print("\n❌ FP16 conversion failed")

        print(str(e))

        conversion_results.append({
            "model": name,
            "status": "conversion_failed",
            "fp32_size_mb": fp32_size,
            "fp16_size_mb": np.nan,
            "size_reduction_mb": np.nan,
            "size_reduction_percent": np.nan,
            "conversion_time_s": np.nan,
            "error": str(e),
        })


# ============================================================
# SAVE CONVERSION RESULTS
# ============================================================

conversion_df = pd.DataFrame(
    conversion_results
)

conversion_csv = (
    FP16_DIR /
    "step3_fp16_conversion_results.csv"
)

conversion_df.to_csv(
    conversion_csv,
    index=False
)

print("\n✓ Conversion results saved:")
print(conversion_csv)


# ============================================================
# STEP 3D
# CUDA SESSION VERIFICATION
# ============================================================

section(
    "STEP 3D — FP16 ONNX RUNTIME CUDA SESSION VERIFICATION"
)

session_results = []

for name, paths in MODELS.items():

    subsection(f"TESTING — {name}")

    fp16_path = paths["fp16"]

    if not fp16_path.exists():

        print(
            "❌ FP16 model does not exist"
        )

        session_results.append({
            "model": name,
            "status": "missing",
            "providers": "",
            "cuda_available": False,
            "error": "FP16 model missing",
        })

        continue

    print("Creating ONNX Runtime FP16 session...")

    try:

        session_start = time.time()

        session = ort.InferenceSession(
            str(fp16_path),
            providers=[
                "CUDAExecutionProvider",
                "CPUExecutionProvider",
            ],
        )

        session_time = (
            time.time() - session_start
        )

        providers = session.get_providers()

        print("\nSession providers:")

        for provider in providers:
            print(f"  - {provider}")

        if (
            "CUDAExecutionProvider"
            not in providers
        ):

            raise RuntimeError(
                "CUDAExecutionProvider "
                "was not selected."
            )

        # ----------------------------------------------------
        # Input/output inspection
        # ----------------------------------------------------

        inputs = session.get_inputs()
        outputs = session.get_outputs()

        print("\nModel inputs:")

        for inp in inputs:

            print(
                f"  {inp.name}"
                f" | type={inp.type}"
                f" | shape={inp.shape}"
            )

        print("\nModel outputs:")

        for out in outputs:

            print(
                f"  {out.name}"
                f" | type={out.type}"
                f" | shape={out.shape}"
            )

        print(
            f"\nSession creation time : "
            f"{session_time:.3f} s"
        )

        print(
            "\n✓ ONNX Runtime CUDA session "
            "successfully created"
        )

        session_results.append({
            "model": name,
            "status": "success",
            "providers": str(providers),
            "cuda_available": True,
            "input_type": str(inputs[0].type),
            "output_type": str(outputs[0].type),
            "session_creation_time_s": session_time,
            "error": "",
        })

    except Exception as e:

        print("\n❌ Session verification failed")

        print(str(e))

        session_results.append({
            "model": name,
            "status": "session_failed",
            "providers": "",
            "cuda_available": False,
            "input_type": "",
            "output_type": "",
            "session_creation_time_s": np.nan,
            "error": str(e),
        })


# ============================================================
# SAVE SESSION RESULTS
# ============================================================

session_df = pd.DataFrame(
    session_results
)

session_csv = (
    FP16_DIR /
    "step3_fp16_session_verification.csv"
)

session_df.to_csv(
    session_csv,
    index=False
)

print("\n✓ Session verification saved:")
print(session_csv)


# ============================================================
# STEP 3E
# FP16 GPU DATASET VALIDATION
# ============================================================

section(
    "STEP 3E — FP16 GPU DATASET VALIDATION"
)

print(f"""
Dataset      : {DATA_YAML}
Split        : {SPLIT}
Image size   : {IMG_SIZE} × {IMG_SIZE}
Batch size   : {BATCH_SIZE}
Device       : {DEVICE}
Confidence   : {CONF}
IoU          : {IOU}
ONNX Runtime : {ort.__version__}
Provider     : CUDAExecutionProvider
""")


validation_results = []

for name, paths in MODELS.items():

    subsection(
        f"ONNX_FP16 — {name}"
    )

    fp16_path = paths["fp16"]

    # --------------------------------------------------------
    # Check previous CUDA session test
    # --------------------------------------------------------

    session_row = session_df[
        session_df["model"] == name
    ]

    if (
        session_row.empty
        or session_row.iloc[0]["status"]
        != "success"
    ):

        print(
            "❌ Skipping validation because "
            "CUDA session test failed."
        )

        validation_results.append({
            "model": name,
            "status": "validation_skipped",
            "precision": np.nan,
            "recall": np.nan,
            "map50": np.nan,
            "map50_95": np.nan,
            "preprocess_ms": np.nan,
            "inference_ms": np.nan,
            "postprocess_ms": np.nan,
            "total_time_s": np.nan,
            "error": "CUDA session verification failed",
        })

        continue

    try:

        print("→ Loading FP16 ONNX model...")

        model = YOLO(
            str(fp16_path),
            task="detect"
        )

        print("✓ FP16 ONNX model loaded")

        print(
            "\n→ Running test dataset validation..."
        )

        start_time = time.time()

        results = model.val(
            data=DATA_YAML,
            split=SPLIT,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            device=DEVICE,
            conf=CONF,
            iou=IOU,
            verbose=True,
        )

        total_time = (
            time.time() - start_time
        )

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        precision = float(
            results.box.mp
        )

        recall = float(
            results.box.mr
        )

        map50 = float(
            results.box.map50
        )

        map50_95 = float(
            results.box.map
        )

        # ----------------------------------------------------
        # Speed
        # ----------------------------------------------------

        speed = results.speed

        preprocess_ms = float(
            speed.get(
                "preprocess",
                np.nan
            )
        )

        inference_ms = float(
            speed.get(
                "inference",
                np.nan
            )
        )

        postprocess_ms = float(
            speed.get(
                "postprocess",
                np.nan
            )
        )

        print("\n✓ FP16 validation completed")

        print(
            f"\nPrecision : {precision:.6f}"
        )

        print(
            f"Recall    : {recall:.6f}"
        )

        print(
            f"mAP50     : {map50:.6f}"
        )

        print(
            f"mAP50-95  : {map50_95:.6f}"
        )

        print(
            f"Preprocess: {preprocess_ms:.3f} ms"
        )

        print(
            f"Inference : {inference_ms:.3f} ms"
        )

        print(
            f"Postprocess: {postprocess_ms:.3f} ms"
        )

        print(
            f"Total     : {total_time:.2f} s"
        )

        validation_results.append({
            "model": name,
            "status": "success",
            "precision": precision,
            "recall": recall,
            "map50": map50,
            "map50_95": map50_95,
            "preprocess_ms": preprocess_ms,
            "inference_ms": inference_ms,
            "postprocess_ms": postprocess_ms,
            "total_time_s": total_time,
            "error": "",
        })

    except Exception as e:

        print(
            "\n❌ FP16 dataset validation failed"
        )

        print(str(e))

        validation_results.append({
            "model": name,
            "status": "validation_failed",
            "precision": np.nan,
            "recall": np.nan,
            "map50": np.nan,
            "map50_95": np.nan,
            "preprocess_ms": np.nan,
            "inference_ms": np.nan,
            "postprocess_ms": np.nan,
            "total_time_s": np.nan,
            "error": str(e),
        })


# ============================================================
# SAVE FP16 VALIDATION
# ============================================================

validation_df = pd.DataFrame(
    validation_results
)

validation_csv = (
    BASELINE_DIR /
    "step3_gpu_onnx_fp16_validation.csv"
)

validation_df.to_csv(
    validation_csv,
    index=False
)

print("\n✓ FP16 validation results saved:")
print(validation_csv)


# ============================================================
# STEP 3F
# LOAD STEP 2D FP32 RESULTS
# ============================================================

section(
    "STEP 3F — LOADING STEP 2D FP32 RESULTS"
)

fp32_comparison_csv = (
    BASELINE_DIR /
    "step2d_gpu_pt_vs_onnx_comparison.csv"
)

if not fp32_comparison_csv.exists():

    raise FileNotFoundError(
        "\nStep 2D comparison file not found:\n"
        f"{fp32_comparison_csv}"
    )

fp32_df = pd.read_csv(
    fp32_comparison_csv
)

print("✓ Step 2D FP32 results loaded")
print(f"  Rows : {len(fp32_df)}")


# ============================================================
# STEP 3G
# FP32 vs FP16 COMPARISON
# ============================================================

section(
    "STEP 3G — ONNX FP32 vs FP16 COMPARISON"
)

comparison_results = []


for name, paths in MODELS.items():

    subsection(name)

    fp32_row = fp32_df[
        fp32_df["model"] == name
    ]

    fp16_row = validation_df[
        validation_df["model"] == name
    ]

    if fp32_row.empty:

        print(
            "⚠️ FP32 result not found"
        )

        comparison_results.append({
            "model": name,
            "status": "fp32_missing",
        })

        continue

    if (
        fp16_row.empty
        or fp16_row.iloc[0]["status"]
        != "success"
    ):

        print(
            "⚠️ FP16 evaluation failed"
        )

        comparison_results.append({
            "model": name,
            "status": "fp16_failed",
        })

        continue

    fp32 = fp32_row.iloc[0]
    fp16 = fp16_row.iloc[0]

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    pt_precision = float(
        fp32["onnx_precision"]
    )

    pt_recall = float(
        fp32["onnx_recall"]
    )

    pt_map50 = float(
        fp32["onnx_map50"]
    )

    pt_map50_95 = float(
        fp32["onnx_map50_95"]
    )

    fp16_precision = float(
        fp16["precision"]
    )

    fp16_recall = float(
        fp16["recall"]
    )

    fp16_map50 = float(
        fp16["map50"]
    )

    fp16_map50_95 = float(
        fp16["map50_95"]
    )

    # --------------------------------------------------------
    # Deltas
    # --------------------------------------------------------

    delta_precision = (
        fp16_precision -
        pt_precision
    )

    delta_recall = (
        fp16_recall -
        pt_recall
    )

    delta_map50 = (
        fp16_map50 -
        pt_map50
    )

    delta_map50_95 = (
        fp16_map50_95 -
        pt_map50_95
    )

    # --------------------------------------------------------
    # Inference
    # --------------------------------------------------------

    fp32_inference = float(
        fp32["onnx_inference_ms"]
    )

    fp16_inference = float(
        fp16["inference_ms"]
    )

    inference_delta = (
        fp16_inference -
        fp32_inference
    )

    if fp32_inference > 0:

        inference_change_percent = (
            (fp16_inference -
             fp32_inference)
            / fp32_inference
        ) * 100

    else:

        inference_change_percent = np.nan

    # --------------------------------------------------------
    # Model size
    # --------------------------------------------------------

    fp32_size = file_size_mb(
        paths["fp32"]
    )

    fp16_size = file_size_mb(
        paths["fp16"]
    )

    size_reduction_percent = (
        (fp32_size - fp16_size)
        / fp32_size
    ) * 100

    # --------------------------------------------------------
    # Assessment
    # --------------------------------------------------------

    abs_delta = abs(
        delta_map50_95
    )

    if abs_delta <= 0.005:

        assessment = (
            "Equivalent — negligible difference"
        )

    elif abs_delta <= 0.010:

        assessment = (
            "Small difference"
        )

    else:

        assessment = (
            "Significant difference"
        )

    print(
        f"FP32 mAP50-95 : "
        f"{pt_map50_95:.6f}"
    )

    print(
        f"FP16 mAP50-95 : "
        f"{fp16_map50_95:.6f}"
    )

    print(
        f"ΔmAP50-95     : "
        f"{delta_map50_95:+.6f}"
    )

    print(
        f"FP32 inference: "
        f"{fp32_inference:.3f} ms"
    )

    print(
        f"FP16 inference: "
        f"{fp16_inference:.3f} ms"
    )

    print(
        f"Size reduction: "
        f"{size_reduction_percent:.2f}%"
    )

    print(
        f"Assessment     : "
        f"{assessment}"
    )

    comparison_results.append({

        "model": name,

        "status": "success",

        "fp32_precision": pt_precision,
        "fp16_precision": fp16_precision,
        "delta_precision": delta_precision,

        "fp32_recall": pt_recall,
        "fp16_recall": fp16_recall,
        "delta_recall": delta_recall,

        "fp32_map50": pt_map50,
        "fp16_map50": fp16_map50,
        "delta_map50": delta_map50,

        "fp32_map50_95": pt_map50_95,
        "fp16_map50_95": fp16_map50_95,
        "delta_map50_95": delta_map50_95,

        "fp32_inference_ms": fp32_inference,
        "fp16_inference_ms": fp16_inference,

        "inference_delta_ms": inference_delta,

        "inference_change_percent":
            inference_change_percent,

        "fp32_size_mb":
            fp32_size,

        "fp16_size_mb":
            fp16_size,

        "size_reduction_mb":
            fp32_size - fp16_size,

        "size_reduction_percent":
            size_reduction_percent,

        "assessment":
            assessment,
    })


# ============================================================
# SAVE COMPARISON
# ============================================================

comparison_df = pd.DataFrame(
    comparison_results
)

comparison_csv = (
    BASELINE_DIR /
    "step3_gpu_onnx_fp32_vs_fp16_comparison.csv"
)

comparison_df.to_csv(
    comparison_csv,
    index=False
)

print("\n✓ FP32 vs FP16 comparison saved:")
print(comparison_csv)


# ============================================================
# FINAL SUMMARY
# ============================================================

section(
    "STEP 3 — FINAL FP16 SUMMARY"
)

conversion_success = int(
    (conversion_df["status"] == "success").sum()
)

session_success = int(
    (session_df["status"] == "success").sum()
)

validation_success = int(
    (validation_df["status"] == "success").sum()
)

comparison_success = int(
    (comparison_df["status"] == "success").sum()
)


print(
    f"""
FP16 conversions successfully completed :
    {conversion_success}/5

CUDA sessions successfully verified :
    {session_success}/5

FP16 validations successfully completed :
    {validation_success}/5

FP32 vs FP16 comparisons completed :
    {comparison_success}/5

ONNX Runtime:
    {ort.__version__}

CUDAExecutionProvider:
    ✓ Available

FP16 strategy:
    ONNX Runtime mixed FP16

Input:
    FP32

Output:
    FP32

Internal supported operations:
    FP16

Original FP32 ONNX:
    NOT MODIFIED

Original .PT:
    NOT MODIFIED
"""
)


# ============================================================
# FINAL GPU MEMORY
# ============================================================

section("FINAL GPU MEMORY STATUS")

allocated = (
    torch.cuda.memory_allocated(0)
    / (1024 ** 3)
)

reserved = (
    torch.cuda.memory_reserved(0)
    / (1024 ** 3)
)

print(
    f"GPU allocated : {allocated:.3f} GB"
)

print(
    f"GPU reserved  : {reserved:.3f} GB"
)


# ============================================================
# SAVE COMPLETE LOG
# ============================================================

experiment_log = {

    "step": "3",

    "experiment": (
        "FP32 ONNX vs Mixed FP16 ONNX "
        "GPU evaluation"
    ),

    "dataset": DATA_YAML,

    "split": SPLIT,

    "image_size": IMG_SIZE,

    "batch_size": BATCH_SIZE,

    "device": DEVICE,

    "confidence": CONF,

    "iou": IOU,

    "onnxruntime_version":
        ort.__version__,

    "providers":
        available_providers,

    "fp16_strategy":
        "ONNX Runtime mixed FP16",

    "keep_io_types": True,

    "models": list(MODELS.keys()),

    "conversion_results":
        conversion_results,

    "session_results":
        session_results,

    "validation_results":
        validation_results,

    "comparison_results":
        comparison_results,
}


LOG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

log_path = (
    LOG_DIR /
    "step3_fp16_gpu_complete_experiment.json"
)

with open(
    log_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        experiment_log,
        f,
        indent=2,
        default=str
    )


print("\n✓ Complete experiment log saved:")
print(log_path)


# ============================================================
# FINAL FILE LIST
# ============================================================

section("STEP 3 — OUTPUT FILES")

print("\nFP16 models:")

for name, paths in MODELS.items():

    path = paths["fp16"]

    if path.exists():

        print(
            f"  ✓ {name}: "
            f"{path}"
        )

    else:

        print(
            f"  ❌ {name}: "
            f"NOT CREATED"
        )


print("\nConversion results:")
print(
    FP16_DIR /
    "step3_fp16_conversion_results.csv"
)

print("\nSession verification:")
print(
    FP16_DIR /
    "step3_fp16_session_verification.csv"
)

print("\nFP16 GPU validation:")
print(
    BASELINE_DIR /
    "step3_gpu_onnx_fp16_validation.csv"
)

print("\nFP32 vs FP16 comparison:")
print(
    BASELINE_DIR /
    "step3_gpu_onnx_fp32_vs_fp16_comparison.csv"
)

print("\nComplete experiment log:")
print(
    LOG_DIR /
    "step3_fp16_gpu_complete_experiment.json"
)


# ============================================================
# FINAL
# ============================================================

section("STEP 3 COMPLETED")

print(
    """
The previous FP16 ONNX files were deleted before conversion.

New FP16 models were generated directly from the
original FP32 ONNX models.

The original FP32 ONNX models were NOT modified.

The original .PT models were NOT modified.

The FP16 models were graph-checked before CUDA testing.

CUDAExecutionProvider was tested before dataset validation.

All five models were processed:

    • yolov8n_50ep
    • yolov8n_75ep
    • yolov8n_100ep
    • yolo11n
    • rtdetr_50ep

Do NOT proceed to INT8 until the final Step 3 output
has been reviewed.
"""
)



STEP 3A — FP16 GPU ENVIRONMENT VERIFICATION
→ Checking Python packages...
✓ ONNX available
  ONNX version : 1.22.0
✓ ONNX Runtime available
  ONNX Runtime version : 1.26.0
✓ ONNX Runtime FP16 converter available

Ultralytics version : 8.4.130
PyTorch version     : 2.11.0+cu128
CUDA available      : True
CUDA version        : 12.8
GPU                 : Tesla T4
GPU memory          : 14.56 GB

✓ PyTorch CUDA GPU verified

ONNX RUNTIME GPU VERIFICATION

Available ONNX Runtime providers:
  - TensorrtExecutionProvider
  - CUDAExecutionProvider
  - CPUExecutionProvider

✓ CUDAExecutionProvider is available
✓ ONNX Runtime GPU is ready


STEP 3 — FP16 CONFIGURATION

Dataset       : /content/drive/MyDrive/PFE/Dataset/Vehicles_Dataset/data.yaml
Split         : test
Image size    : 640 × 640
Batch size    : 8
Device        : cuda:0
Confidence    : 0.001
IoU           : 0.7
ONNX Runtime  : 1.26.0
ORT Provider  : CUDAExecutionProvider

FP16 strategy :
    ONNX Runtime mixed FP16 conversion

Input

## Step 3 Again


In [ ]:
# ============================================================
# STEP 3-F — FP16 ARTIFACT CORRECTION & FINAL VALIDATION
# ============================================================
#
# PURPOSE:
#   Correct and formally validate the FP16 ONNX artifacts
#   generated during Step 3.
#
# IMPORTANT:
#   - Original .PT models are NOT modified.
#   - Original FP32 ONNX models are NOT modified.
#   - Existing FP16 artifacts are regenerated from FP32 ONNX.
#   - No 1900-image dataset validation is performed here.
#   - This step only validates the FP16 artifacts themselves.
#
# WORKFLOW:
#
#   FP32 ONNX
#       ↓
#   Mixed FP16 conversion
#       ↓
#   Graph topological sort
#       ↓
#   ONNX checker
#       ↓
#   Shape inference
#       ↓
#   Save corrected FP16 ONNX
#       ↓
#   ONNX Runtime CUDA verification
#       ↓
#   Single-image inference sanity check
#
# ============================================================

import os
import time
import json
import shutil
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import onnx
import onnxruntime as ort
import torch

from onnxruntime.transformers.float16 import convert_float_to_float16


# ============================================================
# CONFIGURATION
# ============================================================

FP32_DIR = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "MODEL_OPTIMIZATION/02_onnx_fp32"
)

FP16_DIR = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "MODEL_OPTIMIZATION/03_onnx_fp16"
)

LOG_DIR = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "MODEL_OPTIMIZATION/09_logs"
)

# Small sanity-test input
IMG_SIZE = 640

# ============================================================
# MODEL REGISTRY
# ============================================================

MODELS = {
    "yolov8n_50ep": {
        "fp32": FP32_DIR / "yolov8n_50ep_fp32.onnx",
        "fp16": FP16_DIR / "yolov8n_50ep_fp16.onnx",
    },

    "yolov8n_75ep": {
        "fp32": FP32_DIR / "yolov8n_75ep_fp32.onnx",
        "fp16": FP16_DIR / "yolov8n_75ep_fp16.onnx",
    },

    "yolov8n_100ep": {
        "fp32": FP32_DIR / "yolov8n_100ep_fp32.onnx",
        "fp16": FP16_DIR / "yolov8n_100ep_fp16.onnx",
    },

    "yolo11n": {
        "fp32": FP32_DIR / "yolo11n_fp32.onnx",
        "fp16": FP16_DIR / "yolo11n_fp16.onnx",
    },

    "rtdetr_50ep": {
        "fp32": FP32_DIR / "rtdetr_50ep_fp32.onnx",
        "fp16": FP16_DIR / "rtdetr_50ep_fp16.onnx",
    },
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def section(title):
    print("\n")
    print("=" * 100)
    print(title)
    print("=" * 100)


def subsection(title):
    print("\n")
    print("-" * 100)
    print(title)
    print("-" * 100)


def file_size_mb(path):
    return os.path.getsize(path) / (1024 ** 2)


def topologically_sort_graph(graph):
    """
    Reorder graph nodes so every node appears after the nodes
    that produce its inputs.

    This addresses the graph-check failure observed in Step 3:
        graph_input_cast_0
    """

    nodes = list(graph.node)

    # --------------------------------------------------------
    # Build producer map
    # --------------------------------------------------------

    producer = {}

    for index, node in enumerate(nodes):

        for output in node.output:

            if output:
                producer[output] = index

    # --------------------------------------------------------
    # Graph inputs and initializers are already available.
    # --------------------------------------------------------

    available = set()

    for graph_input in graph.input:
        available.add(graph_input.name)

    for initializer in graph.initializer:
        available.add(initializer.name)

    # --------------------------------------------------------
    # Topological ordering
    # --------------------------------------------------------

    remaining = set(range(len(nodes)))
    ordered = []

    while remaining:

        progress = False

        for index in list(remaining):

            node = nodes[index]

            inputs_ready = True

            for input_name in node.input:

                if not input_name:
                    continue

                if input_name not in available:

                    inputs_ready = False
                    break

            if inputs_ready:

                ordered.append(node)
                remaining.remove(index)

                for output in node.output:

                    if output:
                        available.add(output)

                progress = True

        if not progress:

            unresolved = []

            for index in list(remaining):

                node = nodes[index]

                missing = [
                    x for x in node.input
                    if x and x not in available
                ]

                unresolved.append({
                    "node": node.name,
                    "op_type": node.op_type,
                    "missing_inputs": missing,
                })

            raise RuntimeError(
                "Could not topologically sort ONNX graph.\n"
                f"Unresolved nodes:\n{unresolved[:10]}"
            )

    # --------------------------------------------------------
    # Replace original node ordering
    # --------------------------------------------------------

    del graph.node[:]
    graph.node.extend(ordered)

    return graph


def inspect_graph(model):
    """Return useful graph statistics."""

    graph = model.graph

    op_counts = {}

    for node in graph.node:

        op_counts[node.op_type] = (
            op_counts.get(node.op_type, 0) + 1
        )

    return {
        "node_count": len(graph.node),
        "initializer_count": len(graph.initializer),
        "input_count": len(graph.input),
        "output_count": len(graph.output),
        "op_types": op_counts,
    }


# ============================================================
# ENVIRONMENT
# ============================================================

section(
    "STEP 3-F — FP16 ARTIFACT CORRECTION & VALIDATION"
)

print("Environment:")
print(f"  ONNX             : {onnx.__version__}")
print(f"  ONNX Runtime     : {ort.__version__}")
print(f"  PyTorch          : {torch.__version__}")
print(f"  CUDA available   : {torch.cuda.is_available()}")

print("\nONNX Runtime providers:")

for provider in ort.get_available_providers():
    print(f"  - {provider}")

if "CUDAExecutionProvider" not in ort.get_available_providers():

    raise RuntimeError(
        "CUDAExecutionProvider is not available."
    )

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU is not available."
    )

print("\n✓ Environment verified")


# ============================================================
# VERIFY SOURCE FP32 MODELS
# ============================================================

section(
    "STEP 3-F — VERIFYING PRESERVED FP32 SOURCE MODELS"
)

for name, paths in MODELS.items():

    fp32_path = paths["fp32"]

    if not fp32_path.exists():

        raise FileNotFoundError(
            f"Missing FP32 model:\n{fp32_path}"
        )

    print(
        f"✓ {name:<20} "
        f"{file_size_mb(fp32_path):>8.2f} MB"
    )

print("\n✓ All five FP32 source models are available")


# ============================================================
# PREPARE FP16 DIRECTORY
# ============================================================

section(
    "STEP 3-F — PREPARING CLEAN FP16 ARTIFACT DIRECTORY"
)

if FP16_DIR.exists():

    print("→ Removing previous FP16 artifacts...")

    shutil.rmtree(FP16_DIR)

    print("✓ Previous FP16 directory removed")

FP16_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(f"✓ Clean directory created:")
print(f"  {FP16_DIR}")


# ============================================================
# CONVERSION + GRAPH REPAIR
# ============================================================

section(
    "STEP 3-F — FP32 → FP16 CONVERSION + GRAPH REPAIR"
)

conversion_results = []


for name, paths in MODELS.items():

    subsection(
        f"PROCESSING — {name}"
    )

    fp32_path = paths["fp32"]
    fp16_path = paths["fp16"]

    result = {
        "model": name,
        "conversion_status": "failed",
        "graph_repair_status": "not_attempted",
        "onnx_checker_status": "not_attempted",
        "shape_inference_status": "not_attempted",
        "fp32_size_mb": np.nan,
        "fp16_size_mb": np.nan,
        "size_reduction_percent": np.nan,
        "conversion_time_s": np.nan,
        "error": "",
    }

    try:

        # ----------------------------------------------------
        # Load FP32 source
        # ----------------------------------------------------

        print("→ Loading FP32 ONNX...")

        load_start = time.time()

        model = onnx.load(
            str(fp32_path)
        )

        load_time = time.time() - load_start

        print(
            f"✓ Loaded in {load_time:.3f} s"
        )

        fp32_size = file_size_mb(
            fp32_path
        )

        result["fp32_size_mb"] = fp32_size

        # ----------------------------------------------------
        # Display source graph
        # ----------------------------------------------------

        source_info = inspect_graph(model)

        print("\nFP32 graph:")
        print(
            f"  Nodes        : "
            f"{source_info['node_count']}"
        )

        print(
            f"  Initializers : "
            f"{source_info['initializer_count']}"
        )

        # ----------------------------------------------------
        # FP16 conversion
        # ----------------------------------------------------

        print(
            "\n→ Converting to mixed FP16..."
        )

        conversion_start = time.time()

        model_fp16 = convert_float_to_float16(
            model,
            keep_io_types=True,
        )

        conversion_time = (
            time.time() -
            conversion_start
        )

        result[
            "conversion_time_s"
        ] = conversion_time

        result[
            "conversion_status"
        ] = "success"

        print(
            f"✓ Conversion completed in "
            f"{conversion_time:.3f} s"
        )

        # ----------------------------------------------------
        # Inspect generated graph
        # ----------------------------------------------------

        fp16_info_before = inspect_graph(
            model_fp16
        )

        print("\nGenerated FP16 graph:")
        print(
            f"  Nodes        : "
            f"{fp16_info_before['node_count']}"
        )

        print(
            f"  Initializers : "
            f"{fp16_info_before['initializer_count']}"
        )

        # ----------------------------------------------------
        # Topological sorting
        # ----------------------------------------------------

        print(
            "\n→ Topologically sorting FP16 graph..."
        )

        try:

            topologically_sort_graph(
                model_fp16.graph
            )

            result[
                "graph_repair_status"
            ] = "success"

            print(
                "✓ Graph topological ordering completed"
            )

        except Exception as sort_error:

            result[
                "graph_repair_status"
            ] = "failed"

            raise RuntimeError(
                "Topological graph repair failed:\n"
                f"{sort_error}"
            )

        # ----------------------------------------------------
        # ONNX checker
        # ----------------------------------------------------

        print(
            "\n→ Running ONNX checker..."
        )

        try:

            onnx.checker.check_model(
                model_fp16
            )

            result[
                "onnx_checker_status"
            ] = "passed"

            print(
                "✓ ONNX graph checker PASSED"
            )

        except Exception as checker_error:

            print(
                "⚠️ Standard checker failed:"
            )

            print(str(checker_error))

            # ------------------------------------------------
            # Try full checker with no external data loading
            # ------------------------------------------------

            raise RuntimeError(
                "ONNX checker failed after "
                "topological sorting:\n"
                f"{checker_error}"
            )

        # ----------------------------------------------------
        # Shape inference
        # ----------------------------------------------------

        print(
            "\n→ Running ONNX shape inference..."
        )

        try:

            inferred_model = (
                onnx.shape_inference.infer_shapes(
                    model_fp16
                )
            )

            # Replace with inferred model
            model_fp16 = inferred_model

            result[
                "shape_inference_status"
            ] = "passed"

            print(
                "✓ Shape inference PASSED"
            )

        except Exception as shape_error:

            result[
                "shape_inference_status"
            ] = "failed"

            raise RuntimeError(
                "Shape inference failed:\n"
                f"{shape_error}"
            )

        # ----------------------------------------------------
        # Save corrected model
        # ----------------------------------------------------

        print(
            "\n→ Saving corrected FP16 model..."
        )

        onnx.save(
            model_fp16,
            str(fp16_path)
        )

        print(
            "✓ Corrected FP16 model saved"
        )

        # ----------------------------------------------------
        # Reload from disk
        # ----------------------------------------------------

        print(
            "→ Reloading saved model..."
        )

        reloaded = onnx.load(
            str(fp16_path)
        )

        print(
            "✓ Reload successful"
        )

        # ----------------------------------------------------
        # Final checker on SAVED artifact
        # ----------------------------------------------------

        print(
            "→ Running final checker on saved artifact..."
        )

        onnx.checker.check_model(
            reloaded
        )

        print(
            "✓ Saved artifact passed ONNX checker"
        )

        # ----------------------------------------------------
        # Size
        # ----------------------------------------------------

        fp16_size = file_size_mb(
            fp16_path
        )

        reduction = (
            fp32_size -
            fp16_size
        )

        reduction_percent = (
            reduction /
            fp32_size
        ) * 100

        result[
            "fp16_size_mb"
        ] = fp16_size

        result[
            "size_reduction_percent"
        ] = reduction_percent

        print("\nSize:")
        print(
            f"  FP32 : {fp32_size:.2f} MB"
        )

        print(
            f"  FP16 : {fp16_size:.2f} MB"
        )

        print(
            f"  Reduction : "
            f"{reduction_percent:.2f}%"
        )

    except Exception as e:

        result[
            "error"
        ] = str(e)

        print(
            "\n❌ Processing failed:"
        )

        print(str(e))

        traceback.print_exc()

    conversion_results.append(
        result
    )


# ============================================================
# SAVE CONVERSION REPORT
# ============================================================

conversion_df = pd.DataFrame(
    conversion_results
)

conversion_csv = (
    FP16_DIR /
    "step3f_fp16_correction_results.csv"
)

conversion_df.to_csv(
    conversion_csv,
    index=False
)

print(
    "\n✓ Correction results saved:"
)

print(
    conversion_csv
)


# ============================================================
# CUDA SESSION + INFERENCE SANITY TEST
# ============================================================

section(
    "STEP 3-F — CUDA SESSION & INFERENCE SANITY TEST"
)

session_results = []


for name, paths in MODELS.items():

    subsection(
        f"CUDA TEST — {name}"
    )

    fp16_path = paths["fp16"]

    result = {
        "model": name,
        "session_status": "failed",
        "cuda_provider_selected": False,
        "input_type": "",
        "output_type": "",
        "input_shape": "",
        "output_shape": "",
        "inference_status": "not_attempted",
        "inference_time_ms": np.nan,
        "error": "",
    }

    if not fp16_path.exists():

        result["error"] = (
            "Corrected FP16 model does not exist"
        )

        print(
            "❌ FP16 artifact missing"
        )

        session_results.append(result)

        continue

    try:

        # ----------------------------------------------------
        # Create CUDA session
        # ----------------------------------------------------

        print(
            "→ Creating ONNX Runtime CUDA session..."
        )

        session_start = time.time()

        session = ort.InferenceSession(
            str(fp16_path),
            providers=[
                "CUDAExecutionProvider",
                "CPUExecutionProvider",
            ],
        )

        session_time = (
            time.time() -
            session_start
        )

        providers = (
            session.get_providers()
        )

        print("\nProviders:")

        for provider in providers:
            print(
                f"  - {provider}"
            )

        if (
            "CUDAExecutionProvider"
            not in providers
        ):

            raise RuntimeError(
                "CUDAExecutionProvider "
                "was not selected."
            )

        result[
            "session_status"
        ] = "success"

        result[
            "cuda_provider_selected"
        ] = True

        inputs = session.get_inputs()
        outputs = session.get_outputs()

        input_info = inputs[0]
        output_info = outputs[0]

        result[
            "input_type"
        ] = input_info.type

        result[
            "output_type"
        ] = output_info.type

        result[
            "input_shape"
        ] = str(input_info.shape)

        result[
            "output_shape"
        ] = str(output_info.shape)

        print(
            f"\nInput:"
            f" {input_info.name}"
            f" | {input_info.type}"
            f" | {input_info.shape}"
        )

        print(
            f"Output:"
            f" {output_info.name}"
            f" | {output_info.type}"
            f" | {output_info.shape}"
        )

        print(
            f"\nSession creation:"
            f" {session_time:.3f} s"
        )

        print(
            "✓ CUDA session created"
        )

        # ----------------------------------------------------
        # Sanity inference
        #
        # FP32 input because keep_io_types=True.
        # ----------------------------------------------------

        print(
            "\n→ Running single inference sanity test..."
        )

        dummy_input = np.random.rand(
            1,
            3,
            IMG_SIZE,
            IMG_SIZE
        ).astype(
            np.float32
        )

        inference_start = time.perf_counter()

        output = session.run(
            None,
            {
                input_info.name:
                    dummy_input
            }
        )

        inference_time = (
            time.perf_counter() -
            inference_start
        ) * 1000

        result[
            "inference_status"
        ] = "success"

        result[
            "inference_time_ms"
        ] = inference_time

        print(
            f"✓ Inference successful"
        )

        print(
            f"  Time : "
            f"{inference_time:.3f} ms"
        )

        print(
            f"  Outputs : "
            f"{len(output)}"
        )

        for i, arr in enumerate(output):

            print(
                f"  Output {i}: "
                f"shape={arr.shape}, "
                f"dtype={arr.dtype}"
            )

    except Exception as e:

        result[
            "error"
        ] = str(e)

        print(
            "\n❌ CUDA/inference test failed:"
        )

        print(str(e))

        traceback.print_exc()

    session_results.append(
        result
    )


# ============================================================
# SAVE SESSION REPORT
# ============================================================

session_df = pd.DataFrame(
    session_results
)

session_csv = (
    FP16_DIR /
    "step3f_cuda_sanity_results.csv"
)

session_df.to_csv(
    session_csv,
    index=False
)

print(
    "\n✓ CUDA sanity results saved:"
)

print(
    session_csv
)


# ============================================================
# FINAL SUMMARY
# ============================================================

section(
    "STEP 3-F — FINAL CORRECTION SUMMARY"
)

conversion_success = int(
    (
        conversion_df[
            "onnx_checker_status"
        ] == "passed"
    ).sum()
)

graph_success = int(
    (
        conversion_df[
            "graph_repair_status"
        ] == "success"
    ).sum()
)

session_success = int(
    (
        session_df[
            "session_status"
        ] == "success"
    ).sum()
)

inference_success = int(
    (
        session_df[
            "inference_status"
        ] == "success"
    ).sum()
)

print(
    f"""
FP16 conversion completed       : {
        int((conversion_df["conversion_status"] == "success").sum())
    }/5

Graph repair completed          : {
        graph_success
    }/5

ONNX checker passed             : {
        conversion_success
    }/5

CUDA sessions verified          : {
        session_success
    }/5

Inference sanity tests passed  : {
        inference_success
    }/5
"""
)


# ============================================================
# MODEL-BY-MODEL STATUS
# ============================================================

print(
    "\nMODEL STATUS:"
)

for name in MODELS.keys():

    conv_row = conversion_df[
        conversion_df["model"] == name
    ]

    sess_row = session_df[
        session_df["model"] == name
    ]

    if conv_row.empty:
        continue

    conv = conv_row.iloc[0]

    if not sess_row.empty:
        sess = sess_row.iloc[0]
    else:
        sess = None

    graph_status = conv[
        "onnx_checker_status"
    ]

    if sess is not None:
        infer_status = sess[
            "inference_status"
        ]
    else:
        infer_status = "not_tested"

    if (
        graph_status == "passed"
        and infer_status == "success"
    ):

        overall = "READY"

    else:

        overall = "NOT READY"

    print(
        f"\n  {name}"
    )

    print(
        f"    Graph checker : "
        f"{graph_status}"
    )

    print(
        f"    CUDA session  : "
        f"{'success' if sess is not None and sess['session_status'] == 'success' else 'failed'}"
    )

    print(
        f"    Inference     : "
        f"{infer_status}"
    )

    print(
        f"    STATUS        : "
        f"{overall}"
    )


# ============================================================
# SAVE EXPERIMENT LOG
# ============================================================

LOG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

log = {

    "step": "3F",

    "description": (
        "FP16 ONNX artifact correction, "
        "topological graph repair, "
        "ONNX validation, CUDA session "
        "verification and inference sanity test"
    ),

    "onnx_version":
        onnx.__version__,

    "onnxruntime_version":
        ort.__version__,

    "providers":
        ort.get_available_providers(),

    "keep_io_types":
        True,

    "models":
        list(MODELS.keys()),

    "conversion_results":
        conversion_results,

    "session_results":
        session_results,
}


log_path = (
    LOG_DIR /
    "step3f_fp16_correction_complete.json"
)

with open(
    log_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        log,
        f,
        indent=2,
        default=str
    )


# ============================================================
# OUTPUT FILES
# ============================================================

section(
    "STEP 3-F — OUTPUT FILES"
)

print("\nCorrected FP16 models:")

for name, paths in MODELS.items():

    if paths["fp16"].exists():

        print(
            f"  ✓ {name}"
        )
        print(
            f"    {paths['fp16']}"
        )

    else:

        print(
            f"  ❌ {name} — NOT CREATED"
        )


print(
    "\nCorrection report:"
)

print(
    conversion_csv
)

print(
    "\nCUDA sanity report:"
)

print(
    session_csv
)

print(
    "\nComplete log:"
)

print(
    log_path
)


# ============================================================
# FINAL DECISION
# ============================================================

all_graph_valid = (
    conversion_success == len(MODELS)
)

all_cuda_valid = (
    session_success == len(MODELS)
)

all_inference_valid = (
    inference_success == len(MODELS)
)

if (
    all_graph_valid
    and all_cuda_valid
    and all_inference_valid
):

    section(
        "STEP 3-F PASSED — FP16 ARTIFACTS ARE VALID"
    )

    print(
        """
All five FP16 artifacts:

    ✓ Converted
    ✓ Topologically ordered
    ✓ Passed ONNX checker
    ✓ Passed shape inference
    ✓ Loaded with CUDAExecutionProvider
    ✓ Successfully executed inference

The FP16 artifacts are now ready for
full dataset validation.

DO NOT START INT8 YET.

The next step will be the final
FP16 dataset validation and FP32-vs-FP16
comparison using the corrected artifacts.
"""
    )

else:

    section(
        "STEP 3-F NOT PASSED — DO NOT PROCEED"
    )

    print(
        """
At least one FP16 artifact failed
formal validation.

DO NOT proceed to INT8.

Review the model-specific errors
printed above and provide the complete
output for further correction.
"""
    )





STEP 3-F — FP16 ARTIFACT CORRECTION & VALIDATION
Environment:
  ONNX             : 1.22.0
  ONNX Runtime     : 1.26.0
  PyTorch          : 2.11.0+cu128
  CUDA available   : True

ONNX Runtime providers:
  - TensorrtExecutionProvider
  - CUDAExecutionProvider
  - CPUExecutionProvider

✓ Environment verified


STEP 3-F — VERIFYING PRESERVED FP32 SOURCE MODELS
✓ yolov8n_50ep            11.70 MB
✓ yolov8n_75ep            11.70 MB
✓ yolov8n_100ep           11.70 MB
✓ yolo11n                 10.12 MB
✓ rtdetr_50ep            125.53 MB

✓ All five FP32 source models are available


STEP 3-F — PREPARING CLEAN FP16 ARTIFACT DIRECTORY
→ Removing previous FP16 artifacts...
✓ Previous FP16 directory removed
✓ Clean directory created:
  /content/drive/MyDrive/Stage_PFA/MODEL_OPTIMIZATION/03_onnx_fp16


STEP 3-F — FP32 → FP16 CONVERSION + GRAPH REPAIR


----------------------------------------------------------------------------------------------------
PROCESSING — yolov8n_50ep
-------------------

## Step 3 Validation

In [ ]:
# ============================================================
# STEP 3-G — FINAL FP16 DATASET VALIDATION
# ============================================================
#
# PURPOSE:
#   Perform the FINAL dataset-level validation of the corrected
#   FP16 ONNX artifacts generated in Step 3-F.
#
# IMPORTANT:
#   - DOES NOT regenerate FP16 models
#   - DOES NOT delete FP16 models
#   - DOES NOT modify FP32 ONNX models
#   - DOES NOT modify original .PT models
#   - DOES NOT rerun FP32 dataset validation
#   - Uses the existing Step 2D FP32 results
#   - Validates all FIVE corrected FP16 models on the
#     existing 1,900-image TEST dataset
#
# FINAL COMPARISON:
#
#   FP32 ONNX
#       │
#       ├── Precision
#       ├── Recall
#       ├── mAP50
#       ├── mAP50-95
#       └── Inference / FPS
#              │
#              ▼
#        Corrected FP16 ONNX
#              │
#              ├── Precision
#              ├── Recall
#              ├── mAP50
#              ├── mAP50-95
#              └── Inference / FPS
#                       │
#                       ▼
#                  COMPARISON
#
#   Accuracy delta
#   Size reduction
#   Latency delta
#   FPS delta
#
# ============================================================

import os
import time
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import onnx
import onnxruntime as ort

from ultralytics import YOLO


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATA_YAML = (
    "/content/drive/MyDrive/PFE/Dataset/"
    "Vehicles_Dataset/data.yaml"
)

SPLIT = "test"

IMG_SIZE = 640

# Keep the same configuration used previously
BATCH_SIZE = 8
CONF = 0.001
IOU = 0.7
DEVICE = 0


# ============================================================
# 2. DIRECTORIES
# ============================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "MODEL_OPTIMIZATION"
)

FP32_DIR = BASE_DIR / "02_onnx_fp32"
FP16_DIR = BASE_DIR / "03_onnx_fp16"
BASELINE_DIR = BASE_DIR / "02_fp32_baseline"
LOG_DIR = BASE_DIR / "09_logs"


# ============================================================
# 3. EXISTING FP32 RESULTS
# ============================================================
#
# We deliberately reuse the FP32 results already generated
# during Step 2D.
#
# This saves a complete second pass over the 1,900-image
# test dataset for all five FP32 models.
#
# The notebook's previous Step 3 comparison confirms that
# these are the FP32 ONNX baseline values used for comparison.
#
# ============================================================

FP32_RESULTS_CANDIDATES = [

    BASELINE_DIR /
    "step2d_gpu_onnx_fp32_results.csv",

    BASELINE_DIR /
    "step2d_gpu_onnx_fp32_validation.csv",

    BASELINE_DIR /
    "step2d_gpu_onnx_results.csv",

    BASELINE_DIR /
    "step2_gpu_onnx_fp32_results.csv",
]


def find_existing_fp32_results():

    for path in FP32_RESULTS_CANDIDATES:

        if path.exists():

            return path

    return None


# ============================================================
# 4. MODEL REGISTRY
# ============================================================

MODELS = {

    "yolov8n_50ep": {

        "fp32":
            FP32_DIR /
            "yolov8n_50ep_fp32.onnx",

        "fp16":
            FP16_DIR /
            "yolov8n_50ep_fp16.onnx",
    },

    "yolov8n_75ep": {

        "fp32":
            FP32_DIR /
            "yolov8n_75ep_fp32.onnx",

        "fp16":
            FP16_DIR /
            "yolov8n_75ep_fp16.onnx",
    },

    "yolov8n_100ep": {

        "fp32":
            FP32_DIR /
            "yolov8n_100ep_fp32.onnx",

        "fp16":
            FP16_DIR /
            "yolov8n_100ep_fp16.onnx",
    },

    "yolo11n": {

        "fp32":
            FP32_DIR /
            "yolo11n_fp32.onnx",

        "fp16":
            FP16_DIR /
            "yolo11n_fp16.onnx",
    },

    "rtdetr_50ep": {

        "fp32":
            FP32_DIR /
            "rtdetr_50ep_fp32.onnx",

        "fp16":
            FP16_DIR /
            "rtdetr_50ep_fp16.onnx",
    },
}


# ============================================================
# 5. HELPERS
# ============================================================

def section(title):

    print("\n")
    print("=" * 100)
    print(title)
    print("=" * 100)


def subsection(title):

    print("\n")
    print("-" * 100)
    print(title)
    print("-" * 100)


def file_size_mb(path):

    return os.path.getsize(path) / (1024 ** 2)


def find_column(df, candidates):

    lower_map = {
        str(col).lower(): col
        for col in df.columns
    }

    for candidate in candidates:

        if candidate.lower() in lower_map:

            return lower_map[candidate.lower()]

    return None


def normalize_fp32_results(df):

    """
    Normalize the existing Step 2D FP32 results so that the
    final comparison uses a predictable set of column names.
    """

    rename_map = {}

    aliases = {

        "model": [
            "model",
            "model_name",
        ],

        "precision": [
            "precision",
            "box_precision",
            "P",
        ],

        "recall": [
            "recall",
            "box_recall",
            "R",
        ],

        "map50": [
            "map50",
            "mAP50",
            "mAP_50",
            "map_50",
        ],

        "map50_95": [
            "map50_95",
            "mAP50-95",
            "mAP50_95",
            "mAP_50_95",
            "map_50_95",
        ],

        "inference_ms": [
            "inference_ms",
            "inference",
            "inference_time_ms",
        ],
    }

    for target, names in aliases.items():

        col = find_column(
            df,
            names
        )

        if col is not None:

            rename_map[col] = target

    df = df.rename(
        columns=rename_map
    )

    required = [
        "model",
        "precision",
        "recall",
        "map50",
        "map50_95",
        "inference_ms",
    ]

    missing = [
        x for x in required
        if x not in df.columns
    ]

    if missing:

        raise RuntimeError(
            "\nCould not identify required columns "
            "in the existing FP32 results.\n\n"
            f"Missing: {missing}\n\n"
            f"Available columns:\n{list(df.columns)}"
        )

    return df


# ============================================================
# 6. ENVIRONMENT VERIFICATION
# ============================================================

section(
    "STEP 3-G — FINAL FP16 DATASET VALIDATION"
)

print("Environment:")

print(
    f"  ONNX            : "
    f"{onnx.__version__}"
)

print(
    f"  ONNX Runtime    : "
    f"{ort.__version__}"
)

print(
    f"  PyTorch         : "
    f"{torch.__version__}"
)

print(
    f"  Ultralytics     : "
    f"{__import__('ultralytics').__version__}"
)

print(
    f"  CUDA available  : "
    f"{torch.cuda.is_available()}"
)


if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU is not available."
    )


providers = ort.get_available_providers()

print("\nONNX Runtime providers:")

for provider in providers:

    print(
        f"  - {provider}"
    )


if "CUDAExecutionProvider" not in providers:

    raise RuntimeError(
        "CUDAExecutionProvider is not available."
    )


GPU_NAME = torch.cuda.get_device_name(0)

print(
    f"\nGPU             : {GPU_NAME}"
)

print(
    f"CUDA            : "
    f"{torch.version.cuda}"
)

print(
    "\n✓ GPU and ONNX Runtime CUDA "
    "environment verified"
)


# ============================================================
# 7. DATASET VERIFICATION
# ============================================================

section(
    "STEP 3-G — DATASET VERIFICATION"
)

if not os.path.exists(DATA_YAML):

    raise FileNotFoundError(
        f"Dataset YAML not found:\n{DATA_YAML}"
    )

print(
    f"✓ Dataset YAML found:\n"
    f"  {DATA_YAML}"
)

print(
    f"\nSplit       : {SPLIT}"
)

print(
    f"Image size  : {IMG_SIZE} × {IMG_SIZE}"
)

print(
    f"Batch size  : {BATCH_SIZE}"
)

print(
    f"Confidence  : {CONF}"
)

print(
    f"IoU         : {IOU}"
)

print(
    f"Device      : CUDA:{DEVICE}"
)


# ============================================================
# 8. VERIFY CORRECTED FP16 ARTIFACTS
# ============================================================

section(
    "STEP 3-G — VERIFYING CORRECTED FP16 ARTIFACTS"
)

for name, paths in MODELS.items():

    fp16_path = paths["fp16"]

    if not fp16_path.exists():

        raise FileNotFoundError(
            f"\nCorrected FP16 artifact missing:\n"
            f"{fp16_path}"
        )

    print(
        f"✓ {name:<20}"
        f"{file_size_mb(fp16_path):>10.2f} MB"
    )


print(
    "\n✓ All five corrected FP16 artifacts found"
)

print(
    "\nIMPORTANT:"
)

print(
    "  FP16 files will NOT be deleted."
)

print(
    "  FP16 files will NOT be regenerated."
)

print(
    "  FP32 source files will NOT be modified."
)


# ============================================================
# 9. LOAD EXISTING FP32 BASELINE
# ============================================================

section(
    "STEP 3-G — LOADING EXISTING STEP 2D FP32 BASELINE"
)

fp32_results_path = (
    find_existing_fp32_results()
)


if fp32_results_path is None:

    raise FileNotFoundError(
        "\nCould not find the existing Step 2D "
        "FP32 ONNX results CSV.\n\n"
        "Expected one of:\n"
        + "\n".join(
            str(x)
            for x in FP32_RESULTS_CANDIDATES
        )
        + "\n\n"
        "No FP32 dataset re-evaluation will be "
        "performed automatically because the goal "
        "of Step 3-G is to minimize Colab usage."
    )


print(
    f"✓ FP32 baseline found:\n"
    f"  {fp32_results_path}"
)

fp32_df = pd.read_csv(
    fp32_results_path
)

print(
    f"\nRows loaded : {len(fp32_df)}"
)

print(
    f"Columns     : "
    f"{list(fp32_df.columns)}"
)

fp32_df = normalize_fp32_results(
    fp32_df
)

print(
    "\n✓ FP32 baseline normalized"
)


# ============================================================
# 10. VERIFY ALL FIVE FP32 BASELINE ROWS
# ============================================================

missing_models = []

for model_name in MODELS.keys():

    matches = fp32_df[
        fp32_df["model"].astype(str)
        == model_name
    ]

    if matches.empty:

        missing_models.append(
            model_name
        )


if missing_models:

    raise RuntimeError(
        "\nMissing FP32 baseline results for:\n"
        + "\n".join(
            f"  - {x}"
            for x in missing_models
        )
    )


print(
    "✓ FP32 baseline contains all five models"
)


# ============================================================
# 11. FP16 DATASET VALIDATION FUNCTION
# ============================================================

def validate_fp16_model(
    model_name,
    model_path
):

    subsection(
        f"FP16 DATASET VALIDATION — {model_name}"
    )

    print(
        f"Model:\n  {model_path}"
    )

    size_mb = file_size_mb(
        model_path
    )

    print(
        f"\nFP16 model size : "
        f"{size_mb:.2f} MB"
    )

    try:

        print(
            "\n→ Creating Ultralytics ONNX model..."
        )

        model = YOLO(
            str(model_path),
            task="detect"
        )

        print(
            "✓ Model loaded"
        )

        print(
            "\n→ Running test dataset validation..."
        )

        validation_start = time.perf_counter()

        results = model.val(

            data=DATA_YAML,

            split=SPLIT,

            imgsz=IMG_SIZE,

            batch=BATCH_SIZE,

            device=DEVICE,

            conf=CONF,

            iou=IOU,

            verbose=True,

            plots=False,

            save_json=False,

        )

        total_validation_time = (
            time.perf_counter()
            - validation_start
        )

        # ----------------------------------------------------
        # Extract metrics
        # ----------------------------------------------------

        metrics = results.box

        precision = float(
            metrics.mp
        )

        recall = float(
            metrics.mr
        )

        map50 = float(
            metrics.map50
        )

        map50_95 = float(
            metrics.map
        )

        # ----------------------------------------------------
        # Extract timing
        # ----------------------------------------------------

        speed = getattr(
            results,
            "speed",
            {}
        )

        preprocess_ms = float(
            speed.get(
                "preprocess",
                np.nan
            )
        )

        inference_ms = float(
            speed.get(
                "inference",
                np.nan
            )
        )

        postprocess_ms = float(
            speed.get(
                "postprocess",
                np.nan
            )
        )

        total_per_image_ms = (
            preprocess_ms
            + inference_ms
            + postprocess_ms
        )

        fps = (
            1000.0 /
            total_per_image_ms
            if total_per_image_ms > 0
            else np.nan
        )

        print(
            "\n✓ FP16 validation completed"
        )

        print(
            f"\nPrecision : "
            f"{precision:.6f}"
        )

        print(
            f"Recall    : "
            f"{recall:.6f}"
        )

        print(
            f"mAP50     : "
            f"{map50:.6f}"
        )

        print(
            f"mAP50-95  : "
            f"{map50_95:.6f}"
        )

        print(
            f"Preprocess: "
            f"{preprocess_ms:.3f} ms"
        )

        print(
            f"Inference : "
            f"{inference_ms:.3f} ms"
        )

        print(
            f"Postprocess: "
            f"{postprocess_ms:.3f} ms"
        )

        print(
            f"Total/img : "
            f"{total_per_image_ms:.3f} ms"
        )

        print(
            f"FPS       : "
            f"{fps:.3f}"
        )

        print(
            f"Validation wall time : "
            f"{total_validation_time:.2f} s"
        )

        return {

            "model":
                model_name,

            "status":
                "success",

            "fp16_size_mb":
                size_mb,

            "precision":
                precision,

            "recall":
                recall,

            "map50":
                map50,

            "map50_95":
                map50_95,

            "preprocess_ms":
                preprocess_ms,

            "inference_ms":
                inference_ms,

            "postprocess_ms":
                postprocess_ms,

            "total_per_image_ms":
                total_per_image_ms,

            "fps":
                fps,

            "validation_wall_time_s":
                total_validation_time,

        }

    except Exception as e:

        print(
            "\n❌ FP16 validation failed:"
        )

        print(
            str(e)
        )

        return {

            "model":
                model_name,

            "status":
                "failed",

            "fp16_size_mb":
                size_mb,

            "precision":
                np.nan,

            "recall":
                np.nan,

            "map50":
                np.nan,

            "map50_95":
                np.nan,

            "preprocess_ms":
                np.nan,

            "inference_ms":
                np.nan,

            "postprocess_ms":
                np.nan,

            "total_per_image_ms":
                np.nan,

            "fps":
                np.nan,

            "validation_wall_time_s":
                np.nan,

            "error":
                str(e),

        }

    finally:

        try:

            del model

        except Exception:

            pass

        try:

            del results

        except Exception:

            pass

        gc.collect()

        torch.cuda.empty_cache()


# ============================================================
# 12. RUN ALL FIVE FP16 VALIDATIONS
# ============================================================

section(
    "STEP 3-G — RUNNING ALL FIVE FP16 DATASET VALIDATIONS"
)

print(
    """
The corrected FP16 models will now be evaluated
against the existing 1,900-image test dataset.

FP32 dataset validation is NOT repeated.

This is intentional and saves GPU/Colab time.
"""
)

fp16_results = []


for model_name, paths in MODELS.items():

    result = validate_fp16_model(
        model_name=model_name,
        model_path=paths["fp16"]
    )

    fp16_results.append(
        result
    )

    # Small cleanup between models
    gc.collect()
    torch.cuda.empty_cache()


fp16_df = pd.DataFrame(
    fp16_results
)


# ============================================================
# 13. VERIFY ALL FIVE FP16 VALIDATIONS
# ============================================================

section(
    "STEP 3-G — FP16 VALIDATION STATUS"
)

successful_count = int(
    (
        fp16_df["status"]
        == "success"
    ).sum()
)

print(
    f"\nFP16 validations completed:"
    f" {successful_count}/5"
)

if successful_count != 5:

    print(
        "\n⚠️ One or more FP16 validations failed."
    )

    print(
        fp16_df[
            [
                "model",
                "status",
                "error"
            ]
        ]
    )

    raise RuntimeError(
        "Step 3-G cannot be considered passed "
        "until all five FP16 models validate successfully."
    )


print(
    "\n✓ All five corrected FP16 models "
    "validated successfully"
)


# ============================================================
# 14. BUILD FINAL FP32 vs FP16 COMPARISON
# ============================================================

section(
    "STEP 3-G — FINAL FP32 vs FP16 COMPARISON"
)

comparison_rows = []


for model_name in MODELS.keys():

    fp32 = fp32_df[
        fp32_df["model"].astype(str)
        == model_name
    ].iloc[0]

    fp16 = fp16_df[
        fp16_df["model"].astype(str)
        == model_name
    ].iloc[0]

    fp32_size = file_size_mb(
        MODELS[model_name]["fp32"]
    )

    fp16_size = float(
        fp16["fp16_size_mb"]
    )

    size_reduction = (
        (
            fp32_size -
            fp16_size
        )
        / fp32_size
    ) * 100.0

    precision_delta = (
        fp16["precision"]
        - fp32["precision"]
    )

    recall_delta = (
        fp16["recall"]
        - fp32["recall"]
    )

    map50_delta = (
        fp16["map50"]
        - fp32["map50"]
    )

    map50_95_delta = (
        fp16["map50_95"]
        - fp32["map50_95"]
    )

    latency_delta_ms = (
        fp16["inference_ms"]
        - fp32["inference_ms"]
    )

    latency_delta_percent = (
        latency_delta_ms
        / fp32["inference_ms"]
    ) * 100.0

    fps_delta = (
        fp16["fps"]
        - (
            1000.0 /
            (
                float(fp32.get("inference_ms", np.nan))
                if float(fp32.get("inference_ms", np.nan)) > 0
                else np.nan
            )
        )
    )

    # --------------------------------------------------------
    # Accuracy assessment
    # --------------------------------------------------------

    if abs(map50_95_delta) <= 0.005:

        accuracy_assessment = (
            "Equivalent — negligible difference"
        )

    elif map50_95_delta > 0:

        accuracy_assessment = (
            "Improved"
        )

    else:

        accuracy_assessment = (
            "Reduced — investigate"
        )

    # --------------------------------------------------------
    # Latency assessment
    # --------------------------------------------------------

    if latency_delta_percent < -2:

        latency_assessment = (
            "Faster"
        )

    elif latency_delta_percent > 2:

        latency_assessment = (
            "Slower"
        )

    else:

        latency_assessment = (
            "Similar"
        )

    comparison_rows.append({

        "model":
            model_name,

        # ----------------------------------------------------
        # MODEL SIZE
        # ----------------------------------------------------

        "fp32_size_mb":
            fp32_size,

        "fp16_size_mb":
            fp16_size,

        "size_reduction_percent":
            size_reduction,

        # ----------------------------------------------------
        # PRECISION
        # ----------------------------------------------------

        "fp32_precision":
            float(fp32["precision"]),

        "fp16_precision":
            float(fp16["precision"]),

        "precision_delta":
            precision_delta,

        # ----------------------------------------------------
        # RECALL
        # ----------------------------------------------------

        "fp32_recall":
            float(fp32["recall"]),

        "fp16_recall":
            float(fp16["recall"]),

        "recall_delta":
            recall_delta,

        # ----------------------------------------------------
        # mAP50
        # ----------------------------------------------------

        "fp32_map50":
            float(fp32["map50"]),

        "fp16_map50":
            float(fp16["map50"]),

        "map50_delta":
            map50_delta,

        # ----------------------------------------------------
        # mAP50-95
        # ----------------------------------------------------

        "fp32_map50_95":
            float(fp32["map50_95"]),

        "fp16_map50_95":
            float(fp16["map50_95"]),

        "map50_95_delta":
            map50_95_delta,

        # ----------------------------------------------------
        # LATENCY
        # ----------------------------------------------------

        "fp32_inference_ms":
            float(fp32["inference_ms"]),

        "fp16_inference_ms":
            float(fp16["inference_ms"]),

        "latency_delta_ms":
            latency_delta_ms,

        "latency_delta_percent":
            latency_delta_percent,

        # ----------------------------------------------------
        # FPS
        # ----------------------------------------------------

        "fp32_fps":
            (
                1000.0 /
                float(fp32["inference_ms"])
                if float(fp32["inference_ms"]) > 0
                else np.nan
            ),

        "fp16_fps":
            float(fp16["fps"]),

        "fps_delta":
            fps_delta,

        # ----------------------------------------------------
        # ASSESSMENT
        # ----------------------------------------------------

        "accuracy_assessment":
            accuracy_assessment,

        "latency_assessment":
            latency_assessment,

    })


comparison_df = pd.DataFrame(
    comparison_rows
)


# ============================================================
# 15. PRINT FINAL MODEL-BY-MODEL RESULTS
# ============================================================

for _, row in comparison_df.iterrows():

    subsection(
        row["model"]
    )

    print(
        f"""
MODEL SIZE
  FP32 : {row['fp32_size_mb']:.2f} MB
  FP16 : {row['fp16_size_mb']:.2f} MB
  Reduction : {row['size_reduction_percent']:.2f} %

PRECISION
  FP32 : {row['fp32_precision']:.6f}
  FP16 : {row['fp16_precision']:.6f}
  Δ    : {row['precision_delta']:+.6f}

RECALL
  FP32 : {row['fp32_recall']:.6f}
  FP16 : {row['fp16_recall']:.6f}
  Δ    : {row['recall_delta']:+.6f}

mAP50
  FP32 : {row['fp32_map50']:.6f}
  FP16 : {row['fp16_map50']:.6f}
  Δ    : {row['map50_delta']:+.6f}

mAP50-95
  FP32 : {row['fp32_map50_95']:.6f}
  FP16 : {row['fp16_map50_95']:.6f}
  Δ    : {row['map50_95_delta']:+.6f}

INFERENCE
  FP32 : {row['fp32_inference_ms']:.3f} ms
  FP16 : {row['fp16_inference_ms']:.3f} ms
  Δ    : {row['latency_delta_ms']:+.3f} ms
  Δ%   : {row['latency_delta_percent']:+.2f} %

FPS
  FP32 : {row['fp32_fps']:.3f}
  FP16 : {row['fp16_fps']:.3f}
  Δ    : {row['fps_delta']:+.3f}

ACCURACY ASSESSMENT
  {row['accuracy_assessment']}

LATENCY ASSESSMENT
  {row['latency_assessment']}
"""
    )


# ============================================================
# 16. SAVE FINAL RESULTS
# ============================================================

section(
    "STEP 3-G — SAVING FINAL RESULTS"
)

FP16_VALIDATION_CSV = (
    BASELINE_DIR /
    "step3g_final_fp16_dataset_validation.csv"
)

COMPARISON_CSV = (
    BASELINE_DIR /
    "step3g_final_fp32_vs_fp16_comparison.csv"
)

FP16_VALIDATION_CSV.parent.mkdir(
    parents=True,
    exist_ok=True
)

fp16_df.to_csv(
    FP16_VALIDATION_CSV,
    index=False
)

comparison_df.to_csv(
    COMPARISON_CSV,
    index=False
)

print(
    f"✓ FP16 validation results saved:\n"
    f"  {FP16_VALIDATION_CSV}"
)

print(
    f"\n✓ Final FP32 vs FP16 comparison saved:\n"
    f"  {COMPARISON_CSV}"
)


# ============================================================
# 17. FINAL PASS/FAIL LOGIC
# ============================================================

section(
    "STEP 3-G — FINAL FP16 DECISION"
)

all_accuracy_equivalent = all(
    abs(x) <= 0.005
    for x in comparison_df[
        "map50_95_delta"
    ]
)

all_size_reduced = all(
    comparison_df[
        "size_reduction_percent"
    ] > 0
)

if (
    successful_count == 5
    and all_accuracy_equivalent
    and all_size_reduced
):

    final_status = "PASSED"

else:

    final_status = "REQUIRES REVIEW"


print(
    f"""
FINAL STATUS : {final_status}

FP16 dataset validations : {successful_count}/5
Accuracy preservation    : {
    "PASS" if all_accuracy_equivalent
    else "REVIEW"
}
Model-size reduction     : {
    "PASS" if all_size_reduced
    else "REVIEW"
}
"""
)


# ============================================================
# 18. COMPLETE EXPERIMENT LOG
# ============================================================

LOG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

log = {

    "step":
        "3-G",

    "description":
        (
            "Final dataset-level validation of corrected "
            "FP16 ONNX artifacts against the project test "
            "dataset, using the existing Step 2D FP32 "
            "ONNX baseline."
        ),

    "dataset":
        DATA_YAML,

    "split":
        SPLIT,

    "image_size":
        IMG_SIZE,

    "batch_size":
        BATCH_SIZE,

    "confidence":
        CONF,

    "iou":
        IOU,

    "device":
        f"CUDA:{DEVICE}",

    "gpu":
        GPU_NAME,

    "onnx_version":
        onnx.__version__,

    "onnxruntime_version":
        ort.__version__,

    "providers":
        providers,

    "fp32_baseline_source":
        str(fp32_results_path),

    "fp16_validation_results":
        fp16_results,

    "comparison_results":
        comparison_rows,

    "final_status":
        final_status,

    "fp16_models":
        {
            name: str(
                paths["fp16"]
            )
            for name, paths in MODELS.items()
        },

    "fp32_models":
        {
            name: str(
                paths["fp32"]
            )
            for name, paths in MODELS.items()
        },

    "important_notes":
        [
            "FP16 artifacts were NOT regenerated.",
            "FP16 artifacts were NOT deleted.",
            "Original FP32 ONNX models were NOT modified.",
            "Original PT models were NOT modified.",
            "FP32 dataset validation was intentionally not repeated.",
            "Existing Step 2D FP32 baseline was reused.",
        ],
}


LOG_PATH = (
    LOG_DIR /
    "step3g_final_fp16_dataset_validation.json"
)

with open(
    LOG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        log,
        f,
        indent=2,
        default=str
    )


print(
    f"\n✓ Complete Step 3-G log saved:\n"
    f"  {LOG_PATH}"
)


# ============================================================
# 19. FINAL SUMMARY TABLE
# ============================================================

section(
    "STEP 3-G — FINAL SUMMARY TABLE"
)

summary_columns = [

    "model",

    "fp32_map50_95",
    "fp16_map50_95",
    "map50_95_delta",

    "fp32_precision",
    "fp16_precision",
    "precision_delta",

    "fp32_recall",
    "fp16_recall",
    "recall_delta",

    "fp32_inference_ms",
    "fp16_inference_ms",
    "latency_delta_ms",

    "fp32_fps",
    "fp16_fps",
    "fps_delta",

    "fp32_size_mb",
    "fp16_size_mb",
    "size_reduction_percent",

    "accuracy_assessment",
]

print(
    comparison_df[
        summary_columns
    ].to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


# ============================================================
# 20. FINAL INSTRUCTION
# ============================================================

print(
    """
====================================================================

STEP 3-G COMPLETE

If FINAL STATUS = PASSED:

    STEP 3 — FP16 ✓ COMPLETE

Do NOT start INT8 inside this cell.

The next phase will be:

    STEP 4 — INT8

with:

    1. Calibration dataset preparation
    2. INT8 conversion
    3. ONNX graph validation
    4. ONNX Runtime verification
    5. Accuracy evaluation
    6. FP32 vs FP16 vs INT8 comparison

====================================================================
"""
)



STEP 3-G — FINAL FP16 DATASET VALIDATION
Environment:
  ONNX            : 1.22.0
  ONNX Runtime    : 1.26.0
  PyTorch         : 2.11.0+cu128
  Ultralytics     : 8.4.131
  CUDA available  : True

ONNX Runtime providers:
  - TensorrtExecutionProvider
  - CUDAExecutionProvider
  - CPUExecutionProvider

GPU             : Tesla T4
CUDA            : 12.8

✓ GPU and ONNX Runtime CUDA environment verified


STEP 3-G — DATASET VERIFICATION
✓ Dataset YAML found:
  /content/drive/MyDrive/PFE/Dataset/Vehicles_Dataset/data.yaml

Split       : test
Image size  : 640 × 640
Batch size  : 8
Confidence  : 0.001
IoU         : 0.7
Device      : CUDA:0


STEP 3-G — VERIFYING CORRECTED FP16 ARTIFACTS
✓ yolov8n_50ep              5.89 MB
✓ yolov8n_75ep              5.89 MB
✓ yolov8n_100ep             5.89 MB
✓ yolo11n                   5.11 MB
✓ rtdetr_50ep              62.98 MB

✓ All five corrected FP16 artifacts found

IMPORTANT:
  FP16 files will NOT be deleted.
  FP16 files will NOT be regenerated.
  F

## Continuing with the new step 4

In [ ]:
# ============================================================
# STEP 4 — INT8 STATIC QUANTIZATION
# CALIBRATION + CONVERSION ON COLAB GPU (CONSISTENT WITH
# STEPS 2 AND 3 — SAME RUNTIME, SAME DIRECTORY PLAN)
# ============================================================
#
# PURPOSE:
#   PHASE A: Environment verification (GPU, same as Steps 2/3)
#   PHASE B: Build a representative calibration dataset
#   PHASE C: Run ONNX Runtime static INT8 (QDQ) quantization
#   PHASE D: Validate INT8 ONNX graph structure
#
# WORKFLOW CONSISTENCY:
#   Steps 2 (FP32) and 3 (FP16) both ran on the Colab GPU
#   runtime (CUDA:0, Tesla T4). Step 4 follows the same
#   runtime for calibration and conversion, so results stay
#   directly comparable across the whole notebook and the
#   FP32/FP16/INT8 numbers in your report all come from the
#   same execution environment.
#
#   On-device CPU/ARM behavior (the actual Raspberry Pi
#   target) is measured separately in STEP 5, once the models
#   are transferred to the physical Pi. That is intentionally
#   a distinct phase, not mixed into this GPU-based pipeline.
#
#   Folder layout follows the DIRECTORIES plan already defined
#   in Step 1C of this notebook:
#       01_original     — original checkpoint references
#       02_onnx_fp32    — FP32 ONNX exports
#       03_onnx_fp16    — FP16 ONNX exports
#       04_onnx_int8    — INT8 ONNX exports (this step)
#       05_calibration  — calibration image list / cache
#       06_validation   — dataset validation results
#       07_benchmarks   — benchmark / comparison tables
#       08_final        — final selected production model(s)
#       09_logs         — experiment logs (JSON)
#
#   - Does NOT modify FP32 ONNX models
#   - Does NOT modify FP16 ONNX models
#   - Does NOT modify original .PT models
#   - Deletes only previous Step 4 INT8 outputs
#
# ============================================================

import os
import sys
import json
import time
import shutil
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import onnx
import onnxruntime as ort
import cv2

from ultralytics import YOLO

warnings.filterwarnings("ignore")


# ============================================================
# HELPERS
# ============================================================

def section(title):
    print("\n")
    print("=" * 100)
    print(title)
    print("=" * 100)


def subsection(title):
    print("\n")
    print("-" * 100)
    print(title)
    print("-" * 100)


def file_size_mb(path):
    return os.path.getsize(path) / (1024 ** 2)


# ============================================================
# STEP 4A — ENVIRONMENT VERIFICATION (GPU — SAME AS STEPS 2/3)
# ============================================================

section("STEP 4A — INT8 GPU ENVIRONMENT VERIFICATION")

print("→ Checking Python packages...")

try:
    import onnx
    print(f"✓ ONNX available")
    print(f"  ONNX version : {onnx.__version__}")
except Exception as e:
    raise RuntimeError(f"ONNX import failed: {e}")

try:
    import onnxruntime
    print(f"✓ ONNX Runtime available")
    print(f"  ONNX Runtime version : {onnxruntime.__version__}")
except Exception as e:
    raise RuntimeError(f"ONNX Runtime import failed: {e}")

try:
    from onnxruntime.quantization import (
        quantize_static,
        CalibrationDataReader,
        QuantType,
        QuantFormat,
        CalibrationMethod,
    )
    print("✓ ONNX Runtime static quantization toolkit available")
except Exception as e:
    raise RuntimeError(
        "\nCould not import onnxruntime.quantization.\n"
        f"Error:\n{e}\n"
    )

try:
    import cv2
    print(f"✓ OpenCV available ({cv2.__version__})")
except Exception as e:
    raise RuntimeError(f"OpenCV import failed: {e}")

print("\nUltralytics version :", __import__("ultralytics").__version__)
print("PyTorch version     :", torch.__version__)
print("CUDA available      :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "\nCUDA GPU is NOT available.\n"
        "Go to:\n"
        "Runtime → Change runtime type → GPU\n"
        "and reconnect the runtime.\n"
        "(Steps 2 and 3 both required CUDA — Step 4 keeps "
        "the same runtime for consistency.)"
    )

print("CUDA version        :", torch.version.cuda)
print("GPU                 :", torch.cuda.get_device_name(0))

gpu_memory_gb = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
)

print(f"GPU memory          : {gpu_memory_gb:.2f} GB")

print("\n✓ PyTorch CUDA GPU verified")

available_providers = ort.get_available_providers()

print("\nAvailable ONNX Runtime providers:")
for provider in available_providers:
    print(f"  - {provider}")

if "CUDAExecutionProvider" not in available_providers:
    raise RuntimeError(
        "\nCUDAExecutionProvider is NOT available.\n"
        "Do not continue with Step 4."
    )

print("\n✓ CUDAExecutionProvider is available")
print("✓ ONNX Runtime GPU is ready")
print(
    "\nNote: calibration inference during quantization runs on "
    "CPU regardless (ONNX Runtime's CalibrationDataReader path "
    "does not use CUDA), but conversion/validation of the "
    "resulting artifacts follows the notebook's GPU runtime, "
    "matching Steps 2 and 3."
)


# ============================================================
# STEP 4 CONFIGURATION
# ============================================================

section("STEP 4 — INT8 CONFIGURATION")

DATA_YAML = (
    "/content/drive/MyDrive/PFE/Dataset/"
    "Vehicles_Dataset/data.yaml"
)

DATASET_ROOT = (
    "/content/drive/MyDrive/PFE/Dataset/Vehicles_Dataset"
)

SPLIT = "test"

IMG_SIZE = 640
BATCH_SIZE = 8
CONF = 0.001
IOU = 0.7

# GPU validation — same runtime as Steps 2 and 3
DEVICE = "cuda:0"

# Calibration set size (per your decision)
N_CALIBRATION_IMAGES = 250

RANDOM_SEED = 42

# ------------------------------------------------------------
# Directories — following the DIRECTORIES plan from Step 1C
# ------------------------------------------------------------

STAGE_PFA = "/content/drive/MyDrive/Stage_PFA"
OPT_ROOT = os.path.join(STAGE_PFA, "MODEL_OPTIMIZATION")

FP32_DIR = Path(OPT_ROOT) / "02_onnx_fp32"
FP16_DIR = Path(OPT_ROOT) / "03_onnx_fp16"
BASELINE_DIR = Path(OPT_ROOT) / "02_fp32_baseline"
LOG_DIR = Path(OPT_ROOT) / "09_logs"

INT8_DIR = Path(OPT_ROOT) / "04_onnx_int8"
CALIB_DIR = Path(OPT_ROOT) / "05_calibration"
VALIDATION_DIR = Path(OPT_ROOT) / "06_validation"
BENCHMARKS_DIR = Path(OPT_ROOT) / "07_benchmarks"
FINAL_DIR = Path(OPT_ROOT) / "08_final"

for d in (INT8_DIR, CALIB_DIR, VALIDATION_DIR, BENCHMARKS_DIR,
          FINAL_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"""
Dataset          : {DATA_YAML}
Split            : {SPLIT}
Image size       : {IMG_SIZE} × {IMG_SIZE}
Batch size       : {BATCH_SIZE}
Confidence       : {CONF}
IoU              : {IOU}
Validation device: {DEVICE}  (same GPU runtime as Steps 2 and 3)

Calibration images : {N_CALIBRATION_IMAGES}
Random seed         : {RANDOM_SEED}

INT8 method:
    ONNX Runtime static quantization (QDQ format)
    Per-channel weight quantization
    MinMax calibration

Quant format     : QDQ (QuantFormat.QDQ)
Weight type      : QInt8
Activation type  : QUInt8

FP32 ONNX source : PRESERVED
FP16 ONNX source : PRESERVED
Original .PT     : PRESERVED

Directory plan (from Step 1C):
    INT8 exports      : {INT8_DIR}
    Calibration data  : {CALIB_DIR}
    Validation results: {VALIDATION_DIR}
    Benchmarks        : {BENCHMARKS_DIR}
    Final model(s)     : {FINAL_DIR}
    Logs               : {LOG_DIR}
""")


# ============================================================
# MODEL REGISTRY
# ============================================================

MODELS = {
    "yolov8n_50ep": {
        "fp32": FP32_DIR / "yolov8n_50ep_fp32.onnx",
        "fp16": FP16_DIR / "yolov8n_50ep_fp16.onnx",
        "int8": INT8_DIR / "yolov8n_50ep_int8.onnx",
        "int8_preprocessed": INT8_DIR / "yolov8n_50ep_preproc.onnx",
    },
    "yolov8n_75ep": {
        "fp32": FP32_DIR / "yolov8n_75ep_fp32.onnx",
        "fp16": FP16_DIR / "yolov8n_75ep_fp16.onnx",
        "int8": INT8_DIR / "yolov8n_75ep_int8.onnx",
        "int8_preprocessed": INT8_DIR / "yolov8n_75ep_preproc.onnx",
    },
    "yolov8n_100ep": {
        "fp32": FP32_DIR / "yolov8n_100ep_fp32.onnx",
        "fp16": FP16_DIR / "yolov8n_100ep_fp16.onnx",
        "int8": INT8_DIR / "yolov8n_100ep_int8.onnx",
        "int8_preprocessed": INT8_DIR / "yolov8n_100ep_preproc.onnx",
    },
    "yolo11n": {
        "fp32": FP32_DIR / "yolo11n_fp32.onnx",
        "fp16": FP16_DIR / "yolo11n_fp16.onnx",
        "int8": INT8_DIR / "yolo11n_int8.onnx",
        "int8_preprocessed": INT8_DIR / "yolo11n_preproc.onnx",
    },
    "rtdetr_50ep": {
        "fp32": FP32_DIR / "rtdetr_50ep_fp32.onnx",
        "fp16": FP16_DIR / "rtdetr_50ep_fp16.onnx",
        "int8": INT8_DIR / "rtdetr_50ep_int8.onnx",
        "int8_preprocessed": INT8_DIR / "rtdetr_50ep_preproc.onnx",
    },
}


# ============================================================
# FP32 MODEL VERIFICATION
# ============================================================

section("FP32 ONNX MODEL VERIFICATION")

for name, paths in MODELS.items():
    fp32_path = paths["fp32"]

    if not fp32_path.exists():
        raise FileNotFoundError(
            f"\nMissing FP32 ONNX model:\n{fp32_path}"
        )

    print(f"\n✓ {name}")
    print(f"  File : {fp32_path}")
    print(f"  Size : {file_size_mb(fp32_path):.2f} MB")

print("\n✓ All FP32 ONNX models found")


# ============================================================
# STEP 4B — CLEAN PREVIOUS INT8 OUTPUTS
# ============================================================

section("STEP 4B — CLEANING PREVIOUS INT8 OUTPUTS")

print(f"""
INT8 directory:

{INT8_DIR}

IMPORTANT:
All previous INT8 ONNX models in this directory will be deleted.
FP32 and FP16 ONNX directories will NOT be touched.
""")

if INT8_DIR.exists():
    print("→ Removing previous INT8 directory...")
    shutil.rmtree(INT8_DIR)
    print("✓ Previous INT8 directory deleted")

INT8_DIR.mkdir(parents=True, exist_ok=True)
print("✓ New clean INT8 directory created")

remaining_onnx = list(INT8_DIR.glob("*.onnx"))
if len(remaining_onnx) != 0:
    raise RuntimeError("Old INT8 ONNX files still exist!")

print("\n✓ INT8 directory is completely clean")

old_result_files = [
    INT8_DIR / "step4_int8_conversion_results.csv",
    VALIDATION_DIR / "step4_gpu_int8_dataset_validation.csv",
    BENCHMARKS_DIR / "step4_final_fp32_vs_fp16_vs_int8_comparison.csv",
    LOG_DIR / "step4_int8_calibration_experiment.json",
]

for path in old_result_files:
    if path.exists():
        print(f"→ Deleting old result:\n  {path}")
        path.unlink()
        print("  ✓ Deleted")

print("\n✓ Previous Step 4 result files cleaned")


# ============================================================
# STEP 4C — BUILD CALIBRATION DATASET
# ============================================================

section("STEP 4C — CALIBRATION DATASET PREPARATION")

# ------------------------------------------------------------
# Locate test images
# ------------------------------------------------------------

test_images_dir = Path(DATASET_ROOT) / "test" / "images"

if not test_images_dir.exists():
    # fallback: some Ultralytics dataset layouts use images/test
    alt = Path(DATASET_ROOT) / "images" / "test"
    if alt.exists():
        test_images_dir = alt
    else:
        raise FileNotFoundError(
            f"\nCould not locate test images directory.\n"
            f"Tried:\n  {test_images_dir}\n  {alt}"
        )

print(f"✓ Test images directory found:\n  {test_images_dir}")

all_images = sorted([
    p for p in test_images_dir.iterdir()
    if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")
])

print(f"\nTotal test images available : {len(all_images)}")

if len(all_images) < N_CALIBRATION_IMAGES:
    raise RuntimeError(
        f"Not enough test images ({len(all_images)}) "
        f"for calibration set of {N_CALIBRATION_IMAGES}."
    )

random.seed(RANDOM_SEED)
calibration_images = random.sample(all_images, N_CALIBRATION_IMAGES)

print(f"✓ Sampled {len(calibration_images)} calibration images")
print(f"  Random seed : {RANDOM_SEED}")

# ------------------------------------------------------------
# Save the calibration image list for reproducibility
# ------------------------------------------------------------

calib_list_path = CALIB_DIR / "calibration_image_list.json"

with open(calib_list_path, "w") as f:
    json.dump(
        [str(p) for p in calibration_images],
        f,
        indent=2,
    )

print(f"\n✓ Calibration image list saved:\n  {calib_list_path}")


# ============================================================
# CALIBRATION DATA READER
# ============================================================
#
# Preprocessing MUST match Ultralytics' own preprocessing:
#   - BGR -> RGB
#   - Resize (letterbox) to IMG_SIZE x IMG_SIZE
#   - Normalize to [0, 1]
#   - HWC -> CHW
#   - Add batch dimension
# ============================================================

def letterbox(image, new_shape=(640, 640), color=(114, 114, 114)):
    """Resize + pad image while maintaining aspect ratio (Ultralytics-style)."""
    shape = image.shape[:2]  # h, w

    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    new_unpad = (int(round(shape[1] * r)), int(round(shape[0] * r)))

    dw = new_shape[1] - new_unpad[0]
    dh = new_shape[0] - new_unpad[1]
    dw /= 2
    dh /= 2

    if shape[::-1] != new_unpad:
        image = cv2.resize(image, new_unpad, interpolation=cv2.INTER_LINEAR)

    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))

    image = cv2.copyMakeBorder(
        image, top, bottom, left, right,
        cv2.BORDER_CONSTANT, value=color
    )

    return image


def preprocess_image(path, img_size=IMG_SIZE):
    img = cv2.imread(str(path))
    if img is None:
        raise RuntimeError(f"Failed to read image: {path}")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = letterbox(img, (img_size, img_size))
    img = img.astype(np.float32) / 255.0
    img = img.transpose(2, 0, 1)  # HWC -> CHW
    img = np.expand_dims(img, axis=0)  # add batch dim
    return np.ascontiguousarray(img)


class YoloCalibrationDataReader(CalibrationDataReader):
    """Feeds calibration images to ONNX Runtime's static quantizer."""

    def __init__(self, image_paths, input_name, img_size=IMG_SIZE):
        self.image_paths = image_paths
        self.input_name = input_name
        self.img_size = img_size
        self._index = 0

    def get_next(self):
        if self._index >= len(self.image_paths):
            return None

        path = self.image_paths[self._index]
        self._index += 1

        tensor = preprocess_image(path, self.img_size)

        return {self.input_name: tensor}

    def rewind(self):
        self._index = 0


# ============================================================
# DETECTION-HEAD EXCLUSION
# ============================================================
#
# BACKGROUND — WHY THIS IS NECESSARY:
#
#   A first attempt at Step 4 quantized the ENTIRE graph
#   (every supported op type, including the full detection
#   head) and produced INT8 models that loaded and ran
#   without error, but returned P = R = mAP50 = mAP50-95 = 0.0
#   on all YOLO models, and near-zero garbage on RT-DETR.
#
#   This is a well-documented failure mode for YOLO-family
#   detectors under ONNX Runtime static quantization — the
#   final regression / DFL / objectness layers are extremely
#   sensitive to INT8 rounding, and quantizing them collapses
#   the output logits to a constant, so every detection fails
#   the confidence threshold. Every working example of
#   YOLO + onnxruntime.quantization in the wild excludes the
#   last block of nodes near the model outputs from
#   quantization (via `nodes_to_exclude`), and additionally
#   restricts quantization to Conv layers only (via
#   `op_types_to_quantize=["Conv"]`) rather than quantizing
#   every supported op type.
#
#   Because this notebook batches 5 architecturally different
#   models (YOLOv8 x3, YOLO11, RT-DETR), node names differ
#   per model and cannot be hardcoded. Instead, the tail of
#   each graph is discovered PROGRAMMATICALLY: nodes within
#   a fixed hop distance (upstream, via producer edges) of any
#   graph output are excluded from quantization. This is a
#   generic, architecture-agnostic approximation of "the
#   detection head" and is intentionally conservative.
#
# ============================================================

def get_tail_nodes_to_exclude(model_path, hops=6):
    """
    Return node names within `hops` upstream hops of any graph
    output. These are excluded from INT8 quantization to protect
    the detection head (box regression / DFL / objectness /
    classification layers), which are highly sensitive to
    quantization error in YOLO-family and DETR-family detectors.
    """

    model = onnx.load(str(model_path))
    graph = model.graph

    # Map: output tensor name -> node that produces it
    producer_of = {}
    for node in graph.node:
        for out in node.output:
            producer_of[out] = node

    # Map: node name -> node (for name-based exclusion list)
    # Some nodes may have empty .name; fall back to a synthetic
    # identifier based on op_type + output name in that case.
    def node_identifier(node):
        return node.name if node.name else f"{node.op_type}__{node.output[0]}"

    graph_output_names = [o.name for o in graph.output]

    excluded = set()
    frontier = set(graph_output_names)

    for _ in range(hops):
        next_frontier = set()

        for tensor_name in frontier:
            node = producer_of.get(tensor_name)

            if node is None:
                continue

            excluded.add(node_identifier(node))

            for inp in node.input:
                next_frontier.add(inp)

        frontier = next_frontier

    return list(excluded)


# ============================================================
# STEP 4D — INT8 STATIC QUANTIZATION (PER MODEL)
# ============================================================

section("STEP 4D — INT8 STATIC QUANTIZATION (QDQ, CALIBRATED)")

print("""
Quantization scope for this step, after the all-zero-detections
failure observed on the first attempt:

  - op_types_to_quantize restricted to ["Conv"]
    (backbone/neck convolutions only — the layers that actually
    dominate compute cost and benefit from INT8 speedup)

  - Detection-head tail nodes (within 6 hops of any graph
    output) excluded from quantization via nodes_to_exclude,
    protecting box regression / DFL / classification layers

  - A single-image sanity check runs immediately after each
    model is quantized, BEFORE the full dataset validation in
    Step 4E, so a broken model is caught in seconds rather
    than after another multi-minute validation pass.
""")

from onnxruntime.quantization.shape_inference import quant_pre_process

conversion_results = []

for name, paths in MODELS.items():

    subsection(f"INT8 QUANTIZATION — {name}")

    fp32_path = paths["fp32"]
    preproc_path = paths["int8_preprocessed"]
    int8_path = paths["int8"]

    fp32_size = file_size_mb(fp32_path)

    print(f"\nFP32 source:\n  {fp32_path}")
    print(f"FP32 model size : {fp32_size:.2f} MB")

    try:
        # ------------------------------------------------------------
        # 1. Required shape-inference / symbolic-shape preprocessing
        #    (onnxruntime.quantization requires this before
        #     quantize_static for reliable QDQ insertion)
        # ------------------------------------------------------------

        print("\n→ Running required ONNX preprocessing "
              "(shape inference for quantization)...")

        start_time = time.time()

        quant_pre_process(
            str(fp32_path),
            str(preproc_path),
            skip_symbolic_shape=False,
        )

        preproc_time = time.time() - start_time

        print(f"✓ Preprocessing completed ({preproc_time:.2f} s)")

        # ------------------------------------------------------------
        # 2. Identify graph input name
        # ------------------------------------------------------------

        sess_tmp = ort.InferenceSession(
            str(preproc_path),
            providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
        )
        input_name = sess_tmp.get_inputs()[0].name
        output_names = [o.name for o in sess_tmp.get_outputs()]
        del sess_tmp

        print(f"✓ Graph input identified  : '{input_name}'")
        print(f"✓ Graph output(s) identified : {output_names}")

        # ------------------------------------------------------------
        # 3. Discover detection-head nodes to exclude
        # ------------------------------------------------------------

        print("\n→ Discovering detection-head nodes to protect "
              "from quantization...")

        nodes_to_exclude = get_tail_nodes_to_exclude(
            preproc_path,
            hops=6,
        )

        print(
            f"✓ {len(nodes_to_exclude)} tail node(s) will be "
            f"excluded from quantization"
        )

        # ------------------------------------------------------------
        # 4. Build calibration data reader
        # ------------------------------------------------------------

        calib_reader = YoloCalibrationDataReader(
            calibration_images,
            input_name,
            IMG_SIZE,
        )

        # ------------------------------------------------------------
        # 5. Static INT8 quantization
        #    - Conv-only quantization scope
        #    - detection-head tail excluded
        # ------------------------------------------------------------

        print("\n→ Running static INT8 quantization "
              f"with {N_CALIBRATION_IMAGES} calibration images...")

        start_time = time.time()

        quantize_static(
            model_input=str(preproc_path),
            model_output=str(int8_path),
            calibration_data_reader=calib_reader,
            quant_format=QuantFormat.QDQ,
            weight_type=QuantType.QInt8,
            activation_type=QuantType.QUInt8,
            calibrate_method=CalibrationMethod.MinMax,
            per_channel=True,
            op_types_to_quantize=["Conv"],
            nodes_to_exclude=nodes_to_exclude,
        )

        quant_time = time.time() - start_time

        print(f"✓ INT8 quantization completed ({quant_time:.2f} s)")

        # ------------------------------------------------------------
        # 6. Graph checker
        # ------------------------------------------------------------

        print("→ Running ONNX graph checker...")

        checked_model = onnx.load(str(int8_path))
        onnx.checker.check_model(checked_model)

        print("✓ ONNX graph check passed")

        # ------------------------------------------------------------
        # 7. SANITY CHECK — single-image non-zero prediction test
        #    Fails fast instead of discovering all-zero detections
        #    only after a full multi-minute dataset validation.
        # ------------------------------------------------------------

        print("\n→ Running single-image sanity check "
              "(non-zero output test)...")

        sanity_sess = ort.InferenceSession(
            str(int8_path),
            providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
        )

        sanity_tensor = preprocess_image(calibration_images[0], IMG_SIZE)

        sanity_outputs = sanity_sess.run(
            None,
            {input_name: sanity_tensor},
        )

        del sanity_sess

        max_abs_output = max(
            float(np.max(np.abs(o))) for o in sanity_outputs
        )

        print(f"  Max |output| across all output tensors : "
              f"{max_abs_output:.6f}")

        if max_abs_output < 1e-6:
            raise RuntimeError(
                "\nSANITY CHECK FAILED — quantized model produces "
                "all-zero (or near-zero) output on a real "
                "calibration image.\n"
                "This indicates the detection head was still "
                "damaged by quantization despite exclusion. "
                "Increase `hops` in get_tail_nodes_to_exclude, "
                "or inspect the graph in Netron to identify the "
                "exact head node names for manual exclusion."
            )

        print("✓ Sanity check passed — model produces non-zero output")

        # ------------------------------------------------------------
        # 8. Size comparison
        # ------------------------------------------------------------

        int8_size = file_size_mb(int8_path)
        reduction_mb = fp32_size - int8_size
        reduction_percent = (reduction_mb / fp32_size) * 100

        print("\nModel size comparison:")
        print(f"  FP32 size      : {fp32_size:.2f} MB")
        print(f"  INT8 size      : {int8_size:.2f} MB")
        print(f"  Size reduction : {reduction_mb:.2f} MB")
        print(f"  Reduction      : {reduction_percent:.2f}%")

        conversion_results.append({
            "model": name,
            "status": "success",
            "fp32_size_mb": fp32_size,
            "int8_size_mb": int8_size,
            "size_reduction_mb": reduction_mb,
            "size_reduction_percent": reduction_percent,
            "preprocessing_time_s": preproc_time,
            "quantization_time_s": quant_time,
            "n_calibration_images": N_CALIBRATION_IMAGES,
            "n_excluded_tail_nodes": len(nodes_to_exclude),
            "sanity_check_max_abs_output": max_abs_output,
        })

    except Exception as e:
        print("\n❌ INT8 quantization failed")
        print(str(e))

        conversion_results.append({
            "model": name,
            "status": "quantization_failed",
            "fp32_size_mb": fp32_size,
            "int8_size_mb": np.nan,
            "size_reduction_mb": np.nan,
            "size_reduction_percent": np.nan,
            "preprocessing_time_s": np.nan,
            "quantization_time_s": np.nan,
            "n_calibration_images": N_CALIBRATION_IMAGES,
            "n_excluded_tail_nodes": np.nan,
            "sanity_check_max_abs_output": np.nan,
            "error": str(e),
        })


conversion_df = pd.DataFrame(conversion_results)

conversion_csv_path = INT8_DIR / "step4_int8_conversion_results.csv"
# (kept in 04_onnx_int8 alongside the .onnx artifacts, matching
#  where Step 3 stores its FP16 conversion log)
conversion_df.to_csv(conversion_csv_path, index=False)

print(f"\n\n✓ INT8 conversion results saved:\n  {conversion_csv_path}")

n_success = (conversion_df["status"] == "success").sum()

print(f"\nINT8 quantizations completed: {n_success}/{len(MODELS)}")

if n_success != len(MODELS):
    print(
        "\n⚠ WARNING: Not all models were successfully quantized. "
        "Review errors above before proceeding to Step 4E validation.\n"
        "Do NOT run the full dataset validation for a model that "
        "failed its sanity check — it will reproduce the same "
        "all-zero result and waste GPU time."
    )

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


STEP 4A — INT8 GPU ENVIRONMENT VERIFICATION
→ Checking Python packages...
✓ ONNX available
  ONNX version : 1.22.0
✓ ONNX Runtime available
  ONNX Runtime version : 1.26.0
✓ ONNX Runtime static quantization toolkit available
✓ OpenCV available (4.14.0)

Ultralytics version : 8.4.131
PyTorch version     : 2.11.0+cu128
CUDA available      : True
CUDA version        : 12.8
GPU                 : Tesla T4
GPU memory          : 14.56 GB

✓ PyTorch CUDA GPU verified

Available ONNX Runtime providers:
  - TensorrtExecutionProvider
  - CUDAExecutionProvider
  - CPUExecutionProvider

✓ CUDAExecutionProvider is available
✓ ONNX Runtime GPU is ready

Note: calibration inference during quanti

In [ ]:
# ============================================================
# STEP 4-E THROUGH 4-G
# INT8 GPU DATASET VALIDATION + FINAL FP32 vs FP16 vs INT8
# ============================================================
#
# PURPOSE:
#   PHASE E: Validate all 5 INT8 models on the test dataset
#            using CUDAExecutionProvider (same runtime as
#            Steps 2D and 3G — no runtime switch)
#   PHASE F: Build the final FP32 vs FP16 vs INT8 comparison,
#            reusing the existing Step 2D FP32 results and
#            Step 3G FP16 results (not re-run — same
#            "don't repeat validation" discipline as Step 3G)
#
# IMPORTANT:
#   - Assumes Step 4A–4D already ran in this session
#     (MODELS, DATA_YAML, SPLIT, IMG_SIZE, BATCH_SIZE,
#      CONF, IOU, INT8_DIR, CALIB_DIR, VALIDATION_DIR,
#      BENCHMARKS_DIR, FINAL_DIR, FP32_DIR, FP16_DIR,
#      BASELINE_DIR, LOG_DIR are already defined)
#   - Does NOT modify any ONNX artifacts
#   - Does NOT re-run calibration or quantization
#   - Does NOT repeat FP32/FP16 dataset validation — reuses
#     Step 2D / Step 3G results, same as Step 3G reused Step 2D
#
#   Runtime consistency note:
#   Steps 2, 3, and 4 all validate on CUDA:0 (Colab T4). This
#   keeps FP32/FP16/INT8 numbers directly comparable within
#   this notebook. Actual Raspberry Pi (CPU/ARM) performance
#   is measured separately in STEP 5, on the physical device.
#
# ============================================================

import gc

section("STEP 4E — INT8 GPU DATASET VALIDATION")

print("""
All five INT8 models will now be evaluated on GPU
against the existing 1,900-image test dataset, using
CUDAExecutionProvider — the same runtime used for the
FP32 (Step 2D) and FP16 (Step 3G) validations.
""")


def validate_onnx_gpu(model_path, name, precision_label):
    """Run Ultralytics dataset validation on GPU for a given ONNX model."""

    subsection(f"{precision_label} GPU VALIDATION — {name}")

    print(f"\nModel:\n  {model_path}")
    print(f"Model size : {file_size_mb(model_path):.2f} MB")

    print("\n→ Loading Ultralytics ONNX model...")

    model = YOLO(str(model_path), task="detect")

    print("✓ Model loaded")

    print("\n→ Running test dataset validation...")

    start_time = time.time()

    metrics = model.val(
        data=DATA_YAML,
        split=SPLIT,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        conf=CONF,
        iou=IOU,
        device=DEVICE,
        verbose=True,
    )

    wall_time = time.time() - start_time

    precision = float(metrics.box.mp)
    recall = float(metrics.box.mr)
    map50 = float(metrics.box.map50)
    map50_95 = float(metrics.box.map)

    speed = metrics.speed  # dict: preprocess, inference, postprocess (ms/img)

    preprocess_ms = speed.get("preprocess", np.nan)
    inference_ms = speed.get("inference", np.nan)
    postprocess_ms = speed.get("postprocess", np.nan)

    total_ms = preprocess_ms + inference_ms + postprocess_ms
    fps = 1000.0 / total_ms if total_ms > 0 else np.nan

    print(f"\n✓ {precision_label} validation completed")

    print(f"\nPrecision  : {precision:.6f}")
    print(f"Recall     : {recall:.6f}")
    print(f"mAP50      : {map50:.6f}")
    print(f"mAP50-95   : {map50_95:.6f}")
    print(f"Preprocess : {preprocess_ms:.3f} ms")
    print(f"Inference  : {inference_ms:.3f} ms")
    print(f"Postproc   : {postprocess_ms:.3f} ms")
    print(f"Total/img  : {total_ms:.3f} ms")
    print(f"FPS        : {fps:.3f}")
    print(f"Wall time  : {wall_time:.2f} s")

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "model": name,
        "precision_type": precision_label,
        "status": "success",
        "device": DEVICE,
        "precision": precision,
        "recall": recall,
        "map50": map50,
        "map50_95": map50_95,
        "preprocess_ms": preprocess_ms,
        "inference_ms": inference_ms,
        "postprocess_ms": postprocess_ms,
        "total_ms": total_ms,
        "fps": fps,
        "validation_wall_time_s": wall_time,
    }


int8_results = []

# ------------------------------------------------------------
# Load Step 4D conversion results to skip any model that
# failed its single-image sanity check — running the full
# 1,900-image validation on a model already known to output
# all-zero predictions just reproduces the same failure and
# wastes several minutes of GPU time per model.
# ------------------------------------------------------------

conversion_results_path = INT8_DIR / "step4_int8_conversion_results.csv"

sanity_status = {}

if conversion_results_path.exists():
    _conv_df = pd.read_csv(conversion_results_path)
    for _, row in _conv_df.iterrows():
        sanity_status[row["model"]] = {
            "status": row.get("status", "unknown"),
            "max_abs_output": row.get("sanity_check_max_abs_output", np.nan),
        }
    print(f"✓ Loaded Step 4D sanity check results:\n  {conversion_results_path}")
else:
    print(
        f"\n⚠ Step 4D conversion results not found at:\n  "
        f"{conversion_results_path}\n"
        "Proceeding without sanity-check pre-filtering — if any "
        "model produces all-zero results below, re-run Step 4D "
        "first, it now includes a sanity check."
    )

for name, paths in MODELS.items():

    int8_path = paths["int8"]

    if not int8_path.exists():
        print(f"\n⚠ Skipping {name} — INT8 model not found "
              f"(quantization may have failed)")
        int8_results.append({
            "model": name,
            "precision_type": "INT8",
            "status": "missing",
            "device": DEVICE,
        })
        continue

    info = sanity_status.get(name)

    if info is not None and info["status"] != "success":
        print(
            f"\n⚠ Skipping {name} — Step 4D quantization/sanity "
            f"check did not pass (status: {info['status']}). "
            f"Fix the quantization step before validating."
        )
        int8_results.append({
            "model": name,
            "precision_type": "INT8",
            "status": "skipped_failed_sanity_check",
            "device": DEVICE,
        })
        continue

    if info is not None and (
        pd.isna(info["max_abs_output"]) or info["max_abs_output"] < 1e-6
    ):
        print(
            f"\n⚠ Skipping {name} — sanity check recorded a "
            f"near-zero max output ({info['max_abs_output']}). "
            f"This model would return all-zero detections."
        )
        int8_results.append({
            "model": name,
            "precision_type": "INT8",
            "status": "skipped_zero_output",
            "device": DEVICE,
        })
        continue

    try:
        result = validate_onnx_gpu(int8_path, name, "INT8")
        int8_results.append(result)
    except Exception as e:
        print(f"\n❌ INT8 validation failed for {name}")
        print(str(e))
        int8_results.append({
            "model": name,
            "precision_type": "INT8",
            "status": "validation_failed",
            "device": DEVICE,
            "error": str(e),
        })

int8_results_df = pd.DataFrame(int8_results)

int8_results_csv_path = (
    VALIDATION_DIR / "step4_gpu_int8_dataset_validation.csv"
)
int8_results_df.to_csv(int8_results_csv_path, index=False)

print(f"\n\n✓ INT8 dataset validation results saved:\n  {int8_results_csv_path}")


# ============================================================
# STEP 4F — LOAD EXISTING FP32 AND FP16 RESULTS
# ============================================================
#
# Reuses Step 2D (FP32) and Step 3G (FP16) results instead of
# re-running dataset validation — same discipline Step 3G used
# for FP32, saving a full pass over 1,900 images per model.
# ============================================================

section("STEP 4F — LOADING EXISTING FP32 / FP16 RESULTS")

FP32_RESULTS_CANDIDATES = [
    BASELINE_DIR / "step2d_gpu_onnx_fp32_validation.csv",
    BASELINE_DIR / "step2d_gpu_onnx_fp32_results.csv",
    BASELINE_DIR / "step2d_gpu_onnx_results.csv",
    BASELINE_DIR / "step2_gpu_onnx_fp32_results.csv",
]

FP16_RESULTS_CANDIDATES = [
    BASELINE_DIR / "step3g_final_fp16_dataset_validation.csv",
    BASELINE_DIR / "step3_gpu_onnx_fp16_validation.csv",
]


def find_existing_results(candidates):
    for path in candidates:
        if path.exists():
            return path
    return None


def find_column(df, candidates):
    lower_map = {str(col).lower(): col for col in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    return None


def normalize_results(df, precision_label):
    """Normalize FP32/FP16 result columns to a predictable schema."""

    rename_map = {}

    aliases = {
        "model": ["model", "model_name"],
        "precision": ["precision", "box_precision", "P"],
        "recall": ["recall", "box_recall", "R"],
        "map50": ["map50", "mAP50", "mAP_50", "map_50"],
        "map50_95": ["map50_95", "mAP50-95", "mAP50_95", "mAP_50_95", "map_50_95"],
        "inference_ms": ["inference_ms", "inference", "inference_time_ms"],
        "fps": ["fps", "FPS"],
    }

    for target, names in aliases.items():
        col = find_column(df, names)
        if col is not None:
            rename_map[col] = target

    df = df.rename(columns=rename_map)
    df["precision_type"] = precision_label
    df["status"] = "success"

    keep_cols = [
        "model", "precision_type", "status",
        "precision", "recall", "map50", "map50_95",
        "inference_ms", "fps",
    ]
    keep_cols = [c for c in keep_cols if c in df.columns]

    return df[keep_cols]


fp32_path_found = find_existing_results(FP32_RESULTS_CANDIDATES)
fp16_path_found = find_existing_results(FP16_RESULTS_CANDIDATES)

if fp32_path_found is None:
    raise FileNotFoundError(
        "\nCould not locate existing Step 2D FP32 results.\n"
        f"Checked:\n" + "\n".join(str(p) for p in FP32_RESULTS_CANDIDATES)
    )

if fp16_path_found is None:
    raise FileNotFoundError(
        "\nCould not locate existing Step 3G FP16 results.\n"
        f"Checked:\n" + "\n".join(str(p) for p in FP16_RESULTS_CANDIDATES)
    )

print(f"✓ FP32 results found:\n  {fp32_path_found}")
print(f"✓ FP16 results found:\n  {fp16_path_found}")

fp32_df = normalize_results(pd.read_csv(fp32_path_found), "FP32")
fp16_df = normalize_results(pd.read_csv(fp16_path_found), "FP16")

print(f"\n✓ FP32 results normalized ({len(fp32_df)} rows)")
print(f"✓ FP16 results normalized ({len(fp16_df)} rows)")


# ============================================================
# STEP 4G — FINAL FP32 vs FP16 vs INT8 COMPARISON
# ============================================================

section("STEP 4G — FINAL FP32 vs FP16 vs INT8 COMPARISON")

comparison_rows = []

for name in MODELS.keys():

    fp32_row = fp32_df[fp32_df["model"] == name]
    fp16_row = fp16_df[fp16_df["model"] == name]
    int8_row = int8_results_df[
        (int8_results_df["model"] == name)
        & (int8_results_df["status"] == "success")
    ]

    if fp32_row.empty or int8_row.empty:
        print(f"\n⚠ Skipping comparison for {name} — "
              f"missing FP32 or INT8 results")
        continue

    fp32 = fp32_row.iloc[0]
    int8 = int8_row.iloc[0]
    fp16 = fp16_row.iloc[0] if not fp16_row.empty else None

    subsection(name)

    print(f"\nmAP50-95")
    print(f"  FP32 : {fp32['map50_95']:.6f}")
    if fp16 is not None:
        print(f"  FP16 : {fp16['map50_95']:.6f}")
    print(f"  INT8 : {int8['map50_95']:.6f}")
    print(f"  Δ (INT8 - FP32) : {int8['map50_95'] - fp32['map50_95']:+.6f}")

    print(f"\nInference (GPU, ms/img)")
    print(f"  FP32 : {fp32['inference_ms']:.3f}")
    if fp16 is not None:
        print(f"  FP16 : {fp16['inference_ms']:.3f}")
    print(f"  INT8 : {int8['inference_ms']:.3f}")

    speedup = fp32["inference_ms"] / int8["inference_ms"]
    print(f"  INT8 speedup vs FP32 : {speedup:.2f}x")

    map_delta = int8["map50_95"] - fp32["map50_95"]

    accuracy_assessment = (
        "Equivalent — negligible difference"
        if abs(map_delta) < 0.02
        else ("Degraded — review calibration set / per-tensor fallback"
              if map_delta < 0 else "Improved (unexpected — verify)")
    )

    latency_assessment = (
        "Faster" if int8["inference_ms"] < fp32["inference_ms"]
        else "Slower — investigate (unusual for INT8 on GPU; "
             "GPU int8 kernels are less mature than CPU int8 kernels, "
             "this can happen and does not necessarily rule out the "
             "model for the Raspberry Pi CPU target — see Step 5)"
    )

    comparison_rows.append({
        "model": name,
        "fp32_map50_95": fp32["map50_95"],
        "fp16_map50_95": fp16["map50_95"] if fp16 is not None else np.nan,
        "int8_map50_95": int8["map50_95"],
        "int8_vs_fp32_map_delta": map_delta,
        "fp32_precision": fp32["precision"],
        "int8_precision": int8["precision"],
        "fp32_recall": fp32["recall"],
        "int8_recall": int8["recall"],
        "fp32_inference_ms": fp32["inference_ms"],
        "fp16_inference_ms": fp16["inference_ms"] if fp16 is not None else np.nan,
        "int8_inference_ms": int8["inference_ms"],
        "int8_speedup_vs_fp32": speedup,
        "accuracy_assessment": accuracy_assessment,
        "latency_assessment": latency_assessment,
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_csv_path = (
    BENCHMARKS_DIR / "step4_final_fp32_vs_fp16_vs_int8_comparison.csv"
)
comparison_df.to_csv(comparison_csv_path, index=False)

print(f"\n\n✓ Final FP32 vs FP16 vs INT8 comparison saved:\n  {comparison_csv_path}")


# ============================================================
# FINAL SUMMARY TABLE
# ============================================================

section("STEP 4G — FINAL SUMMARY TABLE")

print(comparison_df.to_string(index=False))


# ============================================================
# STEP 4 — FINAL LOG
# ============================================================

section("STEP 4 — FINAL STATUS")

n_int8_success = (int8_results_df["status"] == "success").sum()

final_status = "PASSED" if n_int8_success == len(MODELS) else "INCOMPLETE"

print(f"\nFINAL STATUS : {final_status}")
print(f"\nINT8 dataset validations : {n_int8_success}/{len(MODELS)}")

log_payload = {
    "step": "STEP 4 — INT8 quantization + GPU validation",
    "runtime": "Colab GPU (CUDA:0, same as Steps 2 and 3)",
    "quantization_method": "ONNX Runtime static QDQ (MinMax calibration)",
    "n_calibration_images": int(N_CALIBRATION_IMAGES),
    "final_status": final_status,
    "int8_validations_passed": int(n_int8_success),
    "total_models": len(MODELS),
    "note": (
        "This step validates INT8 on the same GPU runtime as "
        "Steps 2/3 for direct comparability within the notebook. "
        "Raspberry Pi CPU/ARM on-device performance is measured "
        "separately in Step 5."
    ),
}

log_path = LOG_DIR / "step4_int8_calibration_experiment.json"

with open(log_path, "w") as f:
    json.dump(log_payload, f, indent=2)

print(f"\n✓ Step 4 log saved:\n  {log_path}")

print("""

====================================================================

STEP 4 COMPLETE

If FINAL STATUS = PASSED:

    STEP 4 — INT8 ✓ COMPLETE

Do NOT start Step 5 inside this cell.

Next phase:

    STEP 5 — RASPBERRY PI ON-DEVICE VALIDATION

with:

    1. Transfer FP32 / FP16 / INT8 ONNX models to physical Pi
       (candidates for transfer live in 08_final once selected)
    2. Install onnxruntime (CPU build) on Raspberry Pi OS
    3. Re-run dataset (or sampled) validation ON DEVICE, CPU only
    4. Measure real on-device latency, FPS, thermal behavior
    5. Compare Colab-GPU numbers vs actual Pi CPU numbers
    6. Select final production model into 08_final/

====================================================================
""")



STEP 4E — INT8 GPU DATASET VALIDATION

All five INT8 models will now be evaluated on GPU
against the existing 1,900-image test dataset, using
CUDAExecutionProvider — the same runtime used for the
FP32 (Step 2D) and FP16 (Step 3G) validations.

✓ Loaded Step 4D sanity check results:
  /content/drive/MyDrive/Stage_PFA/MODEL_OPTIMIZATION/04_onnx_int8/step4_int8_conversion_results.csv


----------------------------------------------------------------------------------------------------
INT8 GPU VALIDATION — yolov8n_50ep
----------------------------------------------------------------------------------------------------

Model:
  /content/drive/MyDrive/Stage_PFA/MODEL_OPTIMIZATION/04_onnx_int8/yolov8n_50ep_int8.onnx
Model size : 3.28 MB

→ Loading Ultralytics ONNX model...
✓ Model loaded

→ Running test dataset validation...
Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Loading /content/drive/MyDrive/Stage_PFA/MODEL_OPTIMIZATION/04_onnx_int8/yolov8n_5

In [ ]:
# ============================================================
# STEP 5A-COLAB — BUILD RASPBERRY PI TRANSFER PACKAGE
# ============================================================
#
# PURPOSE:
#   Collect the winning candidate ONNX models (FP32 / FP16 /
#   INT8) plus everything the Pi needs to run validation
#   on-device, into a single self-contained folder under
#   08_final/pi_package/. You download this folder, then scp
#   it to the Pi yourself.
#
# CANDIDATES — ALL FIVE MODELS, full professional comparison
# across your epoch/architecture sweep. Final model selection
# happens AFTER real Pi numbers are in, not before:
#   - yolov8n_50ep    (GPU INT8 borderline, 5.8% mAP50-95 drop)
#   - yolov8n_75ep    (GPU INT8 borderline, 4.7% mAP50-95 drop)
#   - yolov8n_100ep   (GPU INT8 viable, 1.9% mAP50-95 drop)
#   - yolo11n         (GPU INT8 viable, 1.2% mAP50-95 drop — best)
#   - rtdetr_50ep     (GPU INT8 already known broken — 71%
#     mAP50-95 drop, documented transformer-decoder quantization
#     collapse. Included at your request for a complete
#     comparison. Also the slowest model at every GPU precision
#     tier, so expect its Pi CPU numbers to be the most costly
#     of the five — this will make that cost concrete rather
#     than theoretical. Its INT8 result is not expected to
#     become viable on-device; FP32/FP16 are still legitimate
#     data points.)
#
# For each candidate model, copies FP32 + FP16 + INT8 ONNX.
# (FP32/FP16 included too, not just INT8, so Step 5B can
# reproduce the full three-way comparison on real Pi hardware,
# not just validate the INT8 path in isolation.)
#
# DOES NOT modify any existing artifact. Read-only copy.
# ============================================================

import os
import json
import shutil
from pathlib import Path

import pandas as pd


# ============================================================
# 1. CONFIGURATION
# ============================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "MODEL_OPTIMIZATION"
)

FP32_DIR = BASE_DIR / "02_onnx_fp32"
FP16_DIR = BASE_DIR / "03_onnx_fp16"
INT8_MODELS_DIR = BASE_DIR / "04_onnx_int8" #/ "models"

FINAL_DIR = BASE_DIR / "08_final"
PACKAGE_DIR = FINAL_DIR / "pi_package"
PACKAGE_MODELS_DIR = PACKAGE_DIR / "models"

INT8_DIR = BASE_DIR / "07_benchmarks"
FINAL_COMPARISON_CSV = INT8_DIR / "step4_final_fp32_vs_fp16_vs_int8_comparison.csv"

# All five models carried into Step 5 for a complete on-device
# comparison. RT-DETR's INT8 is already known broken on GPU
# (71% mAP50-95 drop) — it's included anyway per your call, so
# the final report can show the Pi-CPU cost concretely rather
# than asserting it. If RT-DETR's ONNX exports used a different
# filename stem than the pattern below, the existence check in
# section 2 will fail loudly and name the missing path.
CANDIDATES = [
    "yolov8n_50ep",
    "yolov8n_75ep",
    "yolov8n_100ep",
    "yolo11n",
    "rtdetr_50ep",
]

DATASET_ROOT = (
    "/content/drive/MyDrive/PFE/"
    "Dataset/Vehicles_Dataset"
)
DATA_YAML = DATASET_ROOT + "/data.yaml"


def section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


# ============================================================
# 2. VERIFY SOURCE FILES EXIST
# ============================================================

section("STEP 5A-COLAB — VERIFYING SOURCE ARTIFACTS")

MODEL_FILES = {}

for model in CANDIDATES:
    entry = {
        "fp32": FP32_DIR / f"{model}_fp32.onnx",
        "fp16": FP16_DIR / f"{model}_fp16.onnx",
        "int8": INT8_MODELS_DIR / f"{model}_int8.onnx",
    }

    for precision, path in entry.items():
        if not path.exists():
            raise FileNotFoundError(
                f"Missing required artifact for {model} [{precision}]:\n"
                f"  {path}\n"
                f"Re-run the corresponding Step 2/3/4 export before "
                f"building the Pi package."
            )
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"  ✓ {model:15s} [{precision}] {size_mb:7.2f} MB  {path.name}")

    MODEL_FILES[model] = entry

print(f"\n✓ All {len(CANDIDATES)} candidate models verified "
      f"({len(CANDIDATES) * 3} ONNX files total)")


# ============================================================
# 3. CREATE PACKAGE DIRECTORY STRUCTURE
# ============================================================

section("CREATING PACKAGE STRUCTURE")

if PACKAGE_DIR.exists():
    print(f"→ Removing previous package at {PACKAGE_DIR}")
    shutil.rmtree(PACKAGE_DIR)

for model in CANDIDATES:
    (PACKAGE_MODELS_DIR / model).mkdir(parents=True, exist_ok=True)

print(f"✓ Created {PACKAGE_DIR}")


# ============================================================
# 4. COPY MODELS
# ============================================================

section("COPYING MODELS INTO PACKAGE")

manifest_rows = []

for model, precisions in MODEL_FILES.items():
    for precision, src_path in precisions.items():
        dst_path = PACKAGE_MODELS_DIR / model / f"{model}_{precision}.onnx"
        shutil.copy2(src_path, dst_path)

        size_mb = dst_path.stat().st_size / (1024 ** 2)
        print(f"  ✓ {model}/{dst_path.name}  ({size_mb:.2f} MB)")

        manifest_rows.append({
            "model": model,
            "precision": precision,
            "filename": f"{model}/{dst_path.name}",
            "size_mb": round(size_mb, 3),
        })


# ============================================================
# 5. COPY GPU BASELINE RESULTS (for Colab-GPU vs Pi-CPU comparison later)
# ============================================================

section("COPYING COLAB-GPU BASELINE RESULTS")

if FINAL_COMPARISON_CSV.exists():
    dst = PACKAGE_DIR / "colab_gpu_fp32_fp16_int8_comparison.csv"
    shutil.copy2(FINAL_COMPARISON_CSV, dst)
    print(f"  ✓ Copied {FINAL_COMPARISON_CSV.name}")
else:
    print(f"  ⚠ Not found, skipping: {FINAL_COMPARISON_CSV}")
    print(f"    (Step 5B will still run — this file is only used "
          f"later for the final Colab-GPU vs Pi-CPU table)")


# ============================================================
# 6. COPY TEST DATASET LABELS + data.yaml
#    (images copied separately — see note below)
# ============================================================

section("PREPARING DATASET REFERENCE FILES")

package_dataset_dir = PACKAGE_DIR / "dataset_ref"
package_dataset_dir.mkdir(exist_ok=True)

data_yaml_src = Path(DATA_YAML)
if data_yaml_src.exists():
    shutil.copy2(data_yaml_src, package_dataset_dir / "data.yaml")
    print(f"  ✓ Copied data.yaml")
else:
    print(f"  ⚠ data.yaml not found at {data_yaml_src} — copy manually")

print(
    "\n  NOTE: This script does NOT copy the 1,900 test images "
    "themselves (too large for this package folder). Transfer "
    "the test split's images/ and labels/ folders to the Pi "
    "separately via scp, matching the layout Step 5B expects:\n"
    "    ~/pfe_pi/dataset/test/images/\n"
    "    ~/pfe_pi/dataset/test/labels/\n"
    "    ~/pfe_pi/dataset/data.yaml"
)


# ============================================================
# 7. WRITE MANIFEST
# ============================================================

section("WRITING MANIFEST")

manifest_df = pd.DataFrame(manifest_rows)
manifest_csv_path = PACKAGE_DIR / "package_manifest.csv"
manifest_df.to_csv(manifest_csv_path, index=False)

manifest_json = {
    "candidates": CANDIDATES,
    "notes": {
        "rtdetr_50ep": "Included for full 5-model comparison per user "
                        "decision. Known from Step 4 (GPU): INT8 mAP50-95 "
                        "drop of 71% (documented transformer-decoder "
                        "quantization sensitivity), and slowest model at "
                        "every precision tier on T4 GPU. Expect this gap "
                        "to widen on Pi ARM CPU. Validate INT8 anyway for "
                        "a real measured number rather than an assumption "
                        "— final model selection is deferred to Step 5 "
                        "on-device results.",
    },
    "files": manifest_rows,
    "total_size_mb": round(sum(r["size_mb"] for r in manifest_rows), 3),
}

with open(PACKAGE_DIR / "package_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest_json, f, indent=2)

print(f"✓ Manifest written: {manifest_csv_path}")
print(f"✓ Total package size: {manifest_json['total_size_mb']:.2f} MB "
      f"(models only, excludes dataset)")


# ============================================================
# 8. FINAL SUMMARY
# ============================================================

section("STEP 5A-COLAB COMPLETE")

print(f"""
Package ready at:
  {PACKAGE_DIR}

Structure:
  pi_package/
    models/
      yolov8n_50ep/   (fp32, fp16, int8)
      yolov8n_75ep/   (fp32, fp16, int8)
      yolov8n_100ep/  (fp32, fp16, int8)
      yolo11n/        (fp32, fp16, int8)
      rtdetr_50ep/    (fp32, fp16, int8)
    dataset_ref/
      data.yaml
    colab_gpu_fp32_fp16_int8_comparison.csv
    package_manifest.csv
    package_manifest.json

NEXT STEPS (manual, on your machine):
  1. Download this folder from Drive to your local machine.
  2. scp -r pi_package/ pi@<PI_IP>:~/pfe_pi/
  3. Separately scp the test dataset images+labels to:
       ~/pfe_pi/dataset/test/images/
       ~/pfe_pi/dataset/test/labels/
  4. On the Pi, run Step 5B setup script (setup_pi.sh), then
     the Python validation script.
""")


STEP 5A-COLAB — VERIFYING SOURCE ARTIFACTS
  ✓ yolov8n_50ep    [fp32]   11.70 MB  yolov8n_50ep_fp32.onnx
  ✓ yolov8n_50ep    [fp16]    5.89 MB  yolov8n_50ep_fp16.onnx
  ✓ yolov8n_50ep    [int8]    3.28 MB  yolov8n_50ep_int8.onnx
  ✓ yolov8n_75ep    [fp32]   11.70 MB  yolov8n_75ep_fp32.onnx
  ✓ yolov8n_75ep    [fp16]    5.89 MB  yolov8n_75ep_fp16.onnx
  ✓ yolov8n_75ep    [int8]    3.28 MB  yolov8n_75ep_int8.onnx
  ✓ yolov8n_100ep   [fp32]   11.70 MB  yolov8n_100ep_fp32.onnx
  ✓ yolov8n_100ep   [fp16]    5.89 MB  yolov8n_100ep_fp16.onnx
  ✓ yolov8n_100ep   [int8]    3.28 MB  yolov8n_100ep_int8.onnx
  ✓ yolo11n         [fp32]   10.12 MB  yolo11n_fp32.onnx
  ✓ yolo11n         [fp16]    5.11 MB  yolo11n_fp16.onnx
  ✓ yolo11n         [int8]    2.99 MB  yolo11n_int8.onnx
  ✓ rtdetr_50ep     [fp32]  125.53 MB  rtdetr_50ep_fp32.onnx
  ✓ rtdetr_50ep     [fp16]   62.98 MB  rtdetr_50ep_fp16.onnx
  ✓ rtdetr_50ep     [int8]   57.19 MB  rtdetr_50ep_int8.onnx

✓ All 5 candidate models verified (15 ON

## End of step 4 in today's session